<a href="https://colab.research.google.com/github/Pratyushan4191/Axiom_Microplastics/blob/main/Axiom_microplastics_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU count:", torch.cuda.device_count())

    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}:", torch.cuda.get_device_name(i))

print("\nKaggle input directories:")
for path in ["/kaggle/input", "/kaggle/working"]:
    print(path, "exists:", os.path.exists(path))

if os.path.exists("/kaggle/input"):
    print("\nDatasets:")
    for item in os.listdir("/kaggle/input"):
        print(" -", item)

In [ ]:
from pathlib import Path

DATA_ROOT = Path("/kaggle/input/datasets")

print("Dataset root:", DATA_ROOT)
print()

for path in sorted(DATA_ROOT.rglob("*")):
    if path.is_file():
        size_mb = path.stat().st_size / (1024**2)
        print(f"{path.relative_to(DATA_ROOT)}  |  {size_mb:.2f} MB")

#Loading Datasets

In [ ]:
from pathlib import Path

ROOT = Path("/kaggle/input/datasets/kaggle_bundle")

for split in ["train", "val", "test"]:
    img_dir = ROOT / "data" / split / "images"
    mask_dir = ROOT / "data" / split / "masks"

    print(
        f"{split:5s} | "
        f"images={len(list(img_dir.glob('*'))):4d} | "
        f"masks={len(list(mask_dir.glob('*'))):4d}"
    )

ckpt = ROOT / "checkpoints" / "beach_unet_resnet50_strong_aug_best_dice.pth"

print("\nCheckpoint:")
print("exists:", ckpt.exists())

if ckpt.exists():
    print("size_MB:", round(ckpt.stat().st_size / (1024**2), 2))

In [ ]:
from pathlib import Path

ROOT = Path("/kaggle/input/datasets")

print("ROOT:", ROOT)
print("\nTop-level contents:")

for p in sorted(ROOT.iterdir()):
    kind = "DIR " if p.is_dir() else "FILE"
    print(f"{kind}  {p.name}")

print("\nFirst 30 files/directories:")
for i, p in enumerate(ROOT.rglob("*")):
    if i >= 30:
        break
    print(p.relative_to(ROOT))

In [ ]:
from pathlib import Path

ROOT = Path(
    "/kaggle/input/datasets/"
    "shanumkhsharma/microplastic-step16a-recovery/"
    "kaggle_bundle"
)

for split in ["train", "val", "test"]:
    img_dir = ROOT / "data" / split / "images"
    mask_dir = ROOT / "data" / split / "masks"

    print(
        f"{split:5s} | "
        f"images={len(list(img_dir.glob('*.png'))):4d} | "
        f"masks={len(list(mask_dir.glob('*.png'))):4d}"
    )

ckpt = ROOT / "checkpoints" / "beach_unet_resnet50_strong_aug_best_dice.pth"

print("\nCheckpoint:")
print("exists:", ckpt.exists())
print("size_MB:", round(ckpt.stat().st_size / (1024**2), 2))

if ckpt.exists():
    import torch

    checkpoint = torch.load(ckpt, map_location="cpu")

    print("checkpoint epoch:", checkpoint.get("epoch"))
    print("best_val_dice:", checkpoint.get("best_val_dice"))
    print("history entries:", len(checkpoint.get("history", [])))

In [ ]:
pip install segmentation_models_pytorch

verify that the checkpoint loads into the exact model

In [ ]:
import torch
import segmentation_models_pytorch as smp
from pathlib import Path

# --------------------------------------------------
# Paths
# --------------------------------------------------
ROOT = Path(
    "/kaggle/input/datasets/"
    "shanumkhsharma/microplastic-step16a-recovery/"
    "kaggle_bundle"
)

CKPT = ROOT / "checkpoints" / "beach_unet_resnet50_strong_aug_best_dice.pth"

# --------------------------------------------------
# Device
# --------------------------------------------------
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print("CUDA available:", torch.cuda.is_available())
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )

# --------------------------------------------------
# Reconstruct model
# IMPORTANT:
# encoder_weights=None because the complete model
# weights are already inside the checkpoint.
# --------------------------------------------------
model = smp.Unet(
    encoder_name="resnet50",
    encoder_weights=None,
    in_channels=3,
    classes=1
).to(device)

# --------------------------------------------------
# Load checkpoint
# --------------------------------------------------
checkpoint = torch.load(CKPT, map_location=device)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("\nCheckpoint loaded successfully.")
print("Checkpoint epoch:", checkpoint["epoch"])
print("Best validation Dice:", checkpoint["best_val_dice"])

# --------------------------------------------------
# Parameter verification
# --------------------------------------------------
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print("\nModel parameters:")
print("Total:", total_params)
print("Trainable:", trainable_params)

# --------------------------------------------------
# GPU forward-pass test
# --------------------------------------------------
x = torch.randn(
    1, 3, 512, 512,
    device=device,
    dtype=torch.float32
)

with torch.no_grad():
    logits = model(x)

print("\nForward pass:")
print("Input shape :", tuple(x.shape))
print("Output shape:", tuple(logits.shape))
print("Output dtype:", logits.dtype)
print("Finite:", torch.isfinite(logits).all().item())

if torch.cuda.is_available():
    print(
        "GPU memory allocated:",
        round(torch.cuda.memory_allocated() / 1024**3, 3),
        "GB"
    )

In [ ]:
from pathlib import Path
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
import random
import numpy as np

# ============================================================
# Paths
# ============================================================

ROOT = Path(
    "/kaggle/input/datasets/"
    "shanumkhsharma/microplastic-step16a-recovery/"
    "kaggle_bundle"
)

# ============================================================
# File lists
# ============================================================

def get_pairs(split):
    img_dir = ROOT / "data" / split / "images"
    mask_dir = ROOT / "data" / split / "masks"

    images = sorted(img_dir.glob("*.png"))

    pairs = []
    for img in images:
        mask = mask_dir / img.name
        if mask.exists():
            pairs.append((img, mask))

    return pairs


train_pairs = get_pairs("train")
val_pairs = get_pairs("val")
test_pairs = get_pairs("test")

print("Dataset pairs:")
print("Train:", len(train_pairs))
print("Val  :", len(val_pairs))
print("Test :", len(test_pairs))


# ============================================================
# Step 16A augmentation parameters
# ============================================================

AUGMENTATION_CONFIG = {
    "hflip_p": 0.50,
    "rotation_degrees": 20,
    "scale_min": 0.90,
    "scale_max": 1.10,
    "translation_fraction": 0.10,
    "brightness_min": 0.75,
    "brightness_max": 1.25,
    "contrast_min": 0.75,
    "contrast_max": 1.25,
    "saturation_min": 0.70,
    "saturation_max": 1.30,
    "gaussian_blur_p": 0.20,
    "gaussian_blur_radius_min": 0.3,
    "gaussian_blur_radius_max": 1.5,
}

print("\nStep 16A augmentation configuration:")
for k, v in AUGMENTATION_CONFIG.items():
    print(f"{k}: {v}")


# ============================================================
# Dataset
# ============================================================

class BeachSegmentationDataset(Dataset):

    def __init__(self, pairs, train=False):
        self.pairs = pairs
        self.train = train

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):

        img_path, mask_path = self.pairs[idx]

        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")

        # 256 -> 512
        image = image.resize((512, 512), Image.Resampling.BILINEAR)
        mask = mask.resize((512, 512), Image.Resampling.NEAREST)

        image = TF.to_tensor(image)
        mask = torch.from_numpy(
            np.array(mask, dtype=np.uint8)
        ).float()

        mask = (mask > 0).float().unsqueeze(0)

        return image, mask


# ============================================================
# Construct datasets/loaders
# ============================================================

train_ds = BeachSegmentationDataset(train_pairs, train=True)
val_ds = BeachSegmentationDataset(val_pairs, train=False)
test_ds = BeachSegmentationDataset(test_pairs, train=False)

train_loader = DataLoader(
    train_ds,
    batch_size=8,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=8,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_ds,
    batch_size=8,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("\nDataLoaders constructed:")
print("Train batches:", len(train_loader))
print("Val batches  :", len(val_loader))
print("Test batches :", len(test_loader))


# ============================================================
# Actual batch verification
# ============================================================

images, masks = next(iter(train_loader))

print("\nActual train batch:")
print("Images:", images.shape, images.dtype)
print("Masks :", masks.shape, masks.dtype)

print("Image range:",
      float(images.min()),
      float(images.max()))

print("Mask values:",
      torch.unique(masks).tolist())

print("Batch positive fraction:",
      float(masks.mean()))


# ============================================================
# Validation batch
# ============================================================

images_v, masks_v = next(iter(val_loader))

print("\nActual validation batch:")
print("Images:", images_v.shape, images_v.dtype)
print("Masks :", masks_v.shape, masks_v.dtype)

print("Mask values:",
      torch.unique(masks_v).tolist())

In [ ]:
import torch
from pathlib import Path

ROOT = Path(
    "/kaggle/input/datasets/"
    "shanumkhsharma/microplastic-step16a-recovery/"
    "kaggle_bundle"
)

CKPT = ROOT / "checkpoints" / "beach_unet_resnet50_strong_aug_best_dice.pth"

checkpoint = torch.load(CKPT, map_location="cpu")

print("Checkpoint epoch:", checkpoint["epoch"])
print("Best validation Dice:", checkpoint["best_val_dice"])

print("\nCheckpoint keys:")
print(sorted(checkpoint.keys()))

optimizer_state = checkpoint.get("optimizer_state_dict")

print("\nOptimizer state exists:", optimizer_state is not None)

if optimizer_state is not None:

    print("Optimizer parameter groups:", len(optimizer_state["param_groups"]))

    for i, group in enumerate(optimizer_state["param_groups"]):
        print(f"\nParameter group {i}:")
        print("  learning rate:", group["lr"])
        print("  weight decay :", group["weight_decay"])
        print("  betas       :", group["betas"])
        print("  eps         :", group["eps"])

    print("\nOptimizer state entries:",
          len(optimizer_state["state"]))

In [ ]:
from PIL import Image
import torchvision.transforms.functional as TF
import torchvision.transforms as T
import torch
import random
import numpy as np

# ------------------------------------------------------------
# Load one training sample
# ------------------------------------------------------------

img_path, mask_path = train_pairs[0]

image = Image.open(img_path).convert("RGB")
mask = Image.open(mask_path).convert("L")

# 256 -> 512
image = image.resize((512, 512), Image.Resampling.BILINEAR)
mask = mask.resize((512, 512), Image.Resampling.NEAREST)

# ------------------------------------------------------------
# Strong Step 16A augmentation
# ------------------------------------------------------------

def strong_augment(image, mask):

    # Horizontal flip
    if random.random() < 0.5:
        image = TF.hflip(image)
        mask = TF.hflip(mask)

    # Random affine
    angle = random.uniform(-20, 20)
    scale = random.uniform(0.90, 1.10)

    max_translate = int(512 * 0.10)
    translate = [
        random.randint(-max_translate, max_translate),
        random.randint(-max_translate, max_translate)
    ]

    image = TF.affine(
        image,
        angle=angle,
        translate=translate,
        scale=scale,
        shear=[0.0, 0.0],
        interpolation=T.InterpolationMode.BILINEAR,
        fill=0
    )

    mask = TF.affine(
        mask,
        angle=angle,
        translate=translate,
        scale=scale,
        shear=[0.0, 0.0],
        interpolation=T.InterpolationMode.NEAREST,
        fill=0
    )

    # Photometric transformations — IMAGE ONLY
    brightness = random.uniform(0.75, 1.25)
    contrast = random.uniform(0.75, 1.25)
    saturation = random.uniform(0.70, 1.30)

    image = TF.adjust_brightness(image, brightness)
    image = TF.adjust_contrast(image, contrast)
    image = TF.adjust_saturation(image, saturation)

    # Gaussian blur
    if random.random() < 0.20:
        radius = random.uniform(0.3, 1.5)
        image = image.filter(
            ImageFilter.GaussianBlur(radius=radius)
        )

    return image, mask


# ------------------------------------------------------------
# Need ImageFilter for Gaussian blur
# ------------------------------------------------------------

from PIL import ImageFilter

# Re-run augmentation several times
print("Testing 10 augmented samples...\n")

for i in range(10):

    aug_img, aug_mask = strong_augment(image.copy(), mask.copy())

    img_tensor = TF.to_tensor(aug_img)

    mask_tensor = torch.from_numpy(
        np.array(aug_mask, dtype=np.uint8)
    ).float()

    mask_tensor = (mask_tensor > 0).float()

    print(
        f"Sample {i+1:2d} | "
        f"image range=({img_tensor.min():.3f}, {img_tensor.max():.3f}) | "
        f"mask values={torch.unique(mask_tensor).tolist()} | "
        f"positive fraction={mask_tensor.mean():.6f}"
    )

In [ ]:
# ============================================================
# STEP 16A — STRONG AUGMENTATION RECOVERY TRAINING
# Resume from epoch 3 -> continue through epoch 50
# ============================================================

import os
import json
import random
import time
from pathlib import Path

import numpy as np
from PIL import Image, ImageFilter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms.functional as TF
import torchvision.transforms as T

import segmentation_models_pytorch as smp


# ============================================================
# 1. REPRODUCIBILITY
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True


# ============================================================
# 2. PATHS
# ============================================================

INPUT_ROOT = Path(
    "/kaggle/input/datasets/"
    "shanumkhsharma/microplastic-step16a-recovery/"
    "kaggle_bundle"
)

OUTPUT_ROOT = Path("/kaggle/working/step16a_recovery")
CHECKPOINT_DIR = OUTPUT_ROOT / "checkpoints"

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

BEST_CKPT = CHECKPOINT_DIR / "beach_unet_resnet50_strong_aug_best_dice.pth"
RECOVERY_CKPT = CHECKPOINT_DIR / "beach_unet_resnet50_strong_aug_recovery_latest.pth"
HISTORY_FILE = OUTPUT_ROOT / "training_history.json"
CONFIG_FILE = OUTPUT_ROOT / "experiment_config.json"

SOURCE_CKPT = (
    INPUT_ROOT
    / "checkpoints"
    / "beach_unet_resnet50_strong_aug_best_dice.pth"
)


# ============================================================
# 3. DEVICE
# ============================================================

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )


# ============================================================
# 4. DATASET
# ============================================================

def get_pairs(split):

    img_dir = INPUT_ROOT / "data" / split / "images"
    mask_dir = INPUT_ROOT / "data" / split / "masks"

    images = sorted(img_dir.glob("*.png"))

    pairs = []

    for img_path in images:

        mask_path = mask_dir / img_path.name

        if mask_path.exists():
            pairs.append((img_path, mask_path))

    return pairs


train_pairs = get_pairs("train")
val_pairs = get_pairs("val")
test_pairs = get_pairs("test")

print("\nDataset:")
print("Train:", len(train_pairs))
print("Val  :", len(val_pairs))
print("Test :", len(test_pairs))


# ============================================================
# 5. STRONG AUGMENTATION
# ============================================================

def strong_augment(image, mask):

    # -------------------------
    # Horizontal flip
    # -------------------------
    if random.random() < 0.50:
        image = TF.hflip(image)
        mask = TF.hflip(mask)

    # -------------------------
    # Shared geometric transform
    # -------------------------
    angle = random.uniform(-20.0, 20.0)

    scale = random.uniform(0.90, 1.10)

    max_translate = int(512 * 0.10)

    translate = [
        random.randint(-max_translate, max_translate),
        random.randint(-max_translate, max_translate)
    ]

    image = TF.affine(
        image,
        angle=angle,
        translate=translate,
        scale=scale,
        shear=[0.0, 0.0],
        interpolation=T.InterpolationMode.BILINEAR,
        fill=0
    )

    mask = TF.affine(
        mask,
        angle=angle,
        translate=translate,
        scale=scale,
        shear=[0.0, 0.0],
        interpolation=T.InterpolationMode.NEAREST,
        fill=0
    )

    # -------------------------
    # Photometric transforms
    # IMAGE ONLY
    # -------------------------

    brightness = random.uniform(0.75, 1.25)
    contrast = random.uniform(0.75, 1.25)
    saturation = random.uniform(0.70, 1.30)

    image = TF.adjust_brightness(image, brightness)
    image = TF.adjust_contrast(image, contrast)
    image = TF.adjust_saturation(image, saturation)

    # -------------------------
    # Gaussian blur
    # -------------------------

    if random.random() < 0.20:

        radius = random.uniform(0.3, 1.5)

        image = image.filter(
            ImageFilter.GaussianBlur(radius=radius)
        )

    return image, mask


# ============================================================
# 6. DATASET CLASS
# ============================================================

class BeachSegmentationDataset(Dataset):

    def __init__(self, pairs, train=False):

        self.pairs = pairs
        self.train = train

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):

        img_path, mask_path = self.pairs[idx]

        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")

        # 256 -> 512
        image = image.resize(
            (512, 512),
            Image.Resampling.BILINEAR
        )

        mask = mask.resize(
            (512, 512),
            Image.Resampling.NEAREST
        )

        # Apply strong augmentation ONLY during training
        if self.train:

            image, mask = strong_augment(
                image,
                mask
            )

        # Image tensor [3,H,W], range 0-1
        image = TF.to_tensor(image)

        # Binary mask
        mask = torch.from_numpy(
            np.array(mask, dtype=np.uint8)
        ).float()

        mask = (mask > 0).float()

        mask = mask.unsqueeze(0)

        return image, mask


# ============================================================
# 7. DATA LOADERS
# ============================================================

train_ds = BeachSegmentationDataset(
    train_pairs,
    train=True
)

val_ds = BeachSegmentationDataset(
    val_pairs,
    train=False
)

test_ds = BeachSegmentationDataset(
    test_pairs,
    train=False
)

train_loader = DataLoader(
    train_ds,
    batch_size=8,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=8,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

test_loader = DataLoader(
    test_ds,
    batch_size=8,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

print("\nBatches:")
print("Train:", len(train_loader))
print("Val  :", len(val_loader))
print("Test :", len(test_loader))


# ============================================================
# 8. MODEL
# ============================================================

model = smp.Unet(
    encoder_name="resnet50",
    encoder_weights=None,
    in_channels=3,
    classes=1
).to(device)

print("\nModel parameters:",
      sum(p.numel() for p in model.parameters()))


# ============================================================
# 9. LOSS FUNCTIONS
# ============================================================

bce_loss = nn.BCEWithLogitsLoss()


def dice_loss(logits, targets, smooth=1.0):

    probs = torch.sigmoid(logits)

    probs = probs.contiguous().view(
        probs.size(0),
        -1
    )

    targets = targets.contiguous().view(
        targets.size(0),
        -1
    )

    intersection = (probs * targets).sum(dim=1)

    dice = (
        (2.0 * intersection + smooth)
        /
        (
            probs.sum(dim=1)
            +
            targets.sum(dim=1)
            +
            smooth
        )
    )

    return (1.0 - dice).mean()


def combined_loss(logits, targets):

    bce = bce_loss(logits, targets)
    dice = dice_loss(logits, targets)

    total = 0.5 * bce + 0.5 * dice

    return total, bce, dice


# ============================================================
# 10. OPTIMIZER
# ============================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)


# ============================================================
# 11. METRICS
# ============================================================

def calculate_metrics(logits, targets, threshold=0.5):

    probs = torch.sigmoid(logits)

    preds = (probs >= threshold).float()

    preds = preds.view(preds.size(0), -1)
    targets = targets.view(targets.size(0), -1)

    intersection = (preds * targets).sum(dim=1)

    pred_sum = preds.sum(dim=1)
    target_sum = targets.sum(dim=1)

    union = pred_sum + target_sum - intersection

    dice = (
        2 * intersection + 1e-7
    ) / (
        pred_sum + target_sum + 1e-7
    )

    iou = (
        intersection + 1e-7
    ) / (
        union + 1e-7
    )

    true_positive = intersection

    precision = (
        true_positive + 1e-7
    ) / (
        pred_sum + 1e-7
    )

    recall = (
        true_positive + 1e-7
    ) / (
        target_sum + 1e-7
    )

    pixel_accuracy = (
        (preds == targets).float().mean(dim=1)
    )

    return {
        "dice": dice.mean().item(),
        "iou": iou.mean().item(),
        "precision": precision.mean().item(),
        "recall": recall.mean().item(),
        "pixel_accuracy": pixel_accuracy.mean().item()
    }


# ============================================================
# 12. VALIDATION
# ============================================================

@torch.no_grad()
def evaluate(loader):

    model.eval()

    losses = []
    metric_values = []

    for images, masks in loader:

        images = images.to(
            device,
            non_blocking=True
        )

        masks = masks.to(
            device,
            non_blocking=True
        )

        logits = model(images)

        loss, _, _ = combined_loss(
            logits,
            masks
        )

        metrics = calculate_metrics(
            logits,
            masks
        )

        losses.append(loss.item())
        metric_values.append(metrics)

    result = {
        "loss": float(np.mean(losses)),
        "dice": float(np.mean(
            [x["dice"] for x in metric_values]
        )),
        "iou": float(np.mean(
            [x["iou"] for x in metric_values]
        )),
        "precision": float(np.mean(
            [x["precision"] for x in metric_values]
        )),
        "recall": float(np.mean(
            [x["recall"] for x in metric_values]
        )),
        "pixel_accuracy": float(np.mean(
            [x["pixel_accuracy"] for x in metric_values]
        ))
    }

    return result


# ============================================================
# 13. LOAD EPOCH-3 CHECKPOINT
# ============================================================

checkpoint = torch.load(
    SOURCE_CKPT,
    map_location=device
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

optimizer.load_state_dict(
    checkpoint["optimizer_state_dict"]
)

start_epoch = checkpoint["epoch"] + 1

best_val_dice = checkpoint["best_val_dice"]

history = checkpoint.get(
    "history",
    []
)

print("\nRecovered checkpoint:")
print("Starting epoch:", start_epoch)
print("Previous best Dice:", best_val_dice)
print("History entries:", len(history))


# ============================================================
# 14. IF A KAGGLE RECOVERY CHECKPOINT EXISTS,
#     PREFER THE LATEST ONE
# ============================================================

if RECOVERY_CKPT.exists():

    print("\nFound existing Kaggle recovery checkpoint.")

    recovery = torch.load(
        RECOVERY_CKPT,
        map_location=device
    )

    model.load_state_dict(
        recovery["model_state_dict"]
    )

    optimizer.load_state_dict(
        recovery["optimizer_state_dict"]
    )

    start_epoch = recovery["epoch"] + 1

    best_val_dice = recovery["best_val_dice"]

    history = recovery["history"]

    print(
        "Resuming from recovery epoch:",
        recovery["epoch"]
    )

    print(
        "Best Dice:",
        best_val_dice
    )


# ============================================================
# 15. TRAINING CONFIG
# ============================================================

MAX_EPOCHS = 50
PATIENCE = 10

config = {
    "experiment": "beach_unet_resnet50_strong_augmentation",
    "seed": SEED,
    "input_size": [512, 512],
    "batch_size": 8,
    "max_epochs": MAX_EPOCHS,
    "patience": PATIENCE,
    "optimizer": "AdamW",
    "learning_rate": 1e-4,
    "weight_decay": 1e-4,
    "loss": "0.5 BCEWithLogits + 0.5 Dice",
    "encoder": "ResNet50",
    "encoder_pretraining": "ImageNet",
    "train_images": len(train_pairs),
    "val_images": len(val_pairs),
    "test_images": len(test_pairs),
    "wwtp_used": False,
    "augmentation": {
        "horizontal_flip_p": 0.5,
        "rotation_degrees": 20,
        "scale": [0.90, 1.10],
        "translation_fraction": 0.10,
        "brightness": [0.75, 1.25],
        "contrast": [0.75, 1.25],
        "saturation": [0.70, 1.30],
        "gaussian_blur_p": 0.20,
        "gaussian_blur_radius": [0.3, 1.5]
    }
}

with open(CONFIG_FILE, "w") as f:
    json.dump(config, f, indent=2)


# ============================================================
# 16. MIXED PRECISION
# ============================================================

scaler = torch.cuda.amp.GradScaler(
    enabled=(device.type == "cuda")
)


# ============================================================
# 17. TRAINING LOOP
# ============================================================

epochs_without_improvement = 0

print("\n" + "=" * 70)
print("STARTING STEP 16A RECOVERY TRAINING")
print("=" * 70)

print(
    f"Epoch range: {start_epoch} -> {MAX_EPOCHS}"
)

print(
    f"Current best validation Dice: {best_val_dice:.6f}"
)

print("=" * 70)


for epoch in range(start_epoch, MAX_EPOCHS + 1):

    epoch_start = time.time()

    model.train()

    train_losses = []
    train_dice_values = []

    for images, masks in train_loader:

        images = images.to(
            device,
            non_blocking=True
        )

        masks = masks.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.cuda.amp.autocast(
            enabled=(device.type == "cuda")
        ):

            logits = model(images)

            loss, bce, dice = combined_loss(
                logits,
                masks
            )

        scaler.scale(loss).backward()

        scaler.step(optimizer)

        scaler.update()

        train_losses.append(
            loss.detach().item()
        )

        with torch.no_grad():

            metrics = calculate_metrics(
                logits.detach(),
                masks
            )

            train_dice_values.append(
                metrics["dice"]
            )


    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------

    val_result = evaluate(val_loader)

    train_loss = float(
        np.mean(train_losses)
    )

    train_dice = float(
        np.mean(train_dice_values)
    )

    val_loss = val_result["loss"]
    val_dice = val_result["dice"]

    epoch_time = time.time() - epoch_start


    # --------------------------------------------------------
    # History
    # --------------------------------------------------------

    record = {
        "epoch": epoch,
        "train_loss": train_loss,
        "train_dice": train_dice,
        "val_loss": val_loss,
        "val_dice": val_dice,
        "val_iou": val_result["iou"],
        "val_precision": val_result["precision"],
        "val_recall": val_result["recall"],
        "val_pixel_accuracy": val_result["pixel_accuracy"],
        "learning_rate": optimizer.param_groups[0]["lr"],
        "epoch_time_seconds": epoch_time
    }

    history.append(record)


    # --------------------------------------------------------
    # Best model
    # --------------------------------------------------------

    is_best = val_dice > best_val_dice

    if is_best:

        best_val_dice = val_dice

        epochs_without_improvement = 0

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_val_dice": best_val_dice,
                "history": history,
                "experiment": "step16a_strong_augmentation",
                "seed": SEED
            },
            BEST_CKPT
        )

    else:

        epochs_without_improvement += 1


    # --------------------------------------------------------
    # ALWAYS save recovery checkpoint
    # --------------------------------------------------------

    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_val_dice": best_val_dice,
            "history": history,
            "experiment": "step16a_strong_augmentation",
            "seed": SEED
        },
        RECOVERY_CKPT
    )


    # --------------------------------------------------------
    # Save history
    # --------------------------------------------------------

    with open(HISTORY_FILE, "w") as f:
        json.dump(history, f, indent=2)


    # --------------------------------------------------------
    # Console output
    # --------------------------------------------------------

    marker = "  <-- BEST" if is_best else ""

    print(
        f"Epoch {epoch:02d}/{MAX_EPOCHS} | "
        f"train_loss={train_loss:.4f} | "
        f"train_dice={train_dice:.4f} | "
        f"val_loss={val_loss:.4f} | "
        f"val_dice={val_dice:.4f} | "
        f"val_iou={val_result['iou']:.4f} | "
        f"val_prec={val_result['precision']:.4f} | "
        f"val_rec={val_result['recall']:.4f} | "
        f"time={epoch_time/60:.1f} min"
        f"{marker}"
    )


    # --------------------------------------------------------
    # Early stopping
    # --------------------------------------------------------

    if epochs_without_improvement >= PATIENCE:

        print(
            f"\nEarly stopping triggered after "
            f"{PATIENCE} epochs without improvement."
        )

        break


# ============================================================
# 18. FINAL STATUS
# ============================================================

print("\n" + "=" * 70)
print("STEP 16A TRAINING FINISHED")
print("=" * 70)

print("Best validation Dice:", best_val_dice)
print("Epochs completed:", history[-1]["epoch"])

print("\nBest checkpoint:")
print(BEST_CKPT)

print("\nLatest recovery checkpoint:")
print(RECOVERY_CKPT)

print("\nHistory:")
print(HISTORY_FILE)

print("=" * 70)

In [ ]:
# ============================================================
# STEP 16A — UNSEEN WWTP EXTERNAL EVALUATION
# Best validation-Dice checkpoint only
# ============================================================

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import segmentation_models_pytorch as smp


# ============================================================
# 1. PATHS
# ============================================================

INPUT_ROOT = Path(
    "/kaggle/input/datasets/"
    "shanumkhsharma/microplastic-step16a-recovery/"
    "kaggle_bundle"
)

# WWTP is NOT part of the recovery bundle.
# Locate it if it was added as a separate Kaggle dataset.

SEARCH_ROOTS = [
    Path("/kaggle/input"),
]

wwtp_candidates = []

for root in SEARCH_ROOTS:
    if root.exists():
        for p in root.rglob("wwtp_canonical_manifest.csv"):
            wwtp_candidates.append(p)

print("WWTP manifest candidates:")
for p in wwtp_candidates:
    print(p)

if not wwtp_candidates:
    raise FileNotFoundError(
        "WWTP canonical manifest was not found under /kaggle/input. "
        "Add the WWTP dataset/manifest to this Kaggle notebook before continuing."
    )

WWTP_MANIFEST = wwtp_candidates[0]

print("\nUsing WWTP manifest:")
print(WWTP_MANIFEST)


# ============================================================
# 2. LOAD MANIFEST
# ============================================================

wwtp_df = pd.read_csv(WWTP_MANIFEST)

print("\nWWTP manifest shape:", wwtp_df.shape)
print("Columns:")
print(list(wwtp_df.columns))


# ============================================================
# 3. DISPLAY RELEVANT INFORMATION
# ============================================================

for col in [
    "image_path",
    "mask_path",
    "split",
    "target_status",
    "evaluation_eligible"
]:
    if col in wwtp_df.columns:
        print(
            f"\n{col}:",
            wwtp_df[col].value_counts(dropna=False).to_dict()
        )


# ============================================================
# 4. DETERMINE IMAGE / MASK PATHS
# ============================================================

def resolve_path(path_value):

    p = Path(str(path_value))

    if p.exists():
        return p

    # Try relative to manifest directory
    p2 = WWTP_MANIFEST.parent / p

    if p2.exists():
        return p2

    # Search by filename
    matches = list(
        WWTP_MANIFEST.parent.rglob(p.name)
    )

    if matches:
        return matches[0]

    raise FileNotFoundError(
        f"Could not resolve path: {path_value}"
    )


# ============================================================
# 5. PRIMARY EVALUATION SELECTION
# ============================================================

# Primary:
#   annotated_positive = 2004 images
#
# Secondary:
#   annotation_empty = 6 images
#
# Original WWTP train/validation/test labels are retained,
# but NONE are used for model selection or training.

if "target_status" in wwtp_df.columns:

    primary_df = wwtp_df[
        wwtp_df["target_status"] == "annotated_positive"
    ].copy()

    secondary_df = wwtp_df[
        wwtp_df["target_status"] == "annotation_empty"
    ].copy()

else:
    raise ValueError(
        "Manifest does not contain target_status."
    )


print("\nPrimary WWTP images:", len(primary_df))
print("Secondary empty-mask images:", len(secondary_df))


# ============================================================
# 6. DEVICE
# ============================================================

device = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)

print("\nDevice:", device)

if device.type == "cuda":
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )


# ============================================================
# 7. LOAD BEST STEP-16A CHECKPOINT
# ============================================================

BEST_CKPT = (
    Path("/kaggle/working/step16a_recovery")
    / "checkpoints"
    / "beach_unet_resnet50_strong_aug_best_dice.pth"
)

if not BEST_CKPT.exists():
    raise FileNotFoundError(
        f"Best Step-16A checkpoint not found: {BEST_CKPT}"
    )

checkpoint = torch.load(
    BEST_CKPT,
    map_location=device
)

print("\nCheckpoint:")
print("Epoch:", checkpoint["epoch"])
print("Best validation Dice:",
      checkpoint["best_val_dice"])


# ============================================================
# 8. RECONSTRUCT MODEL
# ============================================================

model = smp.Unet(
    encoder_name="resnet50",
    encoder_weights=None,
    in_channels=3,
    classes=1
).to(device)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

print(
    "Parameters:",
    sum(p.numel() for p in model.parameters())
)


# ============================================================
# 9. WWTP TILING PARAMETERS
# ============================================================

TILE_SIZE = 512
STRIDE = 384
OVERLAP = TILE_SIZE - STRIDE

print("\nTiling:")
print("Tile:", TILE_SIZE)
print("Stride:", STRIDE)
print("Overlap:", OVERLAP)
print(
    "Overlap percentage:",
    100 * OVERLAP / TILE_SIZE,
    "%"
)


# ============================================================
# 10. TILE POSITIONS
# ============================================================

def get_positions(length):

    positions = list(
        range(
            0,
            length - TILE_SIZE + 1,
            STRIDE
        )
    )

    final_position = length - TILE_SIZE

    if positions[-1] != final_position:
        positions.append(final_position)

    return positions


# ============================================================
# 11. IMAGE PREDICTION
# ============================================================

@torch.no_grad()
def predict_wwtp(image):

    image = np.asarray(image)

    H, W = image.shape[:2]

    ys = get_positions(H)
    xs = get_positions(W)

    probability_sum = np.zeros(
        (H, W),
        dtype=np.float32
    )

    coverage = np.zeros(
        (H, W),
        dtype=np.float32
    )

    for y in ys:

        for x in xs:

            tile = image[
                y:y + TILE_SIZE,
                x:x + TILE_SIZE
            ]

            tile_pil = Image.fromarray(
                tile
            )

            tile_tensor = (
                torch.from_numpy(
                    np.asarray(
                        tile_pil,
                        dtype=np.float32
                    ) / 255.0
                )
                .permute(2, 0, 1)
                .unsqueeze(0)
                .to(device)
            )

            logits = model(
                tile_tensor
            )

            probabilities = torch.sigmoid(
                logits
            )[0, 0].cpu().numpy()

            probability_sum[
                y:y + TILE_SIZE,
                x:x + TILE_SIZE
            ] += probabilities

            coverage[
                y:y + TILE_SIZE,
                x:x + TILE_SIZE
            ] += 1.0

    probabilities = (
        probability_sum /
        np.maximum(coverage, 1e-8)
    )

    return probabilities


# ============================================================
# 12. METRICS
# ============================================================

def segmentation_metrics(
    prediction,
    target
):

    prediction = prediction.astype(bool)
    target = target.astype(bool)

    intersection = np.logical_and(
        prediction,
        target
    ).sum()

    pred_area = prediction.sum()
    target_area = target.sum()

    union = np.logical_or(
        prediction,
        target
    ).sum()

    dice = (
        2.0 * intersection
        /
        max(
            pred_area + target_area,
            1
        )
    )

    iou = (
        intersection
        /
        max(union, 1)
    )

    precision = (
        intersection
        /
        max(pred_area, 1)
    )

    recall = (
        intersection
        /
        max(target_area, 1)
    )

    pixel_accuracy = (
        prediction == target
    ).mean()

    return {
        "dice": float(dice),
        "iou": float(iou),
        "precision": float(precision),
        "recall": float(recall),
        "pixel_accuracy": float(pixel_accuracy),
        "gt_foreground_fraction": float(
            target.mean()
        ),
        "pred_foreground_fraction": float(
            prediction.mean()
        )
    }


# ============================================================
# 13. EVALUATE ONE IMAGE
# ============================================================

def evaluate_row(row):

    image_path = resolve_path(
        row["image_path"]
    )

    mask_path = resolve_path(
        row["mask_path"]
    )

    image = Image.open(
        image_path
    ).convert("RGB")

    mask = np.asarray(
        Image.open(mask_path).convert("L")
    )

    target = mask > 0

    H, W = np.asarray(image).shape[:2]

    probabilities = predict_wwtp(
        image
    )

    prediction = probabilities >= 0.5

    metrics = segmentation_metrics(
        prediction,
        target
    )

    result = {
        "sample_id": row.get(
            "sample_id",
            image_path.stem
        ),
        "original_split": row.get(
            "split",
            ""
        ),
        "image_path": str(image_path),
        "mask_path": str(mask_path),
        "height": H,
        "width": W,
        **metrics
    }

    return result


# ============================================================
# 14. RUN PRIMARY EVALUATION
# ============================================================

results = []

start_time = time.time()

print("\nStarting primary WWTP evaluation...")
print("Images:", len(primary_df))

for i, (_, row) in enumerate(
    primary_df.iterrows(),
    start=1
):

    result = evaluate_row(row)

    results.append(result)

    if (
        i == 1
        or i % 25 == 0
        or i == len(primary_df)
    ):

        elapsed = time.time() - start_time

        print(
            f"{i:4d}/{len(primary_df)} | "
            f"elapsed={elapsed/60:.1f} min | "
            f"Dice={result['dice']:.4f}"
        )


# ============================================================
# 15. RESULTS DATAFRAME
# ============================================================

results_df = pd.DataFrame(
    results
)

print("\nEvaluation complete.")
print(results_df.shape)


# ============================================================
# 16. SUMMARY
# ============================================================

metric_names = [
    "dice",
    "iou",
    "precision",
    "recall",
    "pixel_accuracy"
]

summary = {}

for metric in metric_names:

    summary[metric] = {
        "mean": float(
            results_df[metric].mean()
        ),
        "std": float(
            results_df[metric].std()
        ),
        "median": float(
            results_df[metric].median()
        )
    }

summary["n_primary"] = len(
    results_df
)

summary["checkpoint_epoch"] = int(
    checkpoint["epoch"]
)

summary["best_validation_dice"] = float(
    checkpoint["best_val_dice"]
)

summary["tile_size"] = TILE_SIZE
summary["stride"] = STRIDE
summary["overlap"] = OVERLAP
summary["threshold"] = 0.5


# ============================================================
# 17. SAVE RESULTS
# ============================================================

RESULT_ROOT = (
    Path("/kaggle/working")
    / "step16a_wwtp_external_evaluation"
)

RESULT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

results_file = (
    RESULT_ROOT
    / "wwtp_step16a_per_image_results.csv"
)

summary_file = (
    RESULT_ROOT
    / "wwtp_step16a_summary.json"
)

results_df.to_csv(
    results_file,
    index=False
)

with open(summary_file, "w") as f:
    json.dump(
        summary,
        f,
        indent=2
    )


# ============================================================
# 18. PRINT FINAL METRICS
# ============================================================

print("\n" + "=" * 70)
print("STEP 16A — WWTP EXTERNAL RESULTS")
print("=" * 70)

print(
    "Checkpoint epoch:",
    checkpoint["epoch"]
)

print(
    "Best Beach validation Dice:",
    checkpoint["best_val_dice"]
)

print(
    "Primary images:",
    len(results_df)
)

for metric in metric_names:

    s = summary[metric]

    print(
        f"{metric:17s}: "
        f"{s['mean']:.4f} ± {s['std']:.4f} "
        f"(median {s['median']:.4f})"
    )

print("\nResults:")
print(results_file)

print("\nSummary:")
print(summary_file)

print("=" * 70)

After adding kaggle_microplastic_bundle to the kaggle notebook and trying to resume form step 16

In [ ]:
from pathlib import Path
import zipfile

INPUT_ROOT = Path("/kaggle/input")

print("Kaggle input datasets:")
for p in INPUT_ROOT.iterdir():
    print(" -", p)

# Find the uploaded ZIP
zips = list(INPUT_ROOT.rglob("kaggle_microplastic_bundle.zip"))

print("\nZIP files found:")
for z in zips:
    print(" -", z)

if not zips:
    raise FileNotFoundError("kaggle_microplastic_bundle.zip was not found.")

ZIP_PATH = zips[0]

print("\nUsing:")
print(ZIP_PATH)

# Inspect ZIP contents without extracting
with zipfile.ZipFile(ZIP_PATH, "r") as z:
    names = z.namelist()

print("\nNumber of files in ZIP:", len(names))

print("\nFirst 30 files:")
for name in names[:30]:
    print(name)

print("\nCheckpoint present:",
      any("beach_unet_resnet50_strong_aug_best_dice.pth" in n for n in names))

print("Beach data present:",
      any(n.startswith("data/beach/") for n in names))

print("WWTP data present:",
      any(n.startswith("data/wwtp/") for n in names))

print("Manifests present:",
      any(n.startswith("manifests/") for n in names))

In [ ]:
from pathlib import Path

ROOT = Path("/kaggle/input/datasets")

print("ROOT:", ROOT)
print()

# Check main directories
for name in ["checkpoints", "data", "manifests"]:
    p = ROOT / name
    print(f"{name:12s} exists = {p.exists()}")

# Check checkpoint
ckpt = ROOT / "checkpoints" / "beach_unet_resnet50_strong_aug_best_dice.pth"

print("\nCheckpoint:")
print("exists:", ckpt.exists())

if ckpt.exists():
    print("size_MB:", round(ckpt.stat().st_size / (1024**2), 2))

# Check Beach
beach = ROOT / "data" / "beach"
print("\nBeach:")
print("exists:", beach.exists())

if beach.exists():
    print("contents:")
    for p in sorted(beach.iterdir()):
        print(" ", p.name)

# Check WWTP
wwtp = ROOT / "data" / "wwtp"
print("\nWWTP:")
print("exists:", wwtp.exists())

if wwtp.exists():
    print("contents:")
    for p in sorted(wwtp.iterdir()):
        print(" ", p.name)

# Check manifests
print("\nManifests:")
for p in sorted((ROOT / "manifests").glob("*")):
    print(" ", p.name)

In [ ]:
from pathlib import Path

INPUT = Path("/kaggle/input")

print("Everything directly under /kaggle/input:\n")

for p in sorted(INPUT.iterdir()):
    print(p)

print("\n\nDirectories containing our project files:\n")

for p in sorted(INPUT.rglob("*")):
    if p.is_dir() and p.name in ["checkpoints", "data", "manifests"]:
        print(p)

In [ ]:
from pathlib import Path

ROOT = Path(
    "/kaggle/input/datasets/shanumkhsharma/kaggle-microplastic-bundle"
)

print("ROOT:", ROOT)
print()

# ------------------------------------------------------------
# Check top-level contents
# ------------------------------------------------------------
print("Top-level:")
for p in sorted(ROOT.iterdir()):
    print(" ", p.name)

# ------------------------------------------------------------
# Check checkpoint
# ------------------------------------------------------------
ckpt = ROOT / "checkpoints" / "beach_unet_resnet50_strong_aug_best_dice.pth"

print("\nCheckpoint:")
print("  exists:", ckpt.exists())

if ckpt.exists():
    print("  size_MB:", round(ckpt.stat().st_size / (1024**2), 2))

# ------------------------------------------------------------
# Check Beach
# ------------------------------------------------------------
beach = ROOT / "data" / "beach"

print("\nBeach:")
print("  exists:", beach.exists())

if beach.exists():
    for p in sorted(beach.iterdir()):
        print(" ", p.name)

# ------------------------------------------------------------
# Check WWTP
# ------------------------------------------------------------
wwtp = ROOT / "data" / "wwtp"

print("\nWWTP:")
print("  exists:", wwtp.exists())

if wwtp.exists():
    for p in sorted(wwtp.iterdir()):
        print(" ", p.name)

# ------------------------------------------------------------
# Check manifests
# ------------------------------------------------------------
manifests = ROOT / "manifests"

print("\nManifests:")
if manifests.exists():
    for p in sorted(manifests.iterdir()):
        print(" ", p.name)

In [ ]:
from pathlib import Path

ROOT = Path(
    "/kaggle/input/datasets/shanumkhsharma/kaggle-microplastic-bundle"
)

DATA = ROOT / "data"

print("Everything under data/:\n")

for p in sorted(DATA.rglob("*")):
    if p.is_dir():
        print(p)

In [ ]:
from pathlib import Path

ROOT = Path(
    "/kaggle/input/datasets/shanumkhsharma/microplastic-step16a-recovery/kaggle_bundle"
)

print("ROOT:", ROOT)

for split in ["train", "val", "test"]:
    img_dir = ROOT / "data" / split / "images"
    mask_dir = ROOT / "data" / split / "masks"

    print(
        f"{split:5s} | "
        f"images = {len(list(img_dir.glob('*'))):4d} | "
        f"masks  = {len(list(mask_dir.glob('*'))):4d}"
    )

ckpt = ROOT / "checkpoints" / "beach_unet_resnet50_strong_aug_best_dice.pth"

print("\nCheckpoint:")
print("exists:", ckpt.exists())

if ckpt.exists():
    print("size_MB:", round(ckpt.stat().st_size / (1024**2), 2))

manifest = ROOT / "portable_manifest.csv"

print("\nPortable manifest:")
print("exists:", manifest.exists())

In [ ]:
import torch
from pathlib import Path

ROOT = Path(
    "/kaggle/input/datasets/shanumkhsharma/microplastic-step16a-recovery/kaggle_bundle"
)

CKPT = ROOT / "checkpoints" / "beach_unet_resnet50_strong_aug_best_dice.pth"

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\nCheckpoint:", CKPT)
print("Exists:", CKPT.exists())
print("Size MB:", round(CKPT.stat().st_size / (1024**2), 2))

checkpoint = torch.load(
    CKPT,
    map_location="cpu"
)

print("\nCheckpoint keys:")
print(list(checkpoint.keys()))

print("\nCheckpoint metadata:")

for key in [
    "epoch",
    "best_val_dice",
    "experiment",
    "seed"
]:
    if key in checkpoint:
        print(f"{key}: {checkpoint[key]}")

if "history" in checkpoint:
    print("\nHistory entries:", len(checkpoint["history"]))

In [ ]:
pip install segmentation_models_pytorch

In [ ]:
import torch
import segmentation_models_pytorch as smp
from pathlib import Path

# ============================================================
# PATH
# ============================================================

ROOT = Path(
    "/kaggle/input/datasets/shanumkhsharma/microplastic-step16a-recovery/kaggle_bundle"
)

CKPT = (
    ROOT
    / "checkpoints"
    / "beach_unet_resnet50_strong_aug_best_dice.pth"
)

# ============================================================
# DEVICE
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# ============================================================
# RECREATE EXACT MODEL
# ============================================================

model = smp.Unet(
    encoder_name="resnet50",
    encoder_weights=None,   # checkpoint already contains weights
    in_channels=3,
    classes=1
).to(device)

# ============================================================
# LOAD CHECKPOINT
# ============================================================

checkpoint = torch.load(
    CKPT,
    map_location=device
)

missing, unexpected = model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=False
)

print("\nCheckpoint loaded.")

print("Missing keys:", len(missing))
print("Unexpected keys:", len(unexpected))

if missing:
    print("\nMissing:")
    print(missing[:10])

if unexpected:
    print("\nUnexpected:")
    print(unexpected[:10])

# ============================================================
# MODEL SUMMARY
# ============================================================

total_params = sum(
    p.numel() for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("\nModel parameters:")
print("Total:", total_params)
print("Trainable:", trainable_params)

# ============================================================
# CHECK A PARAMETER IS FINITE
# ============================================================

all_finite = True

for name, param in model.named_parameters():
    if not torch.isfinite(param).all():
        print("NON-FINITE PARAMETER:", name)
        all_finite = False
        break

print("\nAll parameters finite:", all_finite)

# ============================================================
# CHECK CHECKPOINT METADATA
# ============================================================

print("\nCheckpoint metadata:")
print("Epoch:", checkpoint["epoch"])
print("Best validation Dice:", checkpoint["best_val_dice"])
print("Experiment:", checkpoint["experiment"])
print("Seed:", checkpoint["seed"])

In [ ]:
import torch
from pathlib import Path

ROOT = Path(
    "/kaggle/input/datasets/shanumkhsharma/microplastic-step16a-recovery/kaggle_bundle"
)

CKPT = (
    ROOT
    / "checkpoints"
    / "beach_unet_resnet50_strong_aug_best_dice.pth"
)

checkpoint = torch.load(
    CKPT,
    map_location="cuda"
)

# ------------------------------------------------------------
# Recreate optimizer
# ------------------------------------------------------------

import segmentation_models_pytorch as smp

model = smp.Unet(
    encoder_name="resnet50",
    encoder_weights=None,
    in_channels=3,
    classes=1
).cuda()

model.load_state_dict(
    checkpoint["model_state_dict"]
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

optimizer.load_state_dict(
    checkpoint["optimizer_state_dict"]
)

# ------------------------------------------------------------
# Inspect optimizer state
# ------------------------------------------------------------

print("Optimizer loaded successfully.")

print("\nOptimizer configuration:")

for group in optimizer.param_groups:
    print("learning_rate:", group["lr"])
    print("weight_decay:", group["weight_decay"])
    print("betas:", group["betas"])
    print("eps:", group["eps"])

# ------------------------------------------------------------
# Count optimizer state
# ------------------------------------------------------------

state_count = len(optimizer.state)

print("\nOptimizer state entries:", state_count)

# ------------------------------------------------------------
# Recovery position
# ------------------------------------------------------------

start_epoch = checkpoint["epoch"] + 1

print("\nRecovery:")
print("Last completed epoch:", checkpoint["epoch"])
print("Next epoch:", start_epoch)
print("Best validation Dice:", checkpoint["best_val_dice"])

# ------------------------------------------------------------
# Check history
# ------------------------------------------------------------

history = checkpoint["history"]

print("\nTraining history:")
for h in history:
    print(
        f"Epoch {h['epoch']}: "
        f"train_loss={h['train_loss']:.6f}, "
        f"val_loss={h['val_loss']:.6f}, "
        f"val_dice={h['val_dice']:.6f}, "
        f"val_iou={h['val_iou']:.6f}"
    )

In [ ]:
import torch
import torch.nn as nn
from pathlib import Path
from PIL import Image
import numpy as np
import segmentation_models_pytorch as smp

# ============================================================
# PATHS
# ============================================================

ROOT = Path(
    "/kaggle/input/datasets/shanumkhsharma/microplastic-step16a-recovery/kaggle_bundle"
)

TRAIN_IMG = ROOT / "data" / "train" / "images"
TRAIN_MASK = ROOT / "data" / "train" / "masks"

CKPT = ROOT / "checkpoints" / "beach_unet_resnet50_strong_aug_best_dice.pth"

# ============================================================
# DEVICE
# ============================================================

device = torch.device("cuda")

print("Device:", device)
print("GPU:", torch.cuda.get_device_name(0))

# ============================================================
# FIND ONE TRAINING SAMPLE
# ============================================================

images = sorted(TRAIN_IMG.glob("*"))

if len(images) == 0:
    raise RuntimeError("No training images found.")

image_path = images[0]
mask_path = TRAIN_MASK / image_path.name

print("\nSample:")
print("Image:", image_path.name)
print("Mask exists:", mask_path.exists())

# ============================================================
# LOAD IMAGE
# ============================================================

image = Image.open(image_path).convert("RGB")
mask = Image.open(mask_path).convert("L")

print("\nOriginal:")
print("Image size:", image.size)
print("Mask size:", mask.size)

# ============================================================
# SAME 512x512 PREPROCESSING
# ============================================================

image = image.resize(
    (512, 512),
    Image.Resampling.BILINEAR
)

mask = mask.resize(
    (512, 512),
    Image.Resampling.NEAREST
)

image_np = np.asarray(image).astype(np.float32) / 255.0
mask_np = np.asarray(mask)

# Convert mask to binary
mask_np = (mask_np > 0).astype(np.float32)

image_tensor = torch.from_numpy(
    image_np.transpose(2, 0, 1)
).unsqueeze(0).to(device)

mask_tensor = torch.from_numpy(
    mask_np
).unsqueeze(0).unsqueeze(0).to(device)

print("\nTensor:")
print("Image:", image_tensor.shape, image_tensor.dtype)
print("Mask :", mask_tensor.shape, mask_tensor.dtype)
print("Image min/max:", image_tensor.min().item(), image_tensor.max().item())
print("Mask values:", torch.unique(mask_tensor).detach().cpu().numpy())

# ============================================================
# LOAD MODEL + CHECKPOINT
# ============================================================

model = smp.Unet(
    encoder_name="resnet50",
    encoder_weights=None,
    in_channels=3,
    classes=1
).to(device)

checkpoint = torch.load(
    CKPT,
    map_location=device
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.train()

# ============================================================
# FORWARD PASS
# ============================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

optimizer.load_state_dict(
    checkpoint["optimizer_state_dict"]
)

optimizer.zero_grad(set_to_none=True)

logits = model(image_tensor)

print("\nForward:")
print("Logits:", logits.shape)
print("Logits dtype:", logits.dtype)
print("Logits finite:", torch.isfinite(logits).all().item())

# ============================================================
# LOSS
# ============================================================

bce = nn.BCEWithLogitsLoss()

bce_loss = bce(
    logits,
    mask_tensor
)

probs = torch.sigmoid(logits)

intersection = (probs * mask_tensor).sum()

dice = (
    (2.0 * intersection + 1e-6)
    /
    (probs.sum() + mask_tensor.sum() + 1e-6)
)

dice_loss = 1.0 - dice

loss = 0.5 * bce_loss + 0.5 * dice_loss

print("\nLoss:")
print("BCE:", bce_loss.item())
print("Dice loss:", dice_loss.item())
print("Total:", loss.item())

# ============================================================
# BACKWARD
# ============================================================

loss.backward()

grad_count = 0
finite_gradients = True

for p in model.parameters():
    if p.grad is not None:
        grad_count += 1
        if not torch.isfinite(p.grad).all():
            finite_gradients = False

print("\nBackward:")
print("Parameters with gradients:", grad_count)
print("All gradients finite:", finite_gradients)

# ============================================================
# DO NOT UPDATE THE MODEL
# ============================================================

optimizer.zero_grad(set_to_none=True)

print("\n============================================")
print("SINGLE-BATCH RESUME TEST PASSED")
print("NO OPTIMIZER UPDATE WAS PERFORMED")
print("============================================")

In [ ]:
# ============================================================
# STEP16A — VERIFY STRONG AUGMENTATION PIPELINE
# No training / no optimizer update
# ============================================================

import random
import numpy as np
import torch
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.transforms.functional as TF

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

ROOT = Path(
    "/kaggle/input/datasets/shanumkhsharma/microplastic-step16a-recovery/kaggle_bundle"
)

TRAIN_IMG = ROOT / "data" / "train" / "images"
TRAIN_MASK = ROOT / "data" / "train" / "masks"

print("Training images:", len(list(TRAIN_IMG.glob("*"))))
print("Training masks :", len(list(TRAIN_MASK.glob("*"))))

# ------------------------------------------------------------
# Strong augmentation used for Step16A
#
# hflip             0.5
# rotation          +/-20 degrees
# scale             0.90 - 1.10
# translation        +/-10%
# brightness        0.75 - 1.25
# contrast          0.75 - 1.25
# saturation        0.70 - 1.30
# Gaussian blur     p=0.20, radius 0.3 - 1.5
#
# Image-only operations are applied to image.
# Geometric operations are applied identically to image + mask.
# ------------------------------------------------------------

class StrongAugmentation:
    def __init__(self):
        self.color = T.ColorJitter(
            brightness=(0.75, 1.25),
            contrast=(0.75, 1.25),
            saturation=(0.70, 1.30),
        )

    def __call__(self, image, mask):

        # Horizontal flip
        if random.random() < 0.5:
            image = TF.hflip(image)
            mask = TF.hflip(mask)

        # Random affine
        angle = random.uniform(-20, 20)

        scale = random.uniform(0.90, 1.10)

        max_dx = int(0.10 * image.width)
        max_dy = int(0.10 * image.height)

        translate = [
            random.randint(-max_dx, max_dx),
            random.randint(-max_dy, max_dy),
        ]

        image = TF.affine(
            image,
            angle=angle,
            translate=translate,
            scale=scale,
            shear=[0.0, 0.0],
            interpolation=T.InterpolationMode.BILINEAR,
            fill=0,
        )

        mask = TF.affine(
            mask,
            angle=angle,
            translate=translate,
            scale=scale,
            shear=[0.0, 0.0],
            interpolation=T.InterpolationMode.NEAREST,
            fill=0,
        )

        # Image-only colour augmentation
        image = self.color(image)

        # Gaussian blur
        if random.random() < 0.20:
            radius = random.uniform(0.3, 1.5)

            # torchvision GaussianBlur expects kernel size.
            # Use a valid odd kernel corresponding approximately
            # to the requested radius.
            kernel = max(3, int(round(radius * 4)) * 2 + 1)

            image = TF.gaussian_blur(
                image,
                kernel_size=[kernel, kernel],
                sigma=[radius, radius],
            )

        return image, mask


# ------------------------------------------------------------
# Dataset
# ------------------------------------------------------------

class BeachDataset(Dataset):

    def __init__(self, image_dir, mask_dir, augment=False):

        self.image_dir = Path(image_dir)
        self.mask_dir = Path(mask_dir)

        self.images = sorted(
            [
                p for p in self.image_dir.iterdir()
                if p.is_file()
            ]
        )

        self.augment = augment

        if augment:
            self.transform = StrongAugmentation()

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):

        image_path = self.images[idx]
        mask_path = self.mask_dir / image_path.name

        image = Image.open(image_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")

        # Resize exactly as before
        image = image.resize(
            (512, 512),
            Image.Resampling.BILINEAR
        )

        mask = mask.resize(
            (512, 512),
            Image.Resampling.NEAREST
        )

        # Apply strong augmentation
        if self.augment:
            image, mask = self.transform(image, mask)

        # Convert image to tensor [3,H,W], [0,1]
        image = TF.to_tensor(image)

        # Convert mask to binary [1,H,W]
        mask = torch.from_numpy(
            (np.asarray(mask) > 0).astype(np.float32)
        ).unsqueeze(0)

        return image, mask


# ------------------------------------------------------------
# DataLoader
# ------------------------------------------------------------

dataset = BeachDataset(
    TRAIN_IMG,
    TRAIN_MASK,
    augment=True
)

loader = DataLoader(
    dataset,
    batch_size=8,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
)

# ------------------------------------------------------------
# Get ONE augmented batch
# ------------------------------------------------------------

images, masks = next(iter(loader))

print("\n============================================")
print("AUGMENTED BATCH")
print("============================================")

print("Images:", images.shape)
print("Masks :", masks.shape)

print("Image dtype:", images.dtype)
print("Mask dtype :", masks.dtype)

print(
    "Image range:",
    images.min().item(),
    "to",
    images.max().item()
)

print(
    "Mask values:",
    torch.unique(masks).numpy()
)

print(
    "Batch positive fraction:",
    masks.mean().item()
)

# ------------------------------------------------------------
# Validation checks
# ------------------------------------------------------------

assert images.shape == (8, 3, 512, 512)
assert masks.shape == (8, 1, 512, 512)

assert images.dtype == torch.float32
assert masks.dtype == torch.float32

assert torch.isfinite(images).all()
assert torch.isfinite(masks).all()

assert set(torch.unique(masks).numpy().tolist()).issubset(
    {0.0, 1.0}
)

print("\n============================================")
print("STRONG AUGMENTATION BATCH TEST PASSED")
print("NO TRAINING PERFORMED")
print("NO CHECKPOINT MODIFIED")
print("============================================")

epoch 4 → epoch 50.

In [ ]:
# ============================================================
# STEP16A — RESUME TRAINING FROM EPOCH 4
# Strong Augmentation / Domain Generalization
#
# WWTP IS NOT USED.
# ============================================================

import os
import json
import random
import numpy as np
from pathlib import Path

import torch
import torch.nn as nn
import segmentation_models_pytorch as smp

from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.transforms.functional as TF

# ============================================================
# CONFIGURATION
# ============================================================

SEED = 42

BATCH_SIZE = 8
MAX_EPOCHS = 50
PATIENCE = 10

LR = 1e-4
WEIGHT_DECAY = 1e-4

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ============================================================
# REPRODUCIBILITY
# ============================================================

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ============================================================
# PATHS
# ============================================================

ROOT = Path(
    "/kaggle/input/datasets/shanumkhsharma/microplastic-step16a-recovery/kaggle_bundle"
)

TRAIN_IMG = ROOT / "data" / "train" / "images"
TRAIN_MASK = ROOT / "data" / "train" / "masks"

VAL_IMG = ROOT / "data" / "val" / "images"
VAL_MASK = ROOT / "data" / "val" / "masks"

INPUT_CHECKPOINT = (
    ROOT
    / "checkpoints"
    / "beach_unet_resnet50_strong_aug_best_dice.pth"
)

# Persistent Kaggle working directory
OUT = Path("/kaggle/working/step16a_recovery")
OUT.mkdir(parents=True, exist_ok=True)

RECOVERY_DIR = OUT / "recovery_checkpoints"
RECOVERY_DIR.mkdir(parents=True, exist_ok=True)

BEST_PATH = OUT / "beach_unet_resnet50_strong_aug_best_dice.pth"
HISTORY_PATH = OUT / "training_history.json"
CONFIG_PATH = OUT / "training_config.json"

# ============================================================
# PRINT CONFIGURATION
# ============================================================

print("=" * 70)
print("STEP16A RECOVERY TRAINING")
print("=" * 70)

print("Device:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

print("Input checkpoint:", INPUT_CHECKPOINT)
print("Output directory:", OUT)

# ============================================================
# STRONG AUGMENTATION
# ============================================================

class StrongAugmentation:

    def __init__(self):

        self.color = T.ColorJitter(
            brightness=(0.75, 1.25),
            contrast=(0.75, 1.25),
            saturation=(0.70, 1.30),
        )

    def __call__(self, image, mask):

        # Horizontal flip
        if random.random() < 0.5:

            image = TF.hflip(image)
            mask = TF.hflip(mask)

        # Random affine
        angle = random.uniform(-20, 20)

        scale = random.uniform(0.90, 1.10)

        max_dx = int(0.10 * image.width)
        max_dy = int(0.10 * image.height)

        translate = [
            random.randint(-max_dx, max_dx),
            random.randint(-max_dy, max_dy),
        ]

        image = TF.affine(
            image,
            angle=angle,
            translate=translate,
            scale=scale,
            shear=[0.0, 0.0],
            interpolation=T.InterpolationMode.BILINEAR,
            fill=0,
        )

        mask = TF.affine(
            mask,
            angle=angle,
            translate=translate,
            scale=scale,
            shear=[0.0, 0.0],
            interpolation=T.InterpolationMode.NEAREST,
            fill=0,
        )

        # Image-only colour augmentation
        image = self.color(image)

        # Gaussian blur
        if random.random() < 0.20:

            radius = random.uniform(0.3, 1.5)

            kernel = max(
                3,
                int(round(radius * 4)) * 2 + 1
            )

            image = TF.gaussian_blur(
                image,
                kernel_size=[kernel, kernel],
                sigma=[radius, radius],
            )

        return image, mask


# ============================================================
# DATASET
# ============================================================

class BeachDataset(Dataset):

    def __init__(
        self,
        image_dir,
        mask_dir,
        augment=False
    ):

        self.image_dir = Path(image_dir)
        self.mask_dir = Path(mask_dir)

        self.images = sorted([
            p for p in self.image_dir.iterdir()
            if p.is_file()
        ])

        self.augment = augment

        if augment:
            self.transform = StrongAugmentation()

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):

        image_path = self.images[idx]
        mask_path = self.mask_dir / image_path.name

        image = Image.open(
            image_path
        ).convert("RGB")

        mask = Image.open(
            mask_path
        ).convert("L")

        # 256 -> 512
        image = image.resize(
            (512, 512),
            Image.Resampling.BILINEAR
        )

        mask = mask.resize(
            (512, 512),
            Image.Resampling.NEAREST
        )

        if self.augment:

            image, mask = self.transform(
                image,
                mask
            )

        image = TF.to_tensor(image)

        mask = torch.from_numpy(
            (np.asarray(mask) > 0)
            .astype(np.float32)
        ).unsqueeze(0)

        return image, mask


# ============================================================
# DATASETS
# ============================================================

train_dataset = BeachDataset(
    TRAIN_IMG,
    TRAIN_MASK,
    augment=True
)

val_dataset = BeachDataset(
    VAL_IMG,
    VAL_MASK,
    augment=False
)

print("\nDataset sizes:")
print("Train:", len(train_dataset))
print("Val  :", len(val_dataset))

# ============================================================
# DATALOADERS
# ============================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=False,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    drop_last=False,
)

# ============================================================
# MODEL
# ============================================================

model = smp.Unet(
    encoder_name="resnet50",
    encoder_weights=None,
    in_channels=3,
    classes=1
).to(DEVICE)

# ============================================================
# LOSS
# ============================================================

bce_loss_fn = nn.BCEWithLogitsLoss()


def dice_loss_fn(logits, target):

    probs = torch.sigmoid(logits)

    intersection = (
        probs * target
    ).sum(dim=(1, 2, 3))

    denominator = (
        probs.sum(dim=(1, 2, 3))
        +
        target.sum(dim=(1, 2, 3))
    )

    dice = (
        (2.0 * intersection + 1e-6)
        /
        (denominator + 1e-6)
    )

    return (1.0 - dice).mean()


def combined_loss(logits, target):

    bce = bce_loss_fn(
        logits,
        target
    )

    dice = dice_loss_fn(
        logits,
        target
    )

    total = (
        0.5 * bce
        +
        0.5 * dice
    )

    return total, bce, dice


# ============================================================
# METRICS
# ============================================================

@torch.no_grad()
def segmentation_metrics(logits, target):

    probs = torch.sigmoid(logits)

    pred = (
        probs >= 0.5
    ).float()

    eps = 1e-7

    intersection = (
        pred * target
    ).sum(dim=(1, 2, 3))

    pred_area = pred.sum(
        dim=(1, 2, 3)
    )

    target_area = target.sum(
        dim=(1, 2, 3)
    )

    union = (
        pred_area
        +
        target_area
        -
        intersection
    )

    dice = (
        (2.0 * intersection + eps)
        /
        (pred_area + target_area + eps)
    )

    iou = (
        (intersection + eps)
        /
        (union + eps)
    )

    tp = intersection

    fp = (
        pred * (1.0 - target)
    ).sum(dim=(1, 2, 3))

    fn = (
        (1.0 - pred) * target
    ).sum(dim=(1, 2, 3))

    precision = (
        (tp + eps)
        /
        (tp + fp + eps)
    )

    recall = (
        (tp + eps)
        /
        (tp + fn + eps)
    )

    correct = (
        pred == target
    ).float().mean(
        dim=(1, 2, 3)
    )

    return {
        "dice": dice.mean().item(),
        "iou": iou.mean().item(),
        "precision": precision.mean().item(),
        "recall": recall.mean().item(),
        "pixel_accuracy": correct.mean().item(),
    }


# ============================================================
# OPTIMIZER
# ============================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

# ============================================================
# LOAD EPOCH-3 CHECKPOINT
# ============================================================

checkpoint = torch.load(
    INPUT_CHECKPOINT,
    map_location=DEVICE
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

optimizer.load_state_dict(
    checkpoint["optimizer_state_dict"]
)

start_epoch = checkpoint["epoch"] + 1

best_val_dice = checkpoint["best_val_dice"]

history = list(
    checkpoint["history"]
)

print("\nRecovery state:")
print("Last completed epoch:", checkpoint["epoch"])
print("Starting epoch:", start_epoch)
print("Best validation Dice:", best_val_dice)

# ============================================================
# SAVE CONFIG
# ============================================================

config = {
    "experiment": "strong_augmentation",
    "seed": SEED,
    "architecture": "U-Net ResNet-50",
    "input_size": [512, 512],
    "batch_size": BATCH_SIZE,
    "max_epochs": MAX_EPOCHS,
    "patience": PATIENCE,
    "learning_rate": LR,
    "weight_decay": WEIGHT_DECAY,
    "loss": "0.5 BCEWithLogits + 0.5 Dice",
    "train_images": len(train_dataset),
    "validation_images": len(val_dataset),
    "wwtp_used_for_training": False,
    "resume_from_epoch": checkpoint["epoch"],
}

with open(CONFIG_PATH, "w") as f:
    json.dump(config, f, indent=2)

# ============================================================
# AMP
# ============================================================

scaler = torch.cuda.amp.GradScaler(
    enabled=(DEVICE.type == "cuda")
)

# ============================================================
# TRAINING LOOP
# ============================================================

epochs_without_improvement = 0

print("\n" + "=" * 70)
print("STARTING RECOVERY TRAINING")
print("=" * 70)

for epoch in range(start_epoch, MAX_EPOCHS + 1):

    # --------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------

    model.train()

    train_loss_sum = 0.0
    train_count = 0

    for images, masks in train_loader:

        images = images.to(
            DEVICE,
            non_blocking=True
        )

        masks = masks.to(
            DEVICE,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.cuda.amp.autocast(
            enabled=(DEVICE.type == "cuda")
        ):

            logits = model(images)

            loss, bce, dice = combined_loss(
                logits,
                masks
            )

        scaler.scale(loss).backward()

        scaler.step(optimizer)

        scaler.update()

        batch_size = images.size(0)

        train_loss_sum += (
            loss.item() * batch_size
        )

        train_count += batch_size

    train_loss = (
        train_loss_sum
        /
        train_count
    )

    # --------------------------------------------------------
    # VALIDATION
    # --------------------------------------------------------

    model.eval()

    val_loss_sum = 0.0
    val_count = 0

    metric_sums = {
        "dice": 0.0,
        "iou": 0.0,
        "precision": 0.0,
        "recall": 0.0,
        "pixel_accuracy": 0.0,
    }

    with torch.no_grad():

        for images, masks in val_loader:

            images = images.to(
                DEVICE,
                non_blocking=True
            )

            masks = masks.to(
                DEVICE,
                non_blocking=True
            )

            with torch.cuda.amp.autocast(
                enabled=(DEVICE.type == "cuda")
            ):

                logits = model(images)

                loss, _, _ = combined_loss(
                    logits,
                    masks
                )

            batch_size = images.size(0)

            val_loss_sum += (
                loss.item() * batch_size
            )

            val_count += batch_size

            metrics = segmentation_metrics(
                logits,
                masks
            )

            for key in metric_sums:

                metric_sums[key] += (
                    metrics[key] * batch_size
                )

    val_loss = (
        val_loss_sum
        /
        val_count
    )

    val_metrics = {
        key: value / val_count
        for key, value in metric_sums.items()
    }

    # --------------------------------------------------------
    # HISTORY
    # --------------------------------------------------------

    record = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_dice": val_metrics["dice"],
        "val_iou": val_metrics["iou"],
        "val_precision": val_metrics["precision"],
        "val_recall": val_metrics["recall"],
        "val_pixel_accuracy": val_metrics[
            "pixel_accuracy"
        ],
    }

    history.append(record)

    # --------------------------------------------------------
    # BEST MODEL
    # --------------------------------------------------------

    is_best = (
        val_metrics["dice"]
        >
        best_val_dice
    )

    if is_best:

        best_val_dice = (
            val_metrics["dice"]
        )

        epochs_without_improvement = 0

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_val_dice": best_val_dice,
                "history": history,
                "experiment": "strong_augmentation",
                "seed": SEED,
            },
            BEST_PATH
        )

    else:

        epochs_without_improvement += 1

    # --------------------------------------------------------
    # EVERY-EPOCH RECOVERY CHECKPOINT
    # --------------------------------------------------------

    recovery_path = (
        RECOVERY_DIR
        /
        f"epoch_{epoch:02d}_recovery.pth"
    )

    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_val_dice": best_val_dice,
            "history": history,
            "experiment": "strong_augmentation",
            "seed": SEED,
        },
        recovery_path
    )

    # --------------------------------------------------------
    # SAVE HISTORY
    # --------------------------------------------------------

    with open(HISTORY_PATH, "w") as f:
        json.dump(history, f, indent=2)

    # --------------------------------------------------------
    # PRINT
    # --------------------------------------------------------

    marker = " <-- BEST" if is_best else ""

    print(
        f"Epoch {epoch:02d}/50 | "
        f"Train Loss {train_loss:.4f} | "
        f"Val Loss {val_loss:.4f} | "
        f"Dice {val_metrics['dice']:.4f} | "
        f"IoU {val_metrics['iou']:.4f} | "
        f"Precision {val_metrics['precision']:.4f} | "
        f"Recall {val_metrics['recall']:.4f}"
        f"{marker}"
    )

    # --------------------------------------------------------
    # EARLY STOPPING
    # --------------------------------------------------------

    if epochs_without_improvement >= PATIENCE:

        print(
            f"\nEarly stopping after epoch {epoch}."
        )

        break

# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("RECOVERY TRAINING FINISHED")
print("=" * 70)

print("Best validation Dice:", best_val_dice)
print("Best checkpoint:", BEST_PATH)
print("History:", HISTORY_PATH)
print("Recovery checkpoints:", RECOVERY_DIR)

print("\nCompleted epochs:")

for h in history:
    print(
        f"Epoch {h['epoch']:02d}: "
        f"Val Dice={h['val_dice']:.6f}"
    )

Baseline:
Beach TEST       → Dice 0.8323

Strong Aug:
Beach VALIDATION → Dice 0.8140

In [ ]:
from pathlib import Path

ROOT = Path(
    "/kaggle/input/datasets/shanumkhsharma/kaggle-microplastic-bundle"
)

for split in ["train", "validation", "test"]:

    image_dir = ROOT / "data" / "wwtp" / "images" / split
    mask_dir = ROOT / "data" / "wwtp" / "masks" / split

    print("\n" + "=" * 60)
    print(split)
    print("=" * 60)

    print("\nImages:")
    for p in sorted(image_dir.iterdir())[:5]:
        print(" ", p.name)

    print("\nMasks:")
    for p in sorted(mask_dir.iterdir())[:5]:
        print(" ", p.name)

In [ ]:
# ============================================================
# STEP16B — EVALUATE STRONG-AUGMENTATION MODEL
# Corrected WWTP .jpg -> .png mask mapping
# ============================================================

import json
import numpy as np
import pandas as pd
import torch
import segmentation_models_pytorch as smp

from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm


# ============================================================
# PATHS
# ============================================================

RECOVERY_ROOT = Path(
    "/kaggle/working/step16a_recovery"
)

BEST_CKPT = (
    RECOVERY_ROOT
    / "beach_unet_resnet50_strong_aug_best_dice.pth"
)

# Step16A Beach test set
BEACH_ROOT = Path(
    "/kaggle/input/datasets/shanumkhsharma/microplastic-step16a-recovery/kaggle_bundle"
)

BEACH_TEST_IMG = (
    BEACH_ROOT / "data" / "test" / "images"
)

BEACH_TEST_MASK = (
    BEACH_ROOT / "data" / "test" / "masks"
)

# WWTP external dataset
WWTP_ROOT = Path(
    "/kaggle/input/datasets/shanumkhsharma/kaggle-microplastic-bundle"
)

WWTP_IMAGES = (
    WWTP_ROOT / "data" / "wwtp" / "images"
)

WWTP_MASKS = (
    WWTP_ROOT / "data" / "wwtp" / "masks"
)

OUTPUT = (
    RECOVERY_ROOT / "evaluation"
)

OUTPUT.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# DEVICE
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)

if device.type == "cuda":
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )


# ============================================================
# LOAD BEST STEP16A CHECKPOINT
# ============================================================

checkpoint = torch.load(
    BEST_CKPT,
    map_location=device
)

print("\nCheckpoint epoch:", checkpoint["epoch"])
print(
    "Best validation Dice:",
    checkpoint["best_val_dice"]
)

model = smp.Unet(
    encoder_name="resnet50",
    encoder_weights=None,
    in_channels=3,
    classes=1
).to(device)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

print("Model loaded.")


# ============================================================
# METRICS
# ============================================================

def calculate_metrics(
    prediction,
    target
):

    prediction = prediction.astype(bool)
    target = target.astype(bool)

    intersection = np.logical_and(
        prediction,
        target
    ).sum()

    pred_area = prediction.sum()
    target_area = target.sum()

    union = np.logical_or(
        prediction,
        target
    ).sum()

    dice = (
        2.0 * intersection
        /
        (
            pred_area
            + target_area
            + 1e-8
        )
    )

    iou = (
        intersection
        /
        (union + 1e-8)
    )

    tp = intersection

    fp = np.logical_and(
        prediction,
        ~target
    ).sum()

    fn = np.logical_and(
        ~prediction,
        target
    ).sum()

    precision = (
        tp
        /
        (tp + fp + 1e-8)
    )

    recall = (
        tp
        /
        (tp + fn + 1e-8)
    )

    pixel_accuracy = (
        prediction == target
    ).mean()

    return {
        "dice": dice,
        "iou": iou,
        "precision": precision,
        "recall": recall,
        "pixel_accuracy": pixel_accuracy,
    }


# ============================================================
# PART 1 — BEACH TEST
# ============================================================

print("\n" + "=" * 70)
print("PART 1 — BEACH TEST")
print("=" * 70)

beach_images = sorted(
    [
        p
        for p in BEACH_TEST_IMG.iterdir()
        if p.is_file()
    ]
)

print(
    "Beach test images:",
    len(beach_images)
)

beach_results = []

with torch.no_grad():

    for image_path in tqdm(
        beach_images,
        desc="Beach test"
    ):

        mask_path = (
            BEACH_TEST_MASK
            / image_path.name
        )

        image = Image.open(
            image_path
        ).convert("RGB")

        mask = Image.open(
            mask_path
        ).convert("L")

        image = image.resize(
            (512, 512),
            Image.Resampling.BILINEAR
        )

        mask = mask.resize(
            (512, 512),
            Image.Resampling.NEAREST
        )

        image_np = (
            np.asarray(image)
            .astype(np.float32)
            / 255.0
        )

        target = (
            np.asarray(mask) > 0
        )

        tensor = torch.from_numpy(
            image_np.transpose(2, 0, 1)
        ).unsqueeze(0).to(device)

        logits = model(tensor)

        probability = torch.sigmoid(
            logits
        )[0, 0].cpu().numpy()

        prediction = (
            probability >= 0.5
        )

        metrics = calculate_metrics(
            prediction,
            target
        )

        beach_results.append({
            "image": image_path.name,
            **metrics
        })


beach_df = pd.DataFrame(
    beach_results
)

beach_summary = {
    metric: float(
        beach_df[metric].mean()
    )
    for metric in [
        "dice",
        "iou",
        "precision",
        "recall",
        "pixel_accuracy"
    ]
}

beach_df.to_csv(
    OUTPUT
    / "strong_aug_beach_test_per_image.csv",
    index=False
)

with open(
    OUTPUT
    / "strong_aug_beach_test_summary.json",
    "w"
) as f:

    json.dump(
        beach_summary,
        f,
        indent=2
    )

print("\nBeach test results:")

for key, value in beach_summary.items():

    print(
        f"{key:18s}: {value:.4f}"
    )


# ============================================================
# WWTP TILE POSITIONS
# ============================================================

TILE_SIZE = 512
STRIDE = 384


def get_positions(length):

    positions = list(
        range(
            0,
            max(
                1,
                length - TILE_SIZE + 1
            ),
            STRIDE
        )
    )

    final_position = (
        length - TILE_SIZE
    )

    if positions[-1] != final_position:
        positions.append(
            final_position
        )

    return positions


# ============================================================
# WWTP INFERENCE
# ============================================================

def predict_wwtp(image):

    height, width = image.shape[:2]

    xs = get_positions(width)
    ys = get_positions(height)

    probability_sum = np.zeros(
        (height, width),
        dtype=np.float32
    )

    count_map = np.zeros(
        (height, width),
        dtype=np.float32
    )

    with torch.no_grad():

        for y in ys:

            for x in xs:

                tile = image[
                    y:y + TILE_SIZE,
                    x:x + TILE_SIZE
                ]

                tile_tensor = torch.from_numpy(
                    tile.astype(np.float32)
                    .transpose(2, 0, 1)
                    / 255.0
                ).unsqueeze(0).to(device)

                logits = model(
                    tile_tensor
                )

                probability = torch.sigmoid(
                    logits
                )[0, 0].cpu().numpy()

                probability_sum[
                    y:y + TILE_SIZE,
                    x:x + TILE_SIZE
                ] += probability

                count_map[
                    y:y + TILE_SIZE,
                    x:x + TILE_SIZE
                ] += 1

    probability_map = (
        probability_sum
        /
        np.maximum(
            count_map,
            1
        )
    )

    return probability_map


# ============================================================
# PART 2 — UNSEEN WWTP
# ============================================================

print("\n" + "=" * 70)
print("PART 2 — UNSEEN WWTP")
print("=" * 70)

all_wwtp_images = []

for split in [
    "train",
    "validation",
    "test"
]:

    split_dir = (
        WWTP_IMAGES / split
    )

    all_wwtp_images.extend(
        [
            (split, p)
            for p in sorted(
                split_dir.iterdir()
            )
            if p.is_file()
        ]
    )

print(
    "Total WWTP images:",
    len(all_wwtp_images)
)

wwtp_results = []

for split, image_path in tqdm(
    all_wwtp_images,
    desc="WWTP external"
):

    # IMPORTANT:
    # image = .jpg
    # mask  = .png

    mask_path = (
        WWTP_MASKS
        / split
        / f"{image_path.stem}.png"
    )

    if not mask_path.exists():

        raise FileNotFoundError(
            f"Mask not found:\n{mask_path}"
        )

    image = np.asarray(
        Image.open(
            image_path
        ).convert("RGB")
    )

    target = (
        np.asarray(
            Image.open(
                mask_path
            ).convert("L")
        ) > 0
    )

    probability_map = predict_wwtp(
        image
    )

    prediction = (
        probability_map >= 0.5
    )

    metrics = calculate_metrics(
        prediction,
        target
    )

    gt_fraction = target.mean()
    pred_fraction = prediction.mean()

    area_ratio = (
        pred_fraction
        /
        max(
            gt_fraction,
            1e-12
        )
    )

    wwtp_results.append({
        "split": split,
        "image": image_path.name,
        "height": image.shape[0],
        "width": image.shape[1],
        "gt_foreground_fraction": gt_fraction,
        "pred_foreground_fraction": pred_fraction,
        "pred_gt_area_ratio": area_ratio,
        **metrics
    })


# ============================================================
# WWTP SUMMARY
# ============================================================

wwtp_df = pd.DataFrame(
    wwtp_results
)

# Primary evaluation:
# only images containing annotated foreground.
wwtp_primary = wwtp_df[
    wwtp_df[
        "gt_foreground_fraction"
    ] > 0
].copy()

wwtp_summary = {
    "n_images": int(
        len(wwtp_primary)
    ),

    "dice_mean": float(
        wwtp_primary["dice"].mean()
    ),

    "dice_std": float(
        wwtp_primary["dice"].std()
    ),

    "dice_median": float(
        wwtp_primary["dice"].median()
    ),

    "iou_mean": float(
        wwtp_primary["iou"].mean()
    ),

    "precision_mean": float(
        wwtp_primary["precision"].mean()
    ),

    "recall_mean": float(
        wwtp_primary["recall"].mean()
    ),

    "pixel_accuracy_mean": float(
        wwtp_primary[
            "pixel_accuracy"
        ].mean()
    ),

    "gt_foreground_fraction_mean": float(
        wwtp_primary[
            "gt_foreground_fraction"
        ].mean()
    ),

    "pred_foreground_fraction_mean": float(
        wwtp_primary[
            "pred_foreground_fraction"
        ].mean()
    ),

    "pred_gt_area_ratio_mean": float(
        wwtp_primary[
            "pred_gt_area_ratio"
        ].mean()
    ),

    "pred_gt_area_ratio_median": float(
        wwtp_primary[
            "pred_gt_area_ratio"
        ].median()
    )
}


wwtp_df.to_csv(
    OUTPUT
    / "strong_aug_wwtp_per_image.csv",
    index=False
)

with open(
    OUTPUT
    / "strong_aug_wwtp_summary.json",
    "w"
) as f:

    json.dump(
        wwtp_summary,
        f,
        indent=2
    )


print("\nWWTP external results:")

for key, value in wwtp_summary.items():

    if isinstance(value, float):

        print(
            f"{key:35s}: {value:.6f}"
        )

    else:

        print(
            f"{key:35s}: {value}"
        )


print("\n" + "=" * 70)
print("STEP16B EVALUATION COMPLETE")
print("=" * 70)

print(
    "\nResults saved to:",
    OUTPUT
)

In [ ]:
import os
from pathlib import Path

print("Checking Step16A checkpoint...")

paths = [
    Path("/kaggle/working/step16a_recovery/beach_unet_resnet50_strong_aug_best_dice.pth"),
    Path("/kaggle/input/datasets/shanumkhsharma/microplastic-step16a-recovery/kaggle_bundle"),
]

for p in paths:
    print(f"{p}")
    print("  exists:", p.exists())
    if p.exists():
        print("  size:", p.stat().st_size / (1024**2), "MB")

In [ ]:
from pathlib import Path

ROOT = Path("/kaggle/input/datasets/shanumkhsharma/microplastic-step16a-recovery/kaggle_bundle")

print("Searching recovery dataset for checkpoints...\n")

matches = []

for p in ROOT.rglob("*"):
    if p.is_file():
        name = p.name.lower()
        if (
            "checkpoint" in name
            or "best" in name
            or name.endswith(".pth")
            or name.endswith(".pt")
        ):
            matches.append(p)

if matches:
    for p in matches:
        print(f"{p}")
        print(f"  size: {p.stat().st_size / (1024**2):.2f} MB")
else:
    print("NO .pth/.pt/checkpoint files found.")

In [ ]:
import torch
import segmentation_models_pytorch as smp
from pathlib import Path

CKPT = Path(
    "/kaggle/input/datasets/shanumkhsharma/"
    "microplastic-step16a-recovery/kaggle_bundle/"
    "checkpoints/beach_unet_resnet50_strong_aug_best_dice.pth"
)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print("Device:", device)
print("Checkpoint:", CKPT)
print("Size:", CKPT.stat().st_size / (1024**2), "MB")

model = smp.Unet(
    encoder_name="resnet50",
    encoder_weights=None,
    in_channels=3,
    classes=1
).to(device)

checkpoint = torch.load(CKPT, map_location=device)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("\nCheckpoint loaded successfully.")
print("Epoch:", checkpoint.get("epoch"))
print("Best validation Dice:", checkpoint.get("best_val_dice"))
print("Model parameters:", sum(p.numel() for p in model.parameters()))

In [ ]:
from pathlib import Path

roots = [
    Path("/kaggle/input/datasets/shanumkhsharma/kaggle-microplastic-bundle"),
    Path("/kaggle/input/datasets/shanumkhsharma/microplastic-step16a-recovery"),
    Path("/kaggle/working"),
]

print("Searching for ALL Step16A checkpoints...\n")

for root in roots:
    print("=" * 80)
    print(root)

    if not root.exists():
        print("  ROOT DOES NOT EXIST")
        continue

    found = False

    for p in root.rglob("*.pth"):
        size_mb = p.stat().st_size / (1024**2)
        print(f"  {p}")
        print(f"    size: {size_mb:.2f} MB")
        found = True

    if not found:
        print("  No .pth files found.")

#START FROM HERE EXPERIMENT 2A

# Before we ran but we didn't save it so no has to rerun it

In [ ]:
# ============================================================
# STEP16A RECOVERY — RESUME FROM EPOCH 3 → EPOCH 50
# ============================================================

import os
import json
import random
import shutil
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image

import segmentation_models_pytorch as smp
import albumentations as A

# ------------------------------------------------------------
# 1. Reproducibility
# ------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


# ------------------------------------------------------------
# 2. Paths
# ------------------------------------------------------------

BUNDLE = Path(
    "/kaggle/input/datasets/shanumkhsharma/"
    "microplastic-step16a-recovery/kaggle_bundle"
)

TRAIN_IMAGES = BUNDLE / "data/test" / "images"
TRAIN_MASKS  = BUNDLE / "data/test" / "masks"

# IMPORTANT:
# The recovery bundle's actual supervised Beach data should be
# checked before training.
print("\nChecking dataset paths:")
print("TRAIN_IMAGES:", TRAIN_IMAGES, TRAIN_IMAGES.exists())
print("TRAIN_MASKS :", TRAIN_MASKS, TRAIN_MASKS.exists())

RECOVERY_CKPT = (
    BUNDLE / "checkpoints" /
    "beach_unet_resnet50_strong_aug_best_dice.pth"
)

print("Recovery checkpoint:", RECOVERY_CKPT)
print("Exists:", RECOVERY_CKPT.exists())


# ------------------------------------------------------------
# 3. IMPORTANT DATA PATH CHECK
# ------------------------------------------------------------
# Do NOT proceed if the above paths do not correspond to the
# actual train/val data.
#
# We will locate the exact train/val/test directories from the
# bundle rather than assuming names.

DATA_ROOT = BUNDLE / "data"

print("\nData directories:")
for p in sorted(DATA_ROOT.iterdir()):
    print(" ", p)


# ------------------------------------------------------------
# 4. Find supervised Beach split directories
# ------------------------------------------------------------

candidate_dirs = {}

for p in DATA_ROOT.rglob("*"):
    if p.is_dir():
        files = list(p.iterdir())
        if files:
            image_count = sum(
                f.suffix.lower() in [".jpg", ".jpeg", ".png"]
                for f in files if f.is_file()
            )
            if image_count:
                candidate_dirs[str(p)] = image_count

print("\nCandidate image directories:")
for k, v in sorted(candidate_dirs.items()):
    print(f"{v:4d} images  {k}")


# ------------------------------------------------------------
# 5. Locate expected supervised split directories
# ------------------------------------------------------------

def find_split_dir(root, split):
    candidates = []

    for p in root.rglob("*"):
        if not p.is_dir():
            continue

        name = p.name.lower()

        if name == split.lower():
            candidates.append(p)

    return candidates


train_candidates = find_split_dir(DATA_ROOT, "train")
val_candidates   = find_split_dir(DATA_ROOT, "val")
test_candidates  = find_split_dir(DATA_ROOT, "test")

print("\nTrain candidates:", train_candidates)
print("Val candidates:", val_candidates)
print("Test candidates:", test_candidates)

#Experiment 2a -> Strong agumentation using U-Net & ResNet50

In [ ]:
# ============================================================
# STEP16A RECOVERY
# Resume from epoch 3 -> epoch 50
# Strong augmentation | U-Net ResNet50
# ============================================================

import os
import json
import random
import shutil
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image

import segmentation_models_pytorch as smp
import albumentations as A

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

SEED = 42
BATCH_SIZE = 8
MAX_EPOCHS = 50
PATIENCE = 10
LR = 1e-4
WEIGHT_DECAY = 1e-4
IMAGE_SIZE = 512

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print("Device:", device)
print("GPU:", torch.cuda.get_device_name(0))


# ------------------------------------------------------------
# 2. Paths
# ------------------------------------------------------------

BUNDLE = Path(
    "/kaggle/input/datasets/shanumkhsharma/"
    "microplastic-step16a-recovery/kaggle_bundle"
)

TRAIN_IMG = BUNDLE / "data/train/images"
TRAIN_MSK = BUNDLE / "data/train/masks"

VAL_IMG = BUNDLE / "data/val/images"
VAL_MSK = BUNDLE / "data/val/masks"

RECOVERY_CKPT = (
    BUNDLE / "checkpoints/"
    "beach_unet_resnet50_strong_aug_best_dice.pth"
)

OUT = Path("/kaggle/working/step16a_recovery")
OUT.mkdir(parents=True, exist_ok=True)

RECOVERY_DIR = OUT / "recovery_checkpoints"
RECOVERY_DIR.mkdir(parents=True, exist_ok=True)

BEST_CKPT = OUT / "beach_unet_resnet50_strong_aug_best_dice.pth"
HISTORY_FILE = OUT / "training_history.json"


# ------------------------------------------------------------
# 3. Verify dataset counts
# ------------------------------------------------------------

train_images = sorted([
    p for p in TRAIN_IMG.iterdir()
    if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
])

train_masks = sorted([
    p for p in TRAIN_MSK.iterdir()
    if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
])

val_images = sorted([
    p for p in VAL_IMG.iterdir()
    if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
])

val_masks = sorted([
    p for p in VAL_MSK.iterdir()
    if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
])

print("\nDataset:")
print("Train images:", len(train_images))
print("Train masks :", len(train_masks))
print("Val images  :", len(val_images))
print("Val masks   :", len(val_masks))

assert len(train_images) == 790
assert len(train_masks) == 790
assert len(val_images) == 169
assert len(val_masks) == 169


# ------------------------------------------------------------
# 4. Dataset
# ------------------------------------------------------------

class BeachDataset(Dataset):

    def __init__(self, image_dir, mask_dir, augment=False):

        self.image_dir = Path(image_dir)
        self.mask_dir = Path(mask_dir)

        self.images = sorted([
            p for p in self.image_dir.iterdir()
            if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
        ])

        self.augment = augment

        if augment:
            self.transform = A.Compose([
                A.HorizontalFlip(p=0.5),

                A.Affine(
                    scale=(0.90, 1.10),
                    translate_percent=(-0.10, 0.10),
                    rotate=(-20, 20),
                    interpolation=1,
                    mask_interpolation=0,
                    p=0.8
                ),

                A.RandomBrightnessContrast(
                    brightness_limit=0.25,
                    contrast_limit=0.25,
                    p=0.5
                ),

                A.HueSaturationValue(
                    hue_shift_limit=0,
                    sat_shift_limit=30,
                    val_shift_limit=0,
                    p=0.3
                ),

                A.GaussianBlur(
                    blur_limit=(3, 7),
                    p=0.20
                ),
            ])
        else:
            self.transform = None

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):

        img_path = self.images[idx]
        mask_path = self.mask_dir / f"{img_path.stem}.png"

        image = np.array(
            Image.open(img_path).convert("RGB")
        )

        mask = np.array(
            Image.open(mask_path).convert("L")
        )

        if self.transform is not None:
            transformed = self.transform(
                image=image,
                mask=mask
            )

            image = transformed["image"]
            mask = transformed["mask"]

        # 256 -> 512
        image = np.array(
            Image.fromarray(image).resize(
                (IMAGE_SIZE, IMAGE_SIZE),
                Image.Resampling.BILINEAR
            )
        )

        mask = np.array(
            Image.fromarray(mask).resize(
                (IMAGE_SIZE, IMAGE_SIZE),
                Image.Resampling.NEAREST
            )
        )

        image = torch.from_numpy(
            image.astype(np.float32) / 255.0
        ).permute(2, 0, 1)

        mask = torch.from_numpy(
            (mask > 127).astype(np.float32)
        ).unsqueeze(0)

        return image, mask


train_ds = BeachDataset(
    TRAIN_IMG,
    TRAIN_MSK,
    augment=True
)

val_ds = BeachDataset(
    VAL_IMG,
    VAL_MSK,
    augment=False
)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=False
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("\nLoaders ready.")
print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))


# ------------------------------------------------------------
# 5. Model
# ------------------------------------------------------------

model = smp.Unet(
    encoder_name="resnet50",
    encoder_weights=None,
    in_channels=3,
    classes=1
).to(device)

print("\nModel parameters:",
      sum(p.numel() for p in model.parameters()))


# ------------------------------------------------------------
# 6. Loss
# ------------------------------------------------------------

bce_loss = nn.BCEWithLogitsLoss()

def dice_loss(logits, targets, eps=1e-7):

    probs = torch.sigmoid(logits)

    intersection = (
        probs * targets
    ).sum(dim=(1, 2, 3))

    denominator = (
        probs.sum(dim=(1, 2, 3))
        + targets.sum(dim=(1, 2, 3))
    )

    dice = (
        (2 * intersection + eps)
        / (denominator + eps)
    )

    return 1 - dice.mean()


def total_loss(logits, targets):

    bce = bce_loss(logits, targets)
    dice = dice_loss(logits, targets)

    return 0.5 * bce + 0.5 * dice


# ------------------------------------------------------------
# 7. Metrics
# ------------------------------------------------------------

def calculate_metrics(logits, targets):

    probs = torch.sigmoid(logits)
    preds = (probs >= 0.5).float()

    tp = (preds * targets).sum()
    fp = (preds * (1 - targets)).sum()
    fn = ((1 - preds) * targets).sum()
    tn = ((1 - preds) * (1 - targets)).sum()

    dice = (2 * tp + 1e-7) / (
        2 * tp + fp + fn + 1e-7
    )

    iou = (tp + 1e-7) / (
        tp + fp + fn + 1e-7
    )

    precision = (tp + 1e-7) / (
        tp + fp + 1e-7
    )

    recall = (tp + 1e-7) / (
        tp + fn + 1e-7
    )

    accuracy = (
        tp + tn
    ) / (
        tp + tn + fp + fn + 1e-7
    )

    return (
        dice.item(),
        iou.item(),
        precision.item(),
        recall.item(),
        accuracy.item()
    )


# ------------------------------------------------------------
# 8. Optimizer
# ------------------------------------------------------------

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)


# ------------------------------------------------------------
# 9. Load epoch-3 recovery checkpoint
# ------------------------------------------------------------

print("\nLoading recovery checkpoint...")

checkpoint = torch.load(
    RECOVERY_CKPT,
    map_location=device
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

optimizer.load_state_dict(
    checkpoint["optimizer_state_dict"]
)

start_epoch = checkpoint["epoch"] + 1

best_val_dice = checkpoint["best_val_dice"]

history = checkpoint.get("history", [])

print("Recovered epoch:", checkpoint["epoch"])
print("Recovered best Dice:", best_val_dice)
print("Resuming from epoch:", start_epoch)
print("Existing history entries:", len(history))


# ------------------------------------------------------------
# 10. AMP
# ------------------------------------------------------------

scaler = torch.cuda.amp.GradScaler(
    enabled=torch.cuda.is_available()
)


# ------------------------------------------------------------
# 11. Validation function
# ------------------------------------------------------------

@torch.no_grad()
def evaluate(loader):

    model.eval()

    loss_values = []

    dice_values = []
    iou_values = []
    precision_values = []
    recall_values = []
    accuracy_values = []

    for images, masks in loader:

        images = images.to(
            device,
            non_blocking=True
        )

        masks = masks.to(
            device,
            non_blocking=True
        )

        with torch.cuda.amp.autocast(
            enabled=torch.cuda.is_available()
        ):
            logits = model(images)
            loss = total_loss(logits, masks)

        loss_values.append(loss.item())

        metrics = calculate_metrics(
            logits,
            masks
        )

        dice_values.append(metrics[0])
        iou_values.append(metrics[1])
        precision_values.append(metrics[2])
        recall_values.append(metrics[3])
        accuracy_values.append(metrics[4])

    return {
        "loss": float(np.mean(loss_values)),
        "dice": float(np.mean(dice_values)),
        "iou": float(np.mean(iou_values)),
        "precision": float(np.mean(precision_values)),
        "recall": float(np.mean(recall_values)),
        "pixel_accuracy": float(np.mean(accuracy_values)),
    }


# ------------------------------------------------------------
# 12. Resume training
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STARTING RECOVERY TRAINING")
print(f"Epoch {start_epoch} -> {MAX_EPOCHS}")
print("=" * 70)

epochs_without_improvement = 0

for epoch in range(start_epoch, MAX_EPOCHS + 1):

    model.train()

    train_losses = []

    for images, masks in train_loader:

        images = images.to(
            device,
            non_blocking=True
        )

        masks = masks.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.cuda.amp.autocast(
            enabled=torch.cuda.is_available()
        ):

            logits = model(images)

            loss = total_loss(
                logits,
                masks
            )

        scaler.scale(loss).backward()

        scaler.step(optimizer)

        scaler.update()

        train_losses.append(
            loss.item()
        )

    train_loss = float(
        np.mean(train_losses)
    )

    val_metrics = evaluate(
        val_loader
    )

    record = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_metrics["loss"],
        "val_dice": val_metrics["dice"],
        "val_iou": val_metrics["iou"],
        "val_precision": val_metrics["precision"],
        "val_recall": val_metrics["recall"],
        "val_pixel_accuracy":
            val_metrics["pixel_accuracy"]
    }

    history.append(record)

    improved = (
        val_metrics["dice"] >
        best_val_dice
    )

    if improved:

        best_val_dice = (
            val_metrics["dice"]
        )

        epochs_without_improvement = 0

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict":
                    model.state_dict(),
                "optimizer_state_dict":
                    optimizer.state_dict(),
                "best_val_dice":
                    best_val_dice,
                "history": history,
                "experiment":
                    "strong_augmentation_recovery",
                "seed": SEED
            },
            BEST_CKPT
        )

        print(
            f"  >>> NEW BEST CHECKPOINT"
        )

    else:

        epochs_without_improvement += 1

    # --------------------------------------------------------
    # Save recovery checkpoint EVERY epoch
    # --------------------------------------------------------

    recovery_path = (
        RECOVERY_DIR /
        f"epoch_{epoch:03d}.pth"
    )

    torch.save(
        {
            "epoch": epoch,
            "model_state_dict":
                model.state_dict(),
            "optimizer_state_dict":
                optimizer.state_dict(),
            "best_val_dice":
                best_val_dice,
            "history": history,
            "experiment":
                "strong_augmentation_recovery",
            "seed": SEED
        },
        recovery_path
    )

    # Save history every epoch
    with open(
        HISTORY_FILE,
        "w"
    ) as f:
        json.dump(
            history,
            f,
            indent=2
        )

    print(
        f"Epoch {epoch:02d}/{MAX_EPOCHS} | "
        f"Train Loss {train_loss:.4f} | "
        f"Val Loss {val_metrics['loss']:.4f} | "
        f"Dice {val_metrics['dice']:.4f} | "
        f"IoU {val_metrics['iou']:.4f} | "
        f"Precision {val_metrics['precision']:.4f} | "
        f"Recall {val_metrics['recall']:.4f} | "
        f"Best {best_val_dice:.4f}"
    )

    # --------------------------------------------------------
    # Early stopping
    # --------------------------------------------------------

    if epochs_without_improvement >= PATIENCE:

        print(
            f"\nEarly stopping triggered after "
            f"{PATIENCE} epochs without improvement."
        )

        break


print("\n" + "=" * 70)
print("RECOVERY TRAINING FINISHED")
print("=" * 70)

print("Best validation Dice:", best_val_dice)
print("Best checkpoint:", BEST_CKPT)
print("History:", HISTORY_FILE)

if BEST_CKPT.exists():
    print(
        "Best checkpoint size:",
        BEST_CKPT.stat().st_size / (1024**2),
        "MB"
    )

In [ ]:
from pathlib import Path

BEST = Path(
    "/kaggle/working/step16a_recovery/"
    "beach_unet_resnet50_strong_aug_best_dice.pth"
)

print("Exists:", BEST.exists())
print("Size:", BEST.stat().st_size / (1024**2), "MB")

ckpt = torch.load(BEST, map_location="cpu")

print("Saved epoch:", ckpt["epoch"])
print("Best validation Dice:", ckpt["best_val_dice"])
print("History entries:", len(ckpt["history"]))

#Beach Test for 2A

In [ ]:
# ============================================================
# STEP16A — BEACH TEST EVALUATION
# Same 169-image test set as baseline
# ============================================================

import json
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import segmentation_models_pytorch as smp

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

BUNDLE = Path(
    "/kaggle/input/datasets/shanumkhsharma/"
    "microplastic-step16a-recovery/kaggle_bundle"
)

TEST_IMG = BUNDLE / "data/test/images"
TEST_MSK = BUNDLE / "data/test/masks"

BEST_CKPT = Path(
    "/kaggle/working/step16a_recovery/"
    "beach_unet_resnet50_strong_aug_best_dice.pth"
)

device = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)
print("GPU:", torch.cuda.get_device_name(0))


# ------------------------------------------------------------
# Dataset
# ------------------------------------------------------------

class BeachTestDataset(Dataset):

    def __init__(self, image_dir, mask_dir):

        self.image_dir = Path(image_dir)
        self.mask_dir = Path(mask_dir)

        self.images = sorted([
            p for p in self.image_dir.iterdir()
            if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
        ])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):

        img_path = self.images[idx]

        mask_path = (
            self.mask_dir /
            f"{img_path.stem}.png"
        )

        image = np.array(
            Image.open(img_path).convert("RGB")
        )

        mask = np.array(
            Image.open(mask_path).convert("L")
        )

        # Same preprocessing as baseline:
        # image -> 512 bilinear
        # mask  -> 512 nearest

        image = np.array(
            Image.fromarray(image).resize(
                (512, 512),
                Image.Resampling.BILINEAR
            )
        )

        mask = np.array(
            Image.fromarray(mask).resize(
                (512, 512),
                Image.Resampling.NEAREST
            )
        )

        image = torch.from_numpy(
            image.astype(np.float32) / 255.0
        ).permute(2, 0, 1)

        mask = torch.from_numpy(
            (mask > 127).astype(np.float32)
        ).unsqueeze(0)

        return image, mask, img_path.name


test_ds = BeachTestDataset(
    TEST_IMG,
    TEST_MSK
)

test_loader = DataLoader(
    test_ds,
    batch_size=8,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Test images:", len(test_ds))


# ------------------------------------------------------------
# Model
# ------------------------------------------------------------

model = smp.Unet(
    encoder_name="resnet50",
    encoder_weights=None,
    in_channels=3,
    classes=1
).to(device)

checkpoint = torch.load(
    BEST_CKPT,
    map_location=device
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

print("Checkpoint epoch:",
      checkpoint["epoch"])

print("Validation Dice:",
      checkpoint["best_val_dice"])


# ------------------------------------------------------------
# Evaluation
# ------------------------------------------------------------

tp_total = 0.0
fp_total = 0.0
fn_total = 0.0
tn_total = 0.0

dice_values = []
iou_values = []
precision_values = []
recall_values = []
accuracy_values = []

with torch.no_grad():

    for batch_idx, (images, masks, names) in enumerate(test_loader):

        images = images.to(
            device,
            non_blocking=True
        )

        masks = masks.to(
            device,
            non_blocking=True
        )

        logits = model(images)

        probs = torch.sigmoid(logits)

        preds = (
            probs >= 0.5
        ).float()

        for i in range(images.shape[0]):

            p = preds[i]
            g = masks[i]

            tp = (
                p * g
            ).sum().item()

            fp = (
                p * (1 - g)
            ).sum().item()

            fn = (
                (1 - p) * g
            ).sum().item()

            tn = (
                (1 - p) * (1 - g)
            ).sum().item()

            dice = (
                2 * tp /
                (2 * tp + fp + fn + 1e-7)
            )

            iou = (
                tp /
                (tp + fp + fn + 1e-7)
            )

            precision = (
                tp /
                (tp + fp + 1e-7)
            )

            recall = (
                tp /
                (tp + fn + 1e-7)
            )

            accuracy = (
                (tp + tn) /
                (tp + tn + fp + fn + 1e-7)
            )

            dice_values.append(dice)
            iou_values.append(iou)
            precision_values.append(precision)
            recall_values.append(recall)
            accuracy_values.append(accuracy)

            tp_total += tp
            fp_total += fp
            fn_total += fn
            tn_total += tn

        if (batch_idx + 1) % 10 == 0:
            print(
                f"Processed "
                f"{min((batch_idx + 1) * 8, len(test_ds))}/"
                f"{len(test_ds)}"
            )


# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

results = {
    "model": "Step16A strong augmentation recovered",
    "checkpoint_epoch": int(checkpoint["epoch"]),
    "validation_dice": float(
        checkpoint["best_val_dice"]
    ),
    "test_images": len(test_ds),

    "dice": float(np.mean(dice_values)),
    "iou": float(np.mean(iou_values)),
    "precision": float(np.mean(precision_values)),
    "recall": float(np.mean(recall_values)),
    "pixel_accuracy": float(
        np.mean(accuracy_values)
    ),

    "dice_std": float(np.std(dice_values)),
    "iou_std": float(np.std(iou_values)),
    "precision_std": float(np.std(precision_values)),
    "recall_std": float(np.std(recall_values)),
    "pixel_accuracy_std": float(
        np.std(accuracy_values)
    )
}

print("\n" + "=" * 60)
print("STEP16A BEACH TEST RESULTS")
print("=" * 60)

for k, v in results.items():
    print(f"{k}: {v}")


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

OUT = Path(
    "/kaggle/working/step16a_recovery/evaluation"
)

OUT.mkdir(
    parents=True,
    exist_ok=True
)

with open(
    OUT / "beach_test_summary.json",
    "w"
) as f:
    json.dump(
        results,
        f,
        indent=2
    )

print(
    "\nSaved:",
    OUT / "beach_test_summary.json"
)

Run the following full-evaluation cell. It will:
1. Reuse the epoch-18 Step16A checkpoint.
2. Evaluate all 2,004 annotated-positive WWTP images.
3. Exclude the six annotation-empty images from the primary metric.
4. Use exactly:
   512 tile → 384 stride → average logits → sigmoid → 0.5
5. Save the CSV every 100 images.
6. Resume automatically if a partial CSV already exists.
7. Print progress every image.
8. Save the final summary.


#Just to check wether the pipeline is working or not

In [ ]:
# ============================================================
# STEP16A — WWTP 20-IMAGE DIAGNOSTIC
# Correct protocol:
# native image -> 512 tiles -> stride 384
# -> average LOGITS -> sigmoid -> threshold 0.5
# ============================================================

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import segmentation_models_pytorch as smp
from PIL import Image


# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

BUNDLE = Path(
    "/kaggle/input/datasets/shanumkhsharma/"
    "kaggle-microplastic-bundle"
)

WWTP_IMAGES = BUNDLE / "data/wwtp/images"
WWTP_MASKS  = BUNDLE / "data/wwtp/masks"

CKPT = Path(
    "/kaggle/working/step16a_recovery/"
    "beach_unet_resnet50_strong_aug_best_dice.pth"
)

OUT = Path(
    "/kaggle/working/step16a_recovery/"
    "evaluation/wwtp_diagnostic"
)

OUT.mkdir(
    parents=True,
    exist_ok=True
)

print("WWTP images:", WWTP_IMAGES)
print("WWTP masks :", WWTP_MASKS)
print("Checkpoint :", CKPT)

assert WWTP_IMAGES.exists()
assert WWTP_MASKS.exists()
assert CKPT.exists()


# ------------------------------------------------------------
# 2. Device
# ------------------------------------------------------------

device = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)

print("\nDevice:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


# ------------------------------------------------------------
# 3. Load model
# ------------------------------------------------------------

model = smp.Unet(
    encoder_name="resnet50",
    encoder_weights=None,
    in_channels=3,
    classes=1
).to(device)

checkpoint = torch.load(
    CKPT,
    map_location=device
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

print("\nCheckpoint epoch:",
      checkpoint["epoch"])

print("Validation Dice:",
      checkpoint["best_val_dice"])


# ------------------------------------------------------------
# 4. Tile-position function
# ------------------------------------------------------------

TILE = 512
STRIDE = 384


def get_positions(length, tile_size=512, stride=384):

    if length <= tile_size:
        return [0]

    positions = list(
        range(
            0,
            length - tile_size + 1,
            stride
        )
    )

    final_position = length - tile_size

    if positions[-1] != final_position:
        positions.append(final_position)

    return positions


# ------------------------------------------------------------
# 5. Single-image WWTP inference
# ------------------------------------------------------------

@torch.inference_mode()
def predict_native(image_np):

    h, w = image_np.shape[:2]

    ys = get_positions(
        h,
        TILE,
        STRIDE
    )

    xs = get_positions(
        w,
        TILE,
        STRIDE
    )

    # IMPORTANT:
    # We accumulate LOGITS, not probabilities.
    logit_sum = np.zeros(
        (h, w),
        dtype=np.float32
    )

    count_map = np.zeros(
        (h, w),
        dtype=np.float32
    )

    for y in ys:

        for x in xs:

            tile = image_np[
                y:y + TILE,
                x:x + TILE
            ]

            # Native WWTP images are 3840x2160.
            # Edge tiles are guaranteed to be 512x512
            # by get_positions().

            tile_tensor = torch.from_numpy(
                tile.astype(np.float32) / 255.0
            ).permute(2, 0, 1).unsqueeze(0)

            tile_tensor = tile_tensor.to(
                device,
                non_blocking=True
            )

            logits = model(
                tile_tensor
            )

            logits = (
                logits[0, 0]
                .detach()
                .float()
                .cpu()
                .numpy()
            )

            logit_sum[
                y:y + TILE,
                x:x + TILE
            ] += logits

            count_map[
                y:y + TILE,
                x:x + TILE
            ] += 1.0

    # Average overlapping LOGITS
    avg_logits = (
        logit_sum /
        np.maximum(count_map, 1.0)
    )

    # Sigmoid only after logit averaging
    probabilities = 1.0 / (
        1.0 + np.exp(-avg_logits)
    )

    prediction = (
        probabilities >= 0.5
    ).astype(np.uint8)

    return prediction, probabilities, len(xs) * len(ys), count_map


# ------------------------------------------------------------
# 6. Metrics
# ------------------------------------------------------------

def calculate_metrics(pred, gt):

    p = pred.astype(bool)
    g = gt.astype(bool)

    tp = np.logical_and(p, g).sum()
    fp = np.logical_and(p, ~g).sum()
    fn = np.logical_and(~p, g).sum()
    tn = np.logical_and(~p, ~g).sum()

    dice = (
        2 * tp /
        (2 * tp + fp + fn + 1e-7)
    )

    iou = (
        tp /
        (tp + fp + fn + 1e-7)
    )

    precision = (
        tp /
        (tp + fp + 1e-7)
    )

    recall = (
        tp /
        (tp + fn + 1e-7)
    )

    accuracy = (
        (tp + tn) /
        (tp + tn + fp + fn + 1e-7)
    )

    return {
        "dice": float(dice),
        "iou": float(iou),
        "precision": float(precision),
        "recall": float(recall),
        "pixel_accuracy": float(accuracy),
        "gt_foreground_fraction":
            float(g.mean()),
        "pred_foreground_fraction":
            float(p.mean()),
        "foreground_area_ratio":
            float(
                p.sum() /
                max(g.sum(), 1)
            )
    }


# ------------------------------------------------------------
# 7. Collect WWTP images
# ------------------------------------------------------------

records = []

for split in ["train", "validation", "test"]:

    image_dir = WWTP_IMAGES / split
    mask_dir = WWTP_MASKS / split

    images = sorted(
        image_dir.glob("*.jpg")
    )

    for image_path in images:

        mask_path = (
            mask_dir /
            f"{image_path.stem}.png"
        )

        if not mask_path.exists():
            raise FileNotFoundError(
                f"Missing mask: {mask_path}"
            )

        records.append({
            "split": split,
            "image_path": image_path,
            "mask_path": mask_path
        })


print("\nTotal WWTP images:", len(records))

assert len(records) == 2010


# ------------------------------------------------------------
# 8. Select annotated-positive images
# ------------------------------------------------------------

positive_records = []

empty_records = []

for r in records:

    mask = np.array(
        Image.open(r["mask_path"]).convert("L")
    )

    if np.any(mask > 0):
        positive_records.append(r)
    else:
        empty_records.append(r)

print("Annotated-positive:", len(positive_records))
print("Annotation-empty  :", len(empty_records))

assert len(positive_records) == 2004
assert len(empty_records) == 6


# ------------------------------------------------------------
# 9. Diagnostic subset
# ------------------------------------------------------------

N = 20

diagnostic_records = positive_records[:N]

print("\nRunning diagnostic on", N, "images.")


# ------------------------------------------------------------
# 10. Run
# ------------------------------------------------------------

results = []

for idx, r in enumerate(
    diagnostic_records,
    start=1
):

    name = r["image_path"].name

    print(
        f"\n[{idx:02d}/{N}] {name}",
        flush=True
    )

    start_time = time.time()

    image = np.array(
        Image.open(
            r["image_path"]
        ).convert("RGB")
    )

    gt = np.array(
        Image.open(
            r["mask_path"]
        ).convert("L")
    )

    print(
        "  image:",
        image.shape,
        "GT:",
        gt.shape,
        flush=True
    )

    pred, probability, n_tiles, coverage = (
        predict_native(image)
    )

    metrics = calculate_metrics(
        pred,
        gt > 127
    )

    elapsed = time.time() - start_time

    result = {
        "split": r["split"],
        "image": name,
        "tiles": n_tiles,
        "seconds": elapsed,
        **metrics
    }

    results.append(result)

    print(
        f"  tiles={n_tiles} "
        f"time={elapsed:.2f}s "
        f"Dice={metrics['dice']:.4f} "
        f"IoU={metrics['iou']:.4f} "
        f"Precision={metrics['precision']:.4f} "
        f"Recall={metrics['recall']:.4f} "
        f"GTfrac={metrics['gt_foreground_fraction']:.6f} "
        f"Predfrac={metrics['pred_foreground_fraction']:.6f}",
        flush=True
    )


# ------------------------------------------------------------
# 11. Save
# ------------------------------------------------------------

df = pd.DataFrame(results)

csv_path = OUT / "wwtp_diagnostic_20_results.csv"

df.to_csv(
    csv_path,
    index=False
)

summary = {
    "checkpoint_epoch":
        int(checkpoint["epoch"]),

    "checkpoint_validation_dice":
        float(checkpoint["best_val_dice"]),

    "n_images":
        len(df),

    "tile_size":
        TILE,

    "stride":
        STRIDE,

    "stitching":
        "average_logits_then_sigmoid",

    "threshold":
        0.5,

    "mean_dice":
        float(df["dice"].mean()),

    "mean_iou":
        float(df["iou"].mean()),

    "mean_precision":
        float(df["precision"].mean()),

    "mean_recall":
        float(df["recall"].mean()),

    "mean_pixel_accuracy":
        float(df["pixel_accuracy"].mean()),

    "mean_gt_foreground_fraction":
        float(
            df["gt_foreground_fraction"].mean()
        ),

    "mean_pred_foreground_fraction":
        float(
            df["pred_foreground_fraction"].mean()
        ),

    "mean_foreground_area_ratio":
        float(
            df["foreground_area_ratio"].mean()
        )
}

summary_path = (
    OUT / "wwtp_diagnostic_20_summary.json"
)

with open(
    summary_path,
    "w"
) as f:
    json.dump(
        summary,
        f,
        indent=2
    )

print("\n" + "=" * 60)
print("WWTP DIAGNOSTIC COMPLETE")
print("=" * 60)

for k, v in summary.items():
    print(f"{k}: {v}")

print("\nResults:")
print(csv_path)

print("\nSummary:")
print(summary_path)

#FULL WWTP EXTERNAL EVALUATION

Run the following full-evaluation cell. It will:
1. Reuse the epoch-18 Step16A checkpoint.
2. Evaluate all 2,004 annotated-positive WWTP images.
3. Exclude the six annotation-empty images from the primary metric.
4. Use exactly:
   512 tile → 384 stride → average logits → sigmoid → 0.5
5. Save the CSV every 100 images.
6. Resume automatically if a partial CSV already exists.
7. Print progress every image.
8. Save the final summary.

In [ ]:
# ============================================================
# STEP16A — FULL WWTP EXTERNAL EVALUATION
#
# Primary evaluation:
#   2004 annotated-positive images
#
# Protocol:
#   native 3840x2160
#   512x512 tiles
#   stride 384
#   average overlapping LOGITS
#   sigmoid
#   threshold 0.5
#
# Incremental saving every 100 images.
# ============================================================

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import segmentation_models_pytorch as smp
from PIL import Image


# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

BUNDLE = Path(
    "/kaggle/input/datasets/shanumkhsharma/"
    "kaggle-microplastic-bundle"
)

WWTP_IMAGES = BUNDLE / "data/wwtp/images"
WWTP_MASKS = BUNDLE / "data/wwtp/masks"

CKPT = Path(
    "/kaggle/working/step16a_recovery/"
    "beach_unet_resnet50_strong_aug_best_dice.pth"
)

OUT = Path(
    "/kaggle/working/step16a_recovery/"
    "evaluation/wwtp_full"
)

OUT.mkdir(
    parents=True,
    exist_ok=True
)

RESULTS_CSV = OUT / "wwtp_step16a_per_image.csv"
SUMMARY_JSON = OUT / "wwtp_step16a_summary.json"

print("WWTP images:", WWTP_IMAGES)
print("WWTP masks :", WWTP_MASKS)
print("Checkpoint :", CKPT)

assert WWTP_IMAGES.exists()
assert WWTP_MASKS.exists()
assert CKPT.exists()


# ------------------------------------------------------------
# 2. Device
# ------------------------------------------------------------

device = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)

print("\nDevice:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


# ------------------------------------------------------------
# 3. Model
# ------------------------------------------------------------

model = smp.Unet(
    encoder_name="resnet50",
    encoder_weights=None,
    in_channels=3,
    classes=1
).to(device)

checkpoint = torch.load(
    CKPT,
    map_location=device
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

print("\nCheckpoint epoch:",
      checkpoint["epoch"])

print("Validation Dice:",
      checkpoint["best_val_dice"])


# ------------------------------------------------------------
# 4. Tiling
# ------------------------------------------------------------

TILE = 512
STRIDE = 384
THRESHOLD = 0.5


def get_positions(
    length,
    tile_size=512,
    stride=384
):

    if length <= tile_size:
        return [0]

    positions = list(
        range(
            0,
            length - tile_size + 1,
            stride
        )
    )

    final_position = (
        length - tile_size
    )

    if positions[-1] != final_position:
        positions.append(
            final_position
        )

    return positions


# ------------------------------------------------------------
# 5. Native-resolution inference
# ------------------------------------------------------------

@torch.inference_mode()
def predict_native(image_np):

    h, w = image_np.shape[:2]

    ys = get_positions(
        h,
        TILE,
        STRIDE
    )

    xs = get_positions(
        w,
        TILE,
        STRIDE
    )

    logit_sum = np.zeros(
        (h, w),
        dtype=np.float32
    )

    count_map = np.zeros(
        (h, w),
        dtype=np.float32
    )

    for y in ys:

        for x in xs:

            tile = image_np[
                y:y + TILE,
                x:x + TILE
            ]

            tensor = torch.from_numpy(
                tile.astype(np.float32) / 255.0
            ).permute(
                2, 0, 1
            ).unsqueeze(0)

            tensor = tensor.to(
                device,
                non_blocking=True
            )

            logits = model(
                tensor
            )[0, 0]

            logits = (
                logits
                .float()
                .cpu()
                .numpy()
            )

            logit_sum[
                y:y + TILE,
                x:x + TILE
            ] += logits

            count_map[
                y:y + TILE,
                x:x + TILE
            ] += 1.0

    avg_logits = (
        logit_sum /
        np.maximum(
            count_map,
            1.0
        )
    )

    probabilities = 1.0 / (
        1.0 +
        np.exp(-avg_logits)
    )

    prediction = (
        probabilities >= THRESHOLD
    ).astype(np.uint8)

    return prediction


# ------------------------------------------------------------
# 6. Metrics
# ------------------------------------------------------------

def calculate_metrics(
    prediction,
    ground_truth
):

    p = prediction.astype(bool)
    g = ground_truth.astype(bool)

    tp = np.logical_and(
        p,
        g
    ).sum()

    fp = np.logical_and(
        p,
        ~g
    ).sum()

    fn = np.logical_and(
        ~p,
        g
    ).sum()

    tn = np.logical_and(
        ~p,
        ~g
    ).sum()

    dice = (
        2 * tp /
        (2 * tp + fp + fn + 1e-7)
    )

    iou = (
        tp /
        (tp + fp + fn + 1e-7)
    )

    precision = (
        tp /
        (tp + fp + 1e-7)
    )

    recall = (
        tp /
        (tp + fn + 1e-7)
    )

    accuracy = (
        (tp + tn) /
        (
            tp + tn +
            fp + fn + 1e-7
        )
    )

    gt_fraction = (
        g.mean()
    )

    pred_fraction = (
        p.mean()
    )

    area_ratio = (
        p.sum() /
        max(g.sum(), 1)
    )

    return {
        "dice": float(dice),
        "iou": float(iou),
        "precision": float(precision),
        "recall": float(recall),
        "pixel_accuracy": float(accuracy),
        "gt_foreground_fraction":
            float(gt_fraction),
        "pred_foreground_fraction":
            float(pred_fraction),
        "foreground_area_ratio":
            float(area_ratio)
    }


# ------------------------------------------------------------
# 7. Collect all WWTP records
# ------------------------------------------------------------

records = []

for split in [
    "train",
    "validation",
    "test"
]:

    image_dir = (
        WWTP_IMAGES / split
    )

    mask_dir = (
        WWTP_MASKS / split
    )

    for image_path in sorted(
        image_dir.glob("*.jpg")
    ):

        mask_path = (
            mask_dir /
            f"{image_path.stem}.png"
        )

        assert mask_path.exists()

        records.append({
            "split": split,
            "image_path": image_path,
            "mask_path": mask_path
        })


print("\nTotal images:", len(records))

assert len(records) == 2010


# ------------------------------------------------------------
# 8. Separate annotated-positive / empty
# ------------------------------------------------------------

positive_records = []
empty_records = []

for r in records:

    mask = np.array(
        Image.open(
            r["mask_path"]
        ).convert("L")
    )

    if np.any(mask > 0):

        positive_records.append(r)

    else:

        empty_records.append(r)


print(
    "Annotated-positive:",
    len(positive_records)
)

print(
    "Annotation-empty:",
    len(empty_records)
)

assert len(positive_records) == 2004
assert len(empty_records) == 6


# ------------------------------------------------------------
# 9. Resume from previous partial CSV if present
# ------------------------------------------------------------

completed = {}

if RESULTS_CSV.exists():

    old_df = pd.read_csv(
        RESULTS_CSV
    )

    for _, row in old_df.iterrows():

        completed[
            row["image"]
        ] = row.to_dict()

    print(
        "\nExisting results found:",
        len(completed)
    )

else:

    print(
        "\nNo previous results found."
    )


# ------------------------------------------------------------
# 10. Full evaluation
# ------------------------------------------------------------

start_global = time.time()

for idx, r in enumerate(
    positive_records,
    start=1
):

    image_name = (
        r["image_path"].name
    )

    if image_name in completed:

        if idx % 100 == 0:

            print(
                f"Skipping completed "
                f"{idx}/2004",
                flush=True
            )

        continue

    print(
        f"\n[{idx}/2004] "
        f"{image_name}",
        flush=True
    )

    t0 = time.time()

    image = np.array(
        Image.open(
            r["image_path"]
        ).convert("RGB")
    )

    gt = np.array(
        Image.open(
            r["mask_path"]
        ).convert("L")
    )

    prediction = predict_native(
        image
    )

    metrics = calculate_metrics(
        prediction,
        gt > 127
    )

    elapsed = (
        time.time() - t0
    )

    completed[
        image_name
    ] = {
        "split": r["split"],
        "image": image_name,
        "seconds": elapsed,
        **metrics
    }

    print(
        f"  time={elapsed:.2f}s "
        f"Dice={metrics['dice']:.4f} "
        f"IoU={metrics['iou']:.4f} "
        f"Precision={metrics['precision']:.4f} "
        f"Recall={metrics['recall']:.4f} "
        f"PredFrac={metrics['pred_foreground_fraction']:.6f}",
        flush=True
    )

    # --------------------------------------------------------
    # Save every 100 completed images
    # --------------------------------------------------------

    if (
        len(completed) % 100 == 0
        or idx == 2004
    ):

        df_save = pd.DataFrame(
            list(completed.values())
        )

        df_save.to_csv(
            RESULTS_CSV,
            index=False
        )

        print(
            f"\n*** CHECKPOINT SAVED: "
            f"{len(completed)}/2004 ***",
            flush=True
        )


# ------------------------------------------------------------
# 11. Final dataframe
# ------------------------------------------------------------

df = pd.DataFrame(
    list(completed.values())
)

df = df.sort_values(
    "image"
).reset_index(drop=True)

df.to_csv(
    RESULTS_CSV,
    index=False
)


# ------------------------------------------------------------
# 12. Final summary
# ------------------------------------------------------------

summary = {

    "model":
        "Step16A strong augmentation recovered",

    "checkpoint_epoch":
        int(checkpoint["epoch"]),

    "validation_dice":
        float(
            checkpoint["best_val_dice"]
        ),

    "primary_images":
        len(df),

    "annotation_empty_images":
        len(empty_records),

    "tile_size":
        TILE,

    "stride":
        STRIDE,

    "threshold":
        THRESHOLD,

    "stitching":
        "average_logits_then_sigmoid",

    "dice_mean":
        float(df["dice"].mean()),

    "dice_std":
        float(df["dice"].std()),

    "dice_median":
        float(df["dice"].median()),

    "iou_mean":
        float(df["iou"].mean()),

    "iou_std":
        float(df["iou"].std()),

    "iou_median":
        float(df["iou"].median()),

    "precision_mean":
        float(df["precision"].mean()),

    "precision_std":
        float(df["precision"].std()),

    "recall_mean":
        float(df["recall"].mean()),

    "recall_std":
        float(df["recall"].std()),

    "pixel_accuracy_mean":
        float(
            df["pixel_accuracy"].mean()
        ),

    "pixel_accuracy_std":
        float(
            df["pixel_accuracy"].std()
        ),

    "gt_foreground_fraction_mean":
        float(
            df[
                "gt_foreground_fraction"
            ].mean()
        ),

    "pred_foreground_fraction_mean":
        float(
            df[
                "pred_foreground_fraction"
            ].mean()
        ),

    "foreground_area_ratio_mean":
        float(
            df[
                "foreground_area_ratio"
            ].mean()
        ),

    "foreground_area_ratio_median":
        float(
            df[
                "foreground_area_ratio"
            ].median()
        )
}

with open(
    SUMMARY_JSON,
    "w"
) as f:

    json.dump(
        summary,
        f,
        indent=2
    )


print("\n" + "=" * 70)
print("FULL WWTP EVALUATION COMPLETE")
print("=" * 70)

for k, v in summary.items():

    print(
        f"{k}: {v}"
    )

print(
    "\nCSV:",
    RESULTS_CSV
)

print(
    "SUMMARY:",
    SUMMARY_JSON
)

#Saving Experiment 2A results



In [ ]:
from pathlib import Path
import shutil

SRC = Path("/kaggle/working/step16a_recovery")

SAVE = Path("/kaggle/working/step16A_final_artifacts")
SAVE.mkdir(parents=True, exist_ok=True)

files_to_save = [
    SRC / "beach_unet_resnet50_strong_aug_best_dice.pth",
    SRC / "training_history.json",
    SRC / "evaluation/beach_test_summary.json",
    SRC / "evaluation/wwtp_full/wwtp_step16a_per_image.csv",
    SRC / "evaluation/wwtp_full/wwtp_step16a_summary.json",
    SRC / "evaluation/wwtp_diagnostic/wwtp_diagnostic_20_results.csv",
    SRC / "evaluation/wwtp_diagnostic/wwtp_diagnostic_20_summary.json",
]

for src in files_to_save:
    if src.exists():
        dst = SAVE / src.name
        shutil.copy2(src, dst)
        print("Saved:", dst,
              f"({dst.stat().st_size / 1024**2:.2f} MB)")
    else:
        print("MISSING:", src)

print("\nFinal artifact directory:")
for p in SAVE.iterdir():
    print(" ", p.name, f"{p.stat().st_size / 1024**2:.2f} MB")

Step16B — Domain-Generalization via source-domain appearance/style robustness

Instead of simply increasing geometric augmentation, introduce stronger appearance variation during Beach training—illumination, contrast, color, blur, and image-quality changes—while retaining the same U-Net + ResNet-50 architecture, loss, optimizer, 512×512 input, and Beach train/validation/test protocol.

#Expeirment 2B - image agumentation

In [ ]:
# ============================================================
# STEP17 — APPEARANCE-FOCUSED DOMAIN GENERALIZATION
#
# Same as Step16A except augmentation policy.
#
# WWTP is COMPLETELY UNSEEN during training/validation.
# ============================================================

import os
import json
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image

import segmentation_models_pytorch as smp
import albumentations as A


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

SEED = 42

BATCH_SIZE = 8
MAX_EPOCHS = 50
PATIENCE = 10

LR = 1e-4
WEIGHT_DECAY = 1e-4

IMAGE_SIZE = 512

device = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


# ------------------------------------------------------------
# 2. Dataset
# ------------------------------------------------------------

BUNDLE = Path(
    "/kaggle/input/datasets/shanumkhsharma/"
    "microplastic-step16a-recovery/kaggle_bundle"
)

TRAIN_IMG = BUNDLE / "data/train/images"
TRAIN_MSK = BUNDLE / "data/train/masks"

VAL_IMG = BUNDLE / "data/val/images"
VAL_MSK = BUNDLE / "data/val/masks"

TEST_IMG = BUNDLE / "data/test/images"
TEST_MSK = BUNDLE / "data/test/masks"


# ------------------------------------------------------------
# 3. Output
# ------------------------------------------------------------

OUT = Path(
    "/kaggle/working/step17_appearance_generalization"
)

OUT.mkdir(
    parents=True,
    exist_ok=True
)

RECOVERY_DIR = OUT / "recovery_checkpoints"
RECOVERY_DIR.mkdir(
    parents=True,
    exist_ok=True
)

BEST_CKPT = (
    OUT /
    "beach_unet_resnet50_step17_best_dice.pth"
)

HISTORY_FILE = (
    OUT /
    "training_history.json"
)


# ------------------------------------------------------------
# 4. Dataset class
# ------------------------------------------------------------

class BeachDataset(Dataset):

    def __init__(
        self,
        image_dir,
        mask_dir,
        augment=False
    ):

        self.image_dir = Path(image_dir)
        self.mask_dir = Path(mask_dir)

        self.images = sorted([
            p for p in self.image_dir.iterdir()
            if p.suffix.lower()
            in [".jpg", ".jpeg", ".png"]
        ])

        self.augment = augment

        if augment:

            self.transform = A.Compose([

                # -----------------------------
                # Geometric robustness
                # -----------------------------

                A.HorizontalFlip(
                    p=0.5
                ),

                A.Affine(
                    scale=(0.90, 1.10),
                    translate_percent=(-0.10, 0.10),
                    rotate=(-20, 20),
                    interpolation=1,
                    mask_interpolation=0,
                    p=0.8
                ),

                # -----------------------------
                # Appearance/domain robustness
                # -----------------------------

                A.RandomBrightnessContrast(
                    brightness_limit=0.25,
                    contrast_limit=0.25,
                    p=0.6
                ),

                A.HueSaturationValue(
                    hue_shift_limit=0,
                    sat_shift_limit=30,
                    val_shift_limit=20,
                    p=0.5
                ),

                A.RandomGamma(
                    gamma_limit=(70, 130),
                    p=0.4
                ),

                A.GaussianBlur(
                    blur_limit=(3, 7),
                    p=0.25
                )
            ])

        else:

            self.transform = None


    def __len__(self):
        return len(self.images)


    def __getitem__(self, idx):

        image_path = self.images[idx]

        mask_path = (
            self.mask_dir /
            f"{image_path.stem}.png"
        )

        image = np.array(
            Image.open(
                image_path
            ).convert("RGB")
        )

        mask = np.array(
            Image.open(
                mask_path
            ).convert("L")
        )

        if self.transform is not None:

            transformed = self.transform(
                image=image,
                mask=mask
            )

            image = transformed["image"]
            mask = transformed["mask"]


        # -------------------------------------
        # 256 -> 512
        # -------------------------------------

        image = np.array(
            Image.fromarray(image).resize(
                (IMAGE_SIZE, IMAGE_SIZE),
                Image.Resampling.BILINEAR
            )
        )

        mask = np.array(
            Image.fromarray(mask).resize(
                (IMAGE_SIZE, IMAGE_SIZE),
                Image.Resampling.NEAREST
            )
        )


        image = torch.from_numpy(
            image.astype(np.float32) / 255.0
        ).permute(2, 0, 1)

        mask = torch.from_numpy(
            (mask > 127).astype(np.float32)
        ).unsqueeze(0)

        return image, mask


# ------------------------------------------------------------
# 5. Loaders
# ------------------------------------------------------------

train_ds = BeachDataset(
    TRAIN_IMG,
    TRAIN_MSK,
    augment=True
)

val_ds = BeachDataset(
    VAL_IMG,
    VAL_MSK,
    augment=False
)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("\nDataset:")
print("Train:", len(train_ds))
print("Val  :", len(val_ds))

assert len(train_ds) == 790
assert len(val_ds) == 169


# ------------------------------------------------------------
# 6. Model
# ------------------------------------------------------------

model = smp.Unet(
    encoder_name="resnet50",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1
).to(device)

print(
    "Parameters:",
    sum(
        p.numel()
        for p in model.parameters()
    )
)


# ------------------------------------------------------------
# 7. Loss
# ------------------------------------------------------------

bce_loss = nn.BCEWithLogitsLoss()


def dice_loss(
    logits,
    targets,
    eps=1e-7
):

    probs = torch.sigmoid(logits)

    intersection = (
        probs * targets
    ).sum(
        dim=(1, 2, 3)
    )

    denominator = (
        probs.sum(
            dim=(1, 2, 3)
        )
        +
        targets.sum(
            dim=(1, 2, 3)
        )
    )

    dice = (
        (2 * intersection + eps)
        /
        (denominator + eps)
    )

    return 1 - dice.mean()


def total_loss(
    logits,
    targets
):

    return (
        0.5 *
        bce_loss(
            logits,
            targets
        )
        +
        0.5 *
        dice_loss(
            logits,
            targets
        )
    )


# ------------------------------------------------------------
# 8. Metrics
# ------------------------------------------------------------

def calculate_metrics(
    logits,
    targets
):

    probs = torch.sigmoid(logits)

    preds = (
        probs >= 0.5
    ).float()

    tp = (
        preds * targets
    ).sum()

    fp = (
        preds * (1 - targets)
    ).sum()

    fn = (
        (1 - preds) * targets
    ).sum()

    tn = (
        (1 - preds) *
        (1 - targets)
    ).sum()

    dice = (
        2 * tp /
        (
            2 * tp +
            fp +
            fn +
            1e-7
        )
    )

    iou = (
        tp /
        (
            tp +
            fp +
            fn +
            1e-7
        )
    )

    precision = (
        tp /
        (
            tp +
            fp +
            1e-7
        )
    )

    recall = (
        tp /
        (
            tp +
            fn +
            1e-7
        )
    )

    accuracy = (
        (tp + tn) /
        (
            tp +
            tn +
            fp +
            fn +
            1e-7
        )
    )

    return (
        dice.item(),
        iou.item(),
        precision.item(),
        recall.item(),
        accuracy.item()
    )


# ------------------------------------------------------------
# 9. Optimizer
# ------------------------------------------------------------

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)


# ------------------------------------------------------------
# 10. Validation
# ------------------------------------------------------------

@torch.no_grad()
def evaluate(loader):

    model.eval()

    losses = []

    dice_values = []
    iou_values = []
    precision_values = []
    recall_values = []
    accuracy_values = []

    for images, masks in loader:

        images = images.to(
            device,
            non_blocking=True
        )

        masks = masks.to(
            device,
            non_blocking=True
        )

        with torch.cuda.amp.autocast(
            enabled=torch.cuda.is_available()
        ):

            logits = model(images)

            loss = total_loss(
                logits,
                masks
            )

        losses.append(
            loss.item()
        )

        metrics = calculate_metrics(
            logits,
            masks
        )

        dice_values.append(
            metrics[0]
        )

        iou_values.append(
            metrics[1]
        )

        precision_values.append(
            metrics[2]
        )

        recall_values.append(
            metrics[3]
        )

        accuracy_values.append(
            metrics[4]
        )

    return {
        "loss": float(
            np.mean(losses)
        ),

        "dice": float(
            np.mean(dice_values)
        ),

        "iou": float(
            np.mean(iou_values)
        ),

        "precision": float(
            np.mean(precision_values)
        ),

        "recall": float(
            np.mean(recall_values)
        ),

        "pixel_accuracy": float(
            np.mean(accuracy_values)
        )
    }


# ------------------------------------------------------------
# 11. Training
# ------------------------------------------------------------

scaler = torch.cuda.amp.GradScaler(
    enabled=torch.cuda.is_available()
)

best_val_dice = -1.0
epochs_without_improvement = 0
history = []

print("\n" + "=" * 70)
print("STEP17 TRAINING")
print("=" * 70)

for epoch in range(
    1,
    MAX_EPOCHS + 1
):

    model.train()

    train_losses = []

    for images, masks in train_loader:

        images = images.to(
            device,
            non_blocking=True
        )

        masks = masks.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.cuda.amp.autocast(
            enabled=torch.cuda.is_available()
        ):

            logits = model(images)

            loss = total_loss(
                logits,
                masks
            )

        scaler.scale(
            loss
        ).backward()

        scaler.step(
            optimizer
        )

        scaler.update()

        train_losses.append(
            loss.item()
        )


    train_loss = float(
        np.mean(train_losses)
    )

    val = evaluate(
        val_loader
    )

    record = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val["loss"],
        "val_dice": val["dice"],
        "val_iou": val["iou"],
        "val_precision": val["precision"],
        "val_recall": val["recall"],
        "val_pixel_accuracy":
            val["pixel_accuracy"]
    }

    history.append(record)


    # --------------------------------------------------------
    # Best checkpoint
    # --------------------------------------------------------

    if val["dice"] > best_val_dice:

        best_val_dice = val["dice"]

        epochs_without_improvement = 0

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict":
                    model.state_dict(),
                "optimizer_state_dict":
                    optimizer.state_dict(),
                "best_val_dice":
                    best_val_dice,
                "history": history,
                "experiment":
                    "step17_appearance_domain_generalization",
                "seed": SEED
            },
            BEST_CKPT
        )

        best_marker = " <<< BEST"

    else:

        epochs_without_improvement += 1

        best_marker = ""


    # --------------------------------------------------------
    # Recovery checkpoint
    # --------------------------------------------------------

    torch.save(
        {
            "epoch": epoch,
            "model_state_dict":
                model.state_dict(),
            "optimizer_state_dict":
                optimizer.state_dict(),
            "best_val_dice":
                best_val_dice,
            "history": history,
            "experiment":
                "step17_appearance_domain_generalization",
            "seed": SEED
        },
        RECOVERY_DIR /
        f"epoch_{epoch:03d}.pth"
    )


    with open(
        HISTORY_FILE,
        "w"
    ) as f:

        json.dump(
            history,
            f,
            indent=2
        )


    print(
        f"Epoch {epoch:02d}/{MAX_EPOCHS} | "
        f"Train {train_loss:.4f} | "
        f"Val {val['loss']:.4f} | "
        f"Dice {val['dice']:.4f} | "
        f"IoU {val['iou']:.4f} | "
        f"Precision {val['precision']:.4f} | "
        f"Recall {val['recall']:.4f} | "
        f"Best {best_val_dice:.4f}"
        f"{best_marker}",
        flush=True
    )


    # --------------------------------------------------------
    # Early stopping
    # --------------------------------------------------------

    if (
        epochs_without_improvement
        >= PATIENCE
    ):

        print(
            "\nEarly stopping triggered."
        )

        break


print("\n" + "=" * 70)
print("STEP17 TRAINING COMPLETE")
print("=" * 70)

print(
    "Best validation Dice:",
    best_val_dice
)

print(
    "Best checkpoint:",
    BEST_CKPT
)

print(
    "History:",
    HISTORY_FILE
)

In [ ]:
from pathlib import Path
import torch

OUT = Path(
    "/kaggle/working/step17_appearance_generalization"
)

BEST = OUT / "beach_unet_resnet50_step17_best_dice.pth"

print("Best checkpoint exists:", BEST.exists())

if BEST.exists():
    print(
        "Size:",
        BEST.stat().st_size / (1024**2),
        "MB"
    )

    ckpt = torch.load(
        BEST,
        map_location="cpu"
    )

    print("Saved epoch:", ckpt["epoch"])
    print("Best validation Dice:",
          ckpt["best_val_dice"])
    print("History entries:",
          len(ckpt["history"]))
else:
    print("BEST CHECKPOINT IS MISSING")

2B-beach test

In [ ]:
# ============================================================
# STEP17 — BEACH TEST EVALUATION
# ============================================================

import json
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import segmentation_models_pytorch as smp


# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

BUNDLE = Path(
    "/kaggle/input/datasets/shanumkhsharma/"
    "microplastic-step16a-recovery/kaggle_bundle"
)

TEST_IMG = BUNDLE / "data/test/images"
TEST_MSK = BUNDLE / "data/test/masks"

CKPT = Path(
    "/kaggle/working/"
    "step17_appearance_generalization/"
    "beach_unet_resnet50_step17_best_dice.pth"
)

OUT = Path(
    "/kaggle/working/"
    "step17_appearance_generalization/"
    "evaluation"
)

OUT.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

device = torch.device(
    "cuda:0" if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )


# ------------------------------------------------------------
# Dataset
# ------------------------------------------------------------

class BeachTestDataset(Dataset):

    def __init__(
        self,
        image_dir,
        mask_dir
    ):

        self.image_dir = Path(
            image_dir
        )

        self.mask_dir = Path(
            mask_dir
        )

        self.images = sorted([
            p for p in self.image_dir.iterdir()
            if p.suffix.lower()
            in [".jpg", ".jpeg", ".png"]
        ])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):

        image_path = self.images[idx]

        mask_path = (
            self.mask_dir /
            f"{image_path.stem}.png"
        )

        image = np.array(
            Image.open(
                image_path
            ).convert("RGB")
        )

        mask = np.array(
            Image.open(
                mask_path
            ).convert("L")
        )

        image = np.array(
            Image.fromarray(image).resize(
                (512, 512),
                Image.Resampling.BILINEAR
            )
        )

        mask = np.array(
            Image.fromarray(mask).resize(
                (512, 512),
                Image.Resampling.NEAREST
            )
        )

        image = torch.from_numpy(
            image.astype(np.float32) / 255.0
        ).permute(2, 0, 1)

        mask = torch.from_numpy(
            (mask > 127).astype(np.float32)
        ).unsqueeze(0)

        return image, mask


test_ds = BeachTestDataset(
    TEST_IMG,
    TEST_MSK
)

test_loader = DataLoader(
    test_ds,
    batch_size=8,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(
    "Test images:",
    len(test_ds)
)

assert len(test_ds) == 169


# ------------------------------------------------------------
# Model
# ------------------------------------------------------------

model = smp.Unet(
    encoder_name="resnet50",
    encoder_weights=None,
    in_channels=3,
    classes=1
).to(device)

checkpoint = torch.load(
    CKPT,
    map_location=device
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

print(
    "Checkpoint epoch:",
    checkpoint["epoch"]
)

print(
    "Validation Dice:",
    checkpoint["best_val_dice"]
)


# ------------------------------------------------------------
# Evaluation
# ------------------------------------------------------------

dice_values = []
iou_values = []
precision_values = []
recall_values = []
accuracy_values = []

with torch.inference_mode():

    for batch_idx, (
        images,
        masks
    ) in enumerate(test_loader):

        images = images.to(
            device,
            non_blocking=True
        )

        masks = masks.to(
            device,
            non_blocking=True
        )

        logits = model(images)

        predictions = (
            torch.sigmoid(logits)
            >= 0.5
        ).float()

        for i in range(
            images.shape[0]
        ):

            p = predictions[i]
            g = masks[i]

            tp = (
                p * g
            ).sum().item()

            fp = (
                p * (1 - g)
            ).sum().item()

            fn = (
                (1 - p) * g
            ).sum().item()

            tn = (
                (1 - p) *
                (1 - g)
            ).sum().item()

            dice = (
                2 * tp /
                (
                    2 * tp +
                    fp +
                    fn +
                    1e-7
                )
            )

            iou = (
                tp /
                (
                    tp +
                    fp +
                    fn +
                    1e-7
                )
            )

            precision = (
                tp /
                (
                    tp +
                    fp +
                    1e-7
                )
            )

            recall = (
                tp /
                (
                    tp +
                    fn +
                    1e-7
                )
            )

            accuracy = (
                (tp + tn) /
                (
                    tp + tn +
                    fp + fn +
                    1e-7
                )
            )

            dice_values.append(dice)
            iou_values.append(iou)
            precision_values.append(
                precision
            )
            recall_values.append(
                recall
            )
            accuracy_values.append(
                accuracy
            )

        if (batch_idx + 1) % 10 == 0:

            print(
                f"Processed "
                f"{min((batch_idx + 1) * 8, 169)}/169",
                flush=True
            )


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

results = {

    "model":
        "Step17 appearance-focused",

    "checkpoint_epoch":
        int(checkpoint["epoch"]),

    "validation_dice":
        float(
            checkpoint["best_val_dice"]
        ),

    "test_images":
        169,

    "dice":
        float(np.mean(dice_values)),

    "iou":
        float(np.mean(iou_values)),

    "precision":
        float(
            np.mean(precision_values)
        ),

    "recall":
        float(
            np.mean(recall_values)
        ),

    "pixel_accuracy":
        float(
            np.mean(accuracy_values)
        ),

    "dice_std":
        float(np.std(dice_values)),

    "iou_std":
        float(np.std(iou_values)),

    "precision_std":
        float(
            np.std(precision_values)
        ),

    "recall_std":
        float(
            np.std(recall_values)
        ),

    "pixel_accuracy_std":
        float(
            np.std(accuracy_values)
        )
}


print("\n" + "=" * 60)
print("STEP17 BEACH TEST RESULTS")
print("=" * 60)

for k, v in results.items():
    print(f"{k}: {v}")


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

SUMMARY = (
    OUT /
    "step17_beach_test_summary.json"
)

with open(
    SUMMARY,
    "w"
) as f:

    json.dump(
        results,
        f,
        indent=2
    )

print(
    "\nSaved:",
    SUMMARY
)

2B-WWTP evaluation


In [ ]:
# ============================================================
# STEP17 — WWTP EXTERNAL EVALUATION
# ============================================================

import json
import math
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import segmentation_models_pytorch as smp
from PIL import Image


# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

WWTP_ROOT = Path(
    "/kaggle/input/datasets/shanumkhsharma/"
    "kaggle-microplastic-bundle/data/wwtp"
)

CKPT = Path(
    "/kaggle/working/"
    "step17_appearance_generalization/"
    "beach_unet_resnet50_step17_best_dice.pth"
)

OUT = Path(
    "/kaggle/working/"
    "step17_appearance_generalization/"
    "evaluation/wwtp_full"
)

OUT.mkdir(
    parents=True,
    exist_ok=True
)

RESULTS_CSV = OUT / "wwtp_step17_per_image.csv"
SUMMARY_JSON = OUT / "wwtp_step17_summary.json"


# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

TILE_SIZE = 512
STRIDE = 384
THRESHOLD = 0.5

OVERLAP = TILE_SIZE - STRIDE

print("=" * 70)
print("STEP17 — WWTP EXTERNAL EVALUATION")
print("=" * 70)

print("Tile size:", TILE_SIZE)
print("Stride:", STRIDE)
print("Overlap:", OVERLAP)
print("Threshold:", THRESHOLD)
print("Checkpoint:", CKPT)


# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

device = torch.device(
    "cuda:0" if torch.cuda.is_available()
    else "cpu"
)

print("\nDevice:", device)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )


# ------------------------------------------------------------
# Load checkpoint
# ------------------------------------------------------------

checkpoint = torch.load(
    CKPT,
    map_location=device
)

print("\nCheckpoint epoch:")
print(checkpoint["epoch"])

print(
    "Best validation Dice:",
    checkpoint["best_val_dice"]
)


# ------------------------------------------------------------
# Model
# ------------------------------------------------------------

model = smp.Unet(
    encoder_name="resnet50",
    encoder_weights=None,
    in_channels=3,
    classes=1
).to(device)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

print(
    "Model loaded successfully."
)


# ------------------------------------------------------------
# WWTP split paths
# ------------------------------------------------------------

image_dirs = {
    "train":
        WWTP_ROOT / "images" / "train",

    "validation":
        WWTP_ROOT / "images" / "validation",

    "test":
        WWTP_ROOT / "images" / "test"
}

mask_dirs = {
    "train":
        WWTP_ROOT / "masks" / "train",

    "validation":
        WWTP_ROOT / "masks" / "validation",

    "test":
        WWTP_ROOT / "masks" / "test"
}


# ------------------------------------------------------------
# Collect images
# ------------------------------------------------------------

records = []

for split in [
    "train",
    "validation",
    "test"
]:

    image_dir = image_dirs[split]

    image_paths = sorted(
        image_dir.glob("*.jpg")
    )

    print(
        f"{split}: {len(image_paths)} images"
    )

    for image_path in image_paths:

        mask_path = (
            mask_dirs[split] /
            f"{image_path.stem}.png"
        )

        if not mask_path.exists():

            raise FileNotFoundError(
                f"Missing mask:\n{mask_path}"
            )

        records.append({
            "split": split,
            "image_path": image_path,
            "mask_path": mask_path
        })


print(
    "\nTotal WWTP images:",
    len(records)
)

assert len(records) == 2010


# ------------------------------------------------------------
# Tile positions
# ------------------------------------------------------------

def get_positions(
    length,
    tile_size=512,
    stride=384
):

    if length <= tile_size:
        return [0]

    positions = list(
        range(
            0,
            length - tile_size + 1,
            stride
        )
    )

    final_position = (
        length - tile_size
    )

    if positions[-1] != final_position:
        positions.append(
            final_position
        )

    return positions


# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------

def compute_metrics(
    prediction,
    target
):

    prediction = prediction.astype(
        np.uint8
    )

    target = target.astype(
        np.uint8
    )

    tp = np.logical_and(
        prediction == 1,
        target == 1
    ).sum()

    fp = np.logical_and(
        prediction == 1,
        target == 0
    ).sum()

    fn = np.logical_and(
        prediction == 0,
        target == 1
    ).sum()

    tn = np.logical_and(
        prediction == 0,
        target == 0
    ).sum()

    dice = (
        2.0 * tp /
        (
            2.0 * tp +
            fp +
            fn +
            1e-7
        )
    )

    iou = (
        tp /
        (
            tp +
            fp +
            fn +
            1e-7
        )
    )

    precision = (
        tp /
        (
            tp +
            fp +
            1e-7
        )
    )

    recall = (
        tp /
        (
            tp +
            fn +
            1e-7
        )
    )

    accuracy = (
        (tp + tn) /
        (
            tp +
            tn +
            fp +
            fn +
            1e-7
        )
    )

    return {
        "dice": float(dice),
        "iou": float(iou),
        "precision": float(precision),
        "recall": float(recall),
        "pixel_accuracy": float(accuracy),
        "tp": int(tp),
        "fp": int(fp),
        "fn": int(fn),
        "tn": int(tn)
    }


# ------------------------------------------------------------
# Single-image tiled inference
# ------------------------------------------------------------

def predict_tiled(
    image_np,
    model,
    device,
    tile_size=512,
    stride=384
):

    height, width = image_np.shape[:2]

    y_positions = get_positions(
        height,
        tile_size,
        stride
    )

    x_positions = get_positions(
        width,
        tile_size,
        stride
    )

    logits_sum = np.zeros(
        (height, width),
        dtype=np.float32
    )

    coverage = np.zeros(
        (height, width),
        dtype=np.float32
    )

    tiles_processed = 0

    with torch.inference_mode():

        for y in y_positions:

            for x in x_positions:

                tile = image_np[
                    y:y + tile_size,
                    x:x + tile_size
                ]

                # Should be exactly 512×512
                if (
                    tile.shape[0] != tile_size
                    or
                    tile.shape[1] != tile_size
                ):

                    padded = np.zeros(
                        (
                            tile_size,
                            tile_size,
                            3
                        ),
                        dtype=np.uint8
                    )

                    padded[
                        :tile.shape[0],
                        :tile.shape[1]
                    ] = tile

                    tile = padded

                tile_tensor = torch.from_numpy(
                    tile.astype(
                        np.float32
                    ) / 255.0
                ).permute(
                    2, 0, 1
                ).unsqueeze(0)

                tile_tensor = tile_tensor.to(
                    device,
                    non_blocking=True
                )

                logits = model(
                    tile_tensor
                )

                logits = (
                    logits[0, 0]
                    .detach()
                    .float()
                    .cpu()
                    .numpy()
                )

                actual_h = min(
                    tile_size,
                    height - y
                )

                actual_w = min(
                    tile_size,
                    width - x
                )

                logits_sum[
                    y:y + actual_h,
                    x:x + actual_w
                ] += logits[
                    :actual_h,
                    :actual_w
                ]

                coverage[
                    y:y + actual_h,
                    x:x + actual_w
                ] += 1.0

                tiles_processed += 1

    assert np.all(
        coverage > 0
    ), "Incomplete tile coverage."

    stitched_logits = (
        logits_sum /
        coverage
    )

    probabilities = 1.0 / (
        1.0 +
        np.exp(
            -np.clip(
                stitched_logits,
                -50,
                50
            )
        )
    )

    prediction = (
        probabilities >= THRESHOLD
    ).astype(np.uint8)

    return (
        prediction,
        probabilities,
        coverage,
        tiles_processed
    )


# ------------------------------------------------------------
# Evaluation
# ------------------------------------------------------------

results = []

start_total = time.time()

for index, record in enumerate(
    records,
    start=1
):

    image_path = record["image_path"]
    mask_path = record["mask_path"]

    image = np.array(
        Image.open(
            image_path
        ).convert("RGB")
    )

    target = np.array(
        Image.open(
            mask_path
        ).convert("L")
    )

    target = (
        target > 127
    ).astype(np.uint8)

    height, width = image.shape[:2]

    prediction, probabilities, coverage, tiles = (
        predict_tiled(
            image,
            model,
            device,
            TILE_SIZE,
            STRIDE
        )
    )

    metrics = compute_metrics(
        prediction,
        target
    )

    gt_fraction = float(
        target.mean()
    )

    predicted_fraction = float(
        prediction.mean()
    )

    if gt_fraction > 0:

        area_ratio = (
            predicted_fraction /
            gt_fraction
        )

    else:

        area_ratio = float("nan")

    row = {
        "index": index,
        "split": record["split"],
        "image": image_path.name,
        "height": height,
        "width": width,
        "tiles": tiles,

        "dice":
            metrics["dice"],

        "iou":
            metrics["iou"],

        "precision":
            metrics["precision"],

        "recall":
            metrics["recall"],

        "pixel_accuracy":
            metrics["pixel_accuracy"],

        "gt_foreground_fraction":
            gt_fraction,

        "pred_foreground_fraction":
            predicted_fraction,

        "foreground_area_ratio":
            area_ratio,

        "tp":
            metrics["tp"],

        "fp":
            metrics["fp"],

        "fn":
            metrics["fn"],

        "tn":
            metrics["tn"]
    }

    results.append(row)

    if index % 50 == 0 or index == len(records):

        elapsed = (
            time.time() -
            start_total
        )

        rate = index / max(
            elapsed,
            1e-6
        )

        remaining = (
            len(records) - index
        ) / max(
            rate,
            1e-6
        )

        print(
            f"{index}/{len(records)} | "
            f"elapsed={elapsed/60:.1f} min | "
            f"ETA={remaining/60:.1f} min",
            flush=True
        )


# ------------------------------------------------------------
# DataFrame
# ------------------------------------------------------------

df = pd.DataFrame(
    results
)

df.to_csv(
    RESULTS_CSV,
    index=False
)


# ------------------------------------------------------------
# Primary annotated-positive evaluation
# ------------------------------------------------------------

# All 2010 WWTP records were audited previously:
# 2004 annotated-positive
# 6 annotation-empty.
#
# Primary evaluation = annotated-positive only.

primary = df[
    df["gt_foreground_fraction"] > 0
].copy()

secondary_empty = df[
    df["gt_foreground_fraction"] == 0
].copy()


summary = {

    "experiment":
        "Step17 appearance-focused generalization",

    "checkpoint_epoch":
        int(checkpoint["epoch"]),

    "validation_dice":
        float(checkpoint["best_val_dice"]),

    "protocol": {
        "native_resolution":
            [2160, 3840],

        "tile_size":
            TILE_SIZE,

        "stride":
            STRIDE,

        "overlap_pixels":
            OVERLAP,

        "stitching":
            "average overlapping logits",

        "activation":
            "sigmoid",

        "threshold":
            THRESHOLD
    },

    "total_images":
        int(len(df)),

    "primary_annotated_positive_images":
        int(len(primary)),

    "secondary_annotation_empty_images":
        int(len(secondary_empty)),

    "primary_metrics": {

        "dice_mean":
            float(primary["dice"].mean()),

        "dice_std":
            float(primary["dice"].std()),

        "dice_median":
            float(primary["dice"].median()),

        "iou_mean":
            float(primary["iou"].mean()),

        "iou_std":
            float(primary["iou"].std()),

        "iou_median":
            float(primary["iou"].median()),

        "precision_mean":
            float(primary["precision"].mean()),

        "precision_std":
            float(primary["precision"].std()),

        "recall_mean":
            float(primary["recall"].mean()),

        "recall_std":
            float(primary["recall"].std()),

        "pixel_accuracy_mean":
            float(
                primary[
                    "pixel_accuracy"
                ].mean()
            ),

        "pixel_accuracy_std":
            float(
                primary[
                    "pixel_accuracy"
                ].std()
            ),

        "gt_foreground_fraction_mean":
            float(
                primary[
                    "gt_foreground_fraction"
                ].mean()
            ),

        "gt_foreground_fraction_median":
            float(
                primary[
                    "gt_foreground_fraction"
                ].median()
            ),

        "pred_foreground_fraction_mean":
            float(
                primary[
                    "pred_foreground_fraction"
                ].mean()
            ),

        "pred_foreground_fraction_median":
            float(
                primary[
                    "pred_foreground_fraction"
                ].median()
            ),

        "foreground_area_ratio_mean":
            float(
                primary[
                    "foreground_area_ratio"
                ].mean()
            ),

        "foreground_area_ratio_median":
            float(
                primary[
                    "foreground_area_ratio"
                ].median()
            )
    },

    "secondary_empty": {

        "count":
            int(len(secondary_empty)),

        "mean_pred_foreground_fraction":
            float(
                secondary_empty[
                    "pred_foreground_fraction"
                ].mean()
            ) if len(secondary_empty)
            else None
    }
}


# ------------------------------------------------------------
# Dice bands
# ------------------------------------------------------------

bins = [
    -np.inf,
    0.01,
    0.05,
    0.10,
    0.25,
    0.50,
    np.inf
]

labels = [
    "<0.01",
    "0.01-0.05",
    "0.05-0.10",
    "0.10-0.25",
    "0.25-0.50",
    ">=0.50"
]

bands = pd.cut(
    primary["dice"],
    bins=bins,
    labels=labels,
    right=False
)

band_counts = (
    bands
    .value_counts()
    .sort_index()
)

print("\n" + "=" * 70)
print("STEP17 WWTP PRIMARY RESULTS")
print("=" * 70)

print(
    "Images:",
    len(primary)
)

print(
    "Dice:",
    summary["primary_metrics"]["dice_mean"]
)

print(
    "Dice std:",
    summary["primary_metrics"]["dice_std"]
)

print(
    "Dice median:",
    summary["primary_metrics"]["dice_median"]
)

print(
    "IoU:",
    summary["primary_metrics"]["iou_mean"]
)

print(
    "Precision:",
    summary["primary_metrics"]["precision_mean"]
)

print(
    "Recall:",
    summary["primary_metrics"]["recall_mean"]
)

print(
    "Pixel accuracy:",
    summary["primary_metrics"]["pixel_accuracy_mean"]
)

print(
    "GT foreground fraction:",
    summary["primary_metrics"]
    ["gt_foreground_fraction_mean"]
)

print(
    "Predicted foreground fraction:",
    summary["primary_metrics"]
    ["pred_foreground_fraction_mean"]
)

print(
    "Foreground area ratio:",
    summary["primary_metrics"]
    ["foreground_area_ratio_mean"]
)

print("\nDice bands:")

for label, count in band_counts.items():

    percentage = (
        100.0 *
        count /
        len(primary)
    )

    print(
        f"{label}: "
        f"{count} "
        f"({percentage:.2f}%)"
    )


# ------------------------------------------------------------
# Save summary
# ------------------------------------------------------------

summary["dice_bands"] = {
    str(label): {
        "count": int(
            band_counts[label]
        ),
        "percentage": float(
            100.0 *
            band_counts[label] /
            len(primary)
        )
    }
    for label in labels
}

with open(
    SUMMARY_JSON,
    "w"
) as f:

    json.dump(
        summary,
        f,
        indent=2
    )


print("\nSaved:")
print(RESULTS_CSV)
print(SUMMARY_JSON)

print(
    "\nTotal runtime:",
    f"{(time.time() - start_total)/60:.2f} minutes"
)

“The baseline demonstrated strong segmentation performance within the Beach domain but exhibited severe degradation on the unseen WWTP domain. Source-only robustness experiments using stronger geometric and appearance augmentation did not substantially improve cross-domain performance. The persistent high false-positive rate on WWTP images indicates that the dominant limitation is domain shift rather than insufficient source-domain segmentation capacity. Therefore, subsequent work should investigate domain-aware representation learning or unsupervised domain adaptation rather than relying solely on conventional training-parameter or augmentation tuning.”

#Experiment 2C- DeepLabv3+

Train DeepLabV3+ + ResNet50 on the same Beach training data, select the best model on Beach validation, then evaluate it on the same Beach test and unseen WWTP dataset.

Architecture: DeepLabV3+
Encoder: ResNet50
Encoder initialization: ImageNet pretrained
Input: 512×512×3 RGB
Output: 512×512×1 logits
Training data: same 790 Beach images
Validation: same 169 Beach images
Test: same 169 Beach images
WWTP: completely unseen during training/model selection
Loss: 0.5 BCE + 0.5 Dice
Optimizer: AdamW, LR 1e-4, weight decay 1e-4
Batch size: 8
Maximum epochs: 50
Patience: 10
Threshold: 0.5
WWTP tiling: 512×512, stride 384
Overlapping tiles: average logits before sigmoid
Same evaluation metrics

In [ ]:
pip install segmentation_models_pytorch

#2c model inistailation

In [ ]:
# ============================================================
# STEP18 — DEEPLABV3+ / RESNET50
# ENVIRONMENT + MODEL INITIALIZATION CHECK
# ============================================================

import os
import json
import random
from pathlib import Path

import numpy as np
import torch
import segmentation_models_pytorch as smp


# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)


# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

BUNDLE = Path(
    "/kaggle/input/datasets/shanumkhsharma/"
    "microplastic-step16a-recovery/kaggle_bundle"
)

TRAIN_IMG = BUNDLE / "data/train/images"
TRAIN_MSK = BUNDLE / "data/train/masks"

VAL_IMG = BUNDLE / "data/val/images"
VAL_MSK = BUNDLE / "data/val/masks"

TEST_IMG = BUNDLE / "data/test/images"
TEST_MSK = BUNDLE / "data/test/masks"

OUT = Path(
    "/kaggle/working/"
    "step18_deeplabv3plus_resnet50"
)

OUT.mkdir(
    parents=True,
    exist_ok=True
)

print("=" * 70)
print("STEP18 — DEEPLABV3+ / RESNET50")
print("=" * 70)


# ------------------------------------------------------------
# GPU
# ------------------------------------------------------------

print("\nCUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():

    print(
        "GPU count:",
        torch.cuda.device_count()
    )

    for i in range(
        torch.cuda.device_count()
    ):

        print(
            f"GPU {i}:",
            torch.cuda.get_device_name(i)
        )

device = torch.device(
    "cuda:0"
    if torch.cuda.is_available()
    else "cpu"
)

print(
    "Selected device:",
    device
)


# ------------------------------------------------------------
# Dataset counts
# ------------------------------------------------------------

train_images = sorted(
    TRAIN_IMG.glob("*.png")
)

val_images = sorted(
    VAL_IMG.glob("*.png")
)

test_images = sorted(
    TEST_IMG.glob("*.png")
)

print("\nDataset counts:")
print("Train:", len(train_images))
print("Validation:", len(val_images))
print("Test:", len(test_images))

assert len(train_images) == 790
assert len(val_images) == 169
assert len(test_images) == 169


# ------------------------------------------------------------
# Model
# ------------------------------------------------------------

print("\nInitializing DeepLabV3+...")

model = smp.DeepLabV3Plus(
    encoder_name="resnet50",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
    activation=None
).to(device)


# ------------------------------------------------------------
# Parameter audit
# ------------------------------------------------------------

total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("\nModel:")
print("Architecture: DeepLabV3+")
print("Encoder: ResNet50")
print("Encoder weights: ImageNet")
print("Input channels: 3")
print("Output classes: 1")
print("Activation: None")

print(
    "\nTotal parameters:",
    total_params
)

print(
    "Trainable parameters:",
    trainable_params
)


# ------------------------------------------------------------
# Verify parameters are finite
# ------------------------------------------------------------

finite = True

for name, parameter in model.named_parameters():

    if not torch.isfinite(
        parameter
    ).all():

        finite = False

        print(
            "Non-finite parameter:",
            name
        )

print(
    "\nAll model parameters finite:",
    finite
)

assert finite


# ------------------------------------------------------------
# Optimizer
# ------------------------------------------------------------

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

print("\nOptimizer:")
print("AdamW")
print("Learning rate:", 1e-4)
print("Weight decay:", 1e-4)


# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

config = {

    "experiment":
        "step18_deeplabv3plus_resnet50",

    "architecture":
        "DeepLabV3Plus",

    "encoder":
        "resnet50",

    "encoder_weights":
        "imagenet",

    "in_channels":
        3,

    "classes":
        1,

    "activation":
        None,

    "input_size":
        [512, 512],

    "train_images":
        790,

    "validation_images":
        169,

    "test_images":
        169,

    "wwtp_training":
        False,

    "wwtp_validation":
        False,

    "wwtp_model_selection":
        False,

    "wwtp_external_evaluation":
        True,

    "loss":
        "0.5 BCEWithLogits + 0.5 Dice",

    "optimizer":
        "AdamW",

    "learning_rate":
        1e-4,

    "weight_decay":
        1e-4,

    "batch_size":
        8,

    "max_epochs":
        50,

    "early_stopping_patience":
        10,

    "threshold":
        0.5,

    "seed":
        SEED,

    "wwtp_tile_size":
        512,

    "wwtp_stride":
        384,

    "wwtp_overlap":
        128,

    "wwtp_stitching":
        "average overlapping logits"
}

CONFIG_PATH = (
    OUT /
    "step18_config.json"
)

with open(
    CONFIG_PATH,
    "w"
) as f:

    json.dump(
        config,
        f,
        indent=2
    )


# ------------------------------------------------------------
# Final checks
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP18 INITIALIZATION PASSED")
print("=" * 70)

print(
    "Architecture:",
    "DeepLabV3+"
)

print(
    "Encoder:",
    "ResNet50"
)

print(
    "Parameters:",
    total_params
)

print(
    "Trainable:",
    trainable_params
)

print(
    "Beach train/val/test:",
    "790 / 169 / 169"
)

print(
    "WWTP used for training:",
    False
)

print(
    "WWTP used for model selection:",
    False
)

print(
    "Configuration saved:",
    CONFIG_PATH
)

#2c training on beach

In [ ]:
# ============================================================
# STEP18 — DEEPLABV3+ / RESNET50 TRAINING
# ============================================================

import json
import random
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp


# ============================================================
# CONFIGURATION
# ============================================================

SEED = 42

BATCH_SIZE = 8
MAX_EPOCHS = 50
PATIENCE = 10

LR = 1e-4
WEIGHT_DECAY = 1e-4

THRESHOLD = 0.5

OUT = Path(
    "/kaggle/working/"
    "step18_deeplabv3plus_resnet50"
)

BEST_PATH = (
    OUT /
    "deeplabv3plus_resnet50_best_dice.pth"
)

HISTORY_PATH = (
    OUT /
    "training_history.json"
)

CONFIG_PATH = (
    OUT /
    "step18_training_config.json"
)

device = torch.device(
    "cuda:0"
    if torch.cuda.is_available()
    else "cpu"
)


# ============================================================
# SEED
# ============================================================

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)


# ============================================================
# PATHS
# ============================================================

BUNDLE = Path(
    "/kaggle/input/datasets/shanumkhsharma/"
    "microplastic-step16a-recovery/kaggle_bundle"
)

TRAIN_IMG = BUNDLE / "data/train/images"
TRAIN_MSK = BUNDLE / "data/train/masks"

VAL_IMG = BUNDLE / "data/val/images"
VAL_MSK = BUNDLE / "data/val/masks"


# ============================================================
# DATASET
# ============================================================

class BeachSegmentationDataset(Dataset):

    def __init__(
        self,
        image_dir,
        mask_dir,
        augment=False
    ):

        self.image_dir = Path(
            image_dir
        )

        self.mask_dir = Path(
            mask_dir
        )

        self.augment = augment

        self.images = sorted([
            p for p in self.image_dir.iterdir()
            if p.suffix.lower()
            in [".png", ".jpg", ".jpeg"]
        ])

    def __len__(self):

        return len(self.images)

    def _load_pair(
        self,
        image_path
    ):

        mask_path = (
            self.mask_dir /
            f"{image_path.stem}.png"
        )

        image = np.array(
            Image.open(
                image_path
            ).convert("RGB")
        )

        mask = np.array(
            Image.open(
                mask_path
            ).convert("L")
        )

        return image, mask

    def _resize(
        self,
        image,
        mask
    ):

        image = np.array(
            Image.fromarray(
                image
            ).resize(
                (512, 512),
                Image.Resampling.BILINEAR
            )
        )

        mask = np.array(
            Image.fromarray(
                mask
            ).resize(
                (512, 512),
                Image.Resampling.NEAREST
            )
        )

        return image, mask

    def _augment(
        self,
        image,
        mask
    ):

        # ----------------------------------------------------
        # Horizontal flip
        # ----------------------------------------------------

        if random.random() < 0.5:

            image = np.ascontiguousarray(
                image[:, ::-1]
            )

            mask = np.ascontiguousarray(
                mask[:, ::-1]
            )

        # ----------------------------------------------------
        # Rotation
        # ----------------------------------------------------

        angle = random.uniform(
            -10.0,
            10.0
        )

        if abs(angle) > 1e-6:

            image_pil = Image.fromarray(
                image
            )

            mask_pil = Image.fromarray(
                mask
            )

            image = np.array(
                image_pil.rotate(
                    angle,
                    resample=Image.Resampling.BILINEAR,
                    fillcolor=(0, 0, 0)
                )
            )

            mask = np.array(
                mask_pil.rotate(
                    angle,
                    resample=Image.Resampling.NEAREST,
                    fillcolor=0
                )
            )

        # ----------------------------------------------------
        # Small scale change
        # ----------------------------------------------------

        scale = random.uniform(
            0.95,
            1.05
        )

        if abs(scale - 1.0) > 1e-6:

            h, w = image.shape[:2]

            new_h = max(
                1,
                int(round(h * scale))
            )

            new_w = max(
                1,
                int(round(w * scale))
            )

            image_resized = np.array(
                Image.fromarray(
                    image
                ).resize(
                    (new_w, new_h),
                    Image.Resampling.BILINEAR
                )
            )

            mask_resized = np.array(
                Image.fromarray(
                    mask
                ).resize(
                    (new_w, new_h),
                    Image.Resampling.NEAREST
                )
            )

            if scale >= 1.0:

                top = (
                    new_h - h
                ) // 2

                left = (
                    new_w - w
                ) // 2

                image = image_resized[
                    top:top+h,
                    left:left+w
                ]

                mask = mask_resized[
                    top:top+h,
                    left:left+w
                ]

            else:

                canvas_image = np.zeros_like(
                    image
                )

                canvas_mask = np.zeros_like(
                    mask
                )

                top = (
                    h - new_h
                ) // 2

                left = (
                    w - new_w
                ) // 2

                canvas_image[
                    top:top+new_h,
                    left:left+new_w
                ] = image_resized

                canvas_mask[
                    top:top+new_h,
                    left:left+new_w
                ] = mask_resized

                image = canvas_image
                mask = canvas_mask

        # ----------------------------------------------------
        # Brightness
        # ----------------------------------------------------

        brightness = random.uniform(
            0.9,
            1.1
        )

        image = np.clip(
            image.astype(np.float32)
            * brightness,
            0,
            255
        ).astype(np.uint8)

        # ----------------------------------------------------
        # Contrast
        # ----------------------------------------------------

        contrast = random.uniform(
            0.9,
            1.1
        )

        mean = image.mean(
            axis=(0, 1),
            keepdims=True
        )

        image = np.clip(
            (
                image.astype(np.float32)
                - mean
            ) * contrast
            + mean,
            0,
            255
        ).astype(np.uint8)

        return image, mask

    def __getitem__(
        self,
        idx
    ):

        image_path = self.images[idx]

        image, mask = self._load_pair(
            image_path
        )

        image, mask = self._resize(
            image,
            mask
        )

        if self.augment:

            image, mask = self._augment(
                image,
                mask
            )

        image = (
            torch.from_numpy(
                image.astype(
                    np.float32
                ) / 255.0
            )
            .permute(2, 0, 1)
            .contiguous()
        )

        mask = torch.from_numpy(
            (
                mask > 127
            ).astype(
                np.float32
            )
        ).unsqueeze(0)

        return image, mask


# ============================================================
# DATASETS
# ============================================================

train_ds = BeachSegmentationDataset(
    TRAIN_IMG,
    TRAIN_MSK,
    augment=True
)

val_ds = BeachSegmentationDataset(
    VAL_IMG,
    VAL_MSK,
    augment=False
)

print("=" * 70)
print("STEP18 TRAINING")
print("=" * 70)

print(
    "Train images:",
    len(train_ds)
)

print(
    "Validation images:",
    len(val_ds)
)

assert len(train_ds) == 790
assert len(val_ds) == 169


# ============================================================
# DATALOADERS
# ============================================================

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=False
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    drop_last=False
)


# ============================================================
# MODEL
# ============================================================

model = smp.DeepLabV3Plus(
    encoder_name="resnet50",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
    activation=None
).to(device)


# ============================================================
# LOSS
# ============================================================

bce_loss = nn.BCEWithLogitsLoss()

dice_loss = smp.losses.DiceLoss(
    mode="binary",
    from_logits=True
)


def combined_loss(
    logits,
    targets
):

    bce = bce_loss(
        logits,
        targets
    )

    dice = dice_loss(
        logits,
        targets
    )

    total = (
        0.5 * bce
        +
        0.5 * dice
    )

    return (
        total,
        bce,
        dice
    )


# ============================================================
# OPTIMIZER
# ============================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)


# ============================================================
# AMP
# ============================================================

scaler = torch.cuda.amp.GradScaler(
    enabled=torch.cuda.is_available()
)


# ============================================================
# METRICS
# ============================================================

def calculate_batch_metrics(
    logits,
    targets
):

    predictions = (
        torch.sigmoid(logits)
        >= THRESHOLD
    ).float()

    tp = (
        predictions * targets
    ).sum().item()

    fp = (
        predictions *
        (1 - targets)
    ).sum().item()

    fn = (
        (1 - predictions) *
        targets
    ).sum().item()

    tn = (
        (1 - predictions) *
        (1 - targets)
    ).sum().item()

    dice = (
        2 * tp /
        (
            2 * tp +
            fp +
            fn +
            1e-7
        )
    )

    iou = (
        tp /
        (
            tp +
            fp +
            fn +
            1e-7
        )
    )

    precision = (
        tp /
        (
            tp +
            fp +
            1e-7
        )
    )

    recall = (
        tp /
        (
            tp +
            fn +
            1e-7
        )
    )

    accuracy = (
        (tp + tn) /
        (
            tp +
            tn +
            fp +
            fn +
            1e-7
        )
    )

    return (
        dice,
        iou,
        precision,
        recall,
        accuracy
    )


# ============================================================
# TRAINING HISTORY
# ============================================================

history = []

best_val_dice = -float("inf")
best_epoch = None
epochs_without_improvement = 0


# ============================================================
# TRAINING LOOP
# ============================================================

training_start = time.time()

for epoch in range(
    1,
    MAX_EPOCHS + 1
):

    epoch_start = time.time()

    # --------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------

    model.train()

    train_losses = []
    train_bce = []
    train_dice_loss = []

    train_dice_values = []
    train_iou_values = []
    train_precision_values = []
    train_recall_values = []
    train_accuracy_values = []

    for images, masks in train_loader:

        images = images.to(
            device,
            non_blocking=True
        )

        masks = masks.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.cuda.amp.autocast(
            enabled=torch.cuda.is_available()
        ):

            logits = model(
                images
            )

            loss, bce, dice = (
                combined_loss(
                    logits,
                    masks
                )
            )

        scaler.scale(
            loss
        ).backward()

        scaler.step(
            optimizer
        )

        scaler.update()

        train_losses.append(
            loss.item()
        )

        train_bce.append(
            bce.item()
        )

        train_dice_loss.append(
            dice.item()
        )

        metrics = (
            calculate_batch_metrics(
                logits.detach(),
                masks
            )
        )

        train_dice_values.append(
            metrics[0]
        )

        train_iou_values.append(
            metrics[1]
        )

        train_precision_values.append(
            metrics[2]
        )

        train_recall_values.append(
            metrics[3]
        )

        train_accuracy_values.append(
            metrics[4]
        )


    # --------------------------------------------------------
    # VALIDATION
    # --------------------------------------------------------

    model.eval()

    val_losses = []
    val_bce = []
    val_dice_loss = []

    val_dice_values = []
    val_iou_values = []
    val_precision_values = []
    val_recall_values = []
    val_accuracy_values = []

    with torch.inference_mode():

        for images, masks in val_loader:

            images = images.to(
                device,
                non_blocking=True
            )

            masks = masks.to(
                device,
                non_blocking=True
            )

            with torch.cuda.amp.autocast(
                enabled=torch.cuda.is_available()
            ):

                logits = model(
                    images
                )

                loss, bce, dice = (
                    combined_loss(
                        logits,
                        masks
                    )
                )

            val_losses.append(
                loss.item()
            )

            val_bce.append(
                bce.item()
            )

            val_dice_loss.append(
                dice.item()
            )

            metrics = (
                calculate_batch_metrics(
                    logits,
                    masks
                )
            )

            val_dice_values.append(
                metrics[0]
            )

            val_iou_values.append(
                metrics[1]
            )

            val_precision_values.append(
                metrics[2]
            )

            val_recall_values.append(
                metrics[3]
            )

            val_accuracy_values.append(
                metrics[4]
            )


    # --------------------------------------------------------
    # Aggregate
    # --------------------------------------------------------

    epoch_result = {

        "epoch": epoch,

        "train_loss":
            float(np.mean(train_losses)),

        "train_bce":
            float(np.mean(train_bce)),

        "train_dice_loss":
            float(np.mean(train_dice_loss)),

        "train_dice":
            float(np.mean(train_dice_values)),

        "train_iou":
            float(np.mean(train_iou_values)),

        "train_precision":
            float(
                np.mean(
                    train_precision_values
                )
            ),

        "train_recall":
            float(
                np.mean(
                    train_recall_values
                )
            ),

        "train_pixel_accuracy":
            float(
                np.mean(
                    train_accuracy_values
                )
            ),

        "val_loss":
            float(np.mean(val_losses)),

        "val_bce":
            float(np.mean(val_bce)),

        "val_dice_loss":
            float(np.mean(val_dice_loss)),

        "val_dice":
            float(np.mean(val_dice_values)),

        "val_iou":
            float(np.mean(val_iou_values)),

        "val_precision":
            float(
                np.mean(
                    val_precision_values
                )
            ),

        "val_recall":
            float(
                np.mean(
                    val_recall_values
                )
            ),

        "val_pixel_accuracy":
            float(
                np.mean(
                    val_accuracy_values
                )
            )
    }

    history.append(
        epoch_result
    )


    # --------------------------------------------------------
    # Best checkpoint
    # --------------------------------------------------------

    current_val_dice = (
        epoch_result["val_dice"]
    )

    improved = (
        current_val_dice >
        best_val_dice
    )

    if improved:

        best_val_dice = (
            current_val_dice
        )

        best_epoch = epoch

        epochs_without_improvement = 0

        checkpoint = {

            "epoch": epoch,

            "best_val_dice":
                best_val_dice,

            "model_state_dict":
                model.state_dict(),

            "optimizer_state_dict":
                optimizer.state_dict(),

            "history":
                history,

            "experiment":
                "step18_deeplabv3plus_resnet50",

            "seed":
                SEED
        }

        # Atomic-ish save:
        # write temporary file first,
        # then replace the previous best.
        TEMP_PATH = (
            OUT /
            "deeplabv3plus_resnet50_best_tmp.pth"
        )

        torch.save(
            checkpoint,
            TEMP_PATH
        )

        TEMP_PATH.replace(
            BEST_PATH
        )

        checkpoint_size = (
            BEST_PATH.stat().st_size
            / (1024 ** 2)
        )

        checkpoint_message = (
            f"  <-- BEST "
            f"(epoch {epoch}, "
            f"{checkpoint_size:.1f} MB)"
        )

    else:

        epochs_without_improvement += 1

        checkpoint_message = (
            f"  "
            f"(no improvement: "
            f"{epochs_without_improvement}/"
            f"{PATIENCE})"
        )


    # --------------------------------------------------------
    # Save lightweight history every epoch
    # --------------------------------------------------------

    with open(
        HISTORY_PATH,
        "w"
    ) as f:

        json.dump(
            history,
            f,
            indent=2
        )


    # --------------------------------------------------------
    # Console
    # --------------------------------------------------------

    elapsed = (
        time.time() -
        epoch_start
    )

    print(
        f"Epoch {epoch:02d}/{MAX_EPOCHS} | "
        f"Train Loss "
        f"{epoch_result['train_loss']:.4f} | "
        f"Val Loss "
        f"{epoch_result['val_loss']:.4f} | "
        f"Val Dice "
        f"{epoch_result['val_dice']:.4f} | "
        f"Val IoU "
        f"{epoch_result['val_iou']:.4f} | "
        f"Val P "
        f"{epoch_result['val_precision']:.4f} | "
        f"Val R "
        f"{epoch_result['val_recall']:.4f} | "
        f"{elapsed/60:.2f} min"
        f"{checkpoint_message}",
        flush=True
    )


    # --------------------------------------------------------
    # Early stopping
    # --------------------------------------------------------

    if (
        epochs_without_improvement
        >= PATIENCE
    ):

        print(
            "\nEarly stopping triggered."
        )

        break


# ============================================================
# FINAL SAVE
# ============================================================

final_config = {

    "experiment":
        "step18_deeplabv3plus_resnet50",

    "architecture":
        "DeepLabV3Plus",

    "encoder":
        "resnet50",

    "encoder_weights":
        "imagenet",

    "loss":
        "0.5 BCEWithLogits + 0.5 Dice",

    "optimizer":
        "AdamW",

    "learning_rate":
        LR,

    "weight_decay":
        WEIGHT_DECAY,

    "batch_size":
        BATCH_SIZE,

    "max_epochs":
        MAX_EPOCHS,

    "patience":
        PATIENCE,

    "best_epoch":
        best_epoch,

    "best_validation_dice":
        best_val_dice,

    "seed":
        SEED,

    "train_images":
        len(train_ds),

    "validation_images":
        len(val_ds),

    "wwtp_used_for_training":
        False,

    "wwtp_used_for_validation":
        False,

    "wwtp_used_for_model_selection":
        False
}

with open(
    CONFIG_PATH,
    "w"
) as f:

    json.dump(
        final_config,
        f,
        indent=2
    )


# ============================================================
# SUMMARY
# ============================================================

total_time = (
    time.time() -
    training_start
)

print("\n" + "=" * 70)
print("STEP18 TRAINING COMPLETE")
print("=" * 70)

print(
    "Epochs completed:",
    len(history)
)

print(
    "Best epoch:",
    best_epoch
)

print(
    "Best validation Dice:",
    best_val_dice
)

print(
    "Best checkpoint:",
    BEST_PATH
)

print(
    "Checkpoint exists:",
    BEST_PATH.exists()
)

if BEST_PATH.exists():

    print(
        "Checkpoint size:",
        f"{BEST_PATH.stat().st_size / (1024**2):.2f} MB"
    )

print(
    "Training time:",
    f"{total_time / 60:.2f} minutes"
)

print(
    "History:",
    HISTORY_PATH
)

2c-beach test

In [ ]:
# ============================================================
# STEP18 — BEACH TEST EVALUATION
# ============================================================

import json
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp


# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

BUNDLE = Path(
    "/kaggle/input/datasets/shanumkhsharma/"
    "microplastic-step16a-recovery/kaggle_bundle"
)

TEST_IMG = BUNDLE / "data/test/images"
TEST_MSK = BUNDLE / "data/test/masks"

CKPT = Path(
    "/kaggle/working/"
    "step18_deeplabv3plus_resnet50/"
    "deeplabv3plus_resnet50_best_dice.pth"
)

OUT = Path(
    "/kaggle/working/"
    "step18_deeplabv3plus_resnet50/"
    "evaluation"
)

OUT.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

device = torch.device(
    "cuda:0"
    if torch.cuda.is_available()
    else "cpu"
)

print("=" * 70)
print("STEP18 — BEACH TEST")
print("=" * 70)

print("Device:", device)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )


# ------------------------------------------------------------
# Dataset
# ------------------------------------------------------------

class BeachTestDataset(Dataset):

    def __init__(
        self,
        image_dir,
        mask_dir
    ):

        self.image_dir = Path(
            image_dir
        )

        self.mask_dir = Path(
            mask_dir
        )

        self.images = sorted([
            p for p in self.image_dir.iterdir()
            if p.suffix.lower()
            in [".png", ".jpg", ".jpeg"]
        ])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):

        image_path = self.images[idx]

        mask_path = (
            self.mask_dir /
            f"{image_path.stem}.png"
        )

        image = np.array(
            Image.open(
                image_path
            ).convert("RGB")
        )

        mask = np.array(
            Image.open(
                mask_path
            ).convert("L")
        )

        image = np.array(
            Image.fromarray(
                image
            ).resize(
                (512, 512),
                Image.Resampling.BILINEAR
            )
        )

        mask = np.array(
            Image.fromarray(
                mask
            ).resize(
                (512, 512),
                Image.Resampling.NEAREST
            )
        )

        image = torch.from_numpy(
            image.astype(
                np.float32
            ) / 255.0
        ).permute(
            2, 0, 1
        )

        mask = torch.from_numpy(
            (
                mask > 127
            ).astype(
                np.float32
            )
        ).unsqueeze(0)

        return image, mask


test_ds = BeachTestDataset(
    TEST_IMG,
    TEST_MSK
)

test_loader = DataLoader(
    test_ds,
    batch_size=8,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(
    "Test images:",
    len(test_ds)
)

assert len(test_ds) == 169


# ------------------------------------------------------------
# Load model
# ------------------------------------------------------------

model = smp.DeepLabV3Plus(
    encoder_name="resnet50",
    encoder_weights=None,
    in_channels=3,
    classes=1,
    activation=None
).to(device)

checkpoint = torch.load(
    CKPT,
    map_location=device
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

print(
    "Checkpoint epoch:",
    checkpoint["epoch"]
)

print(
    "Best validation Dice:",
    checkpoint["best_val_dice"]
)


# ------------------------------------------------------------
# Evaluation
# ------------------------------------------------------------

dice_values = []
iou_values = []
precision_values = []
recall_values = []
accuracy_values = []


with torch.inference_mode():

    for batch_idx, (
        images,
        masks
    ) in enumerate(test_loader):

        images = images.to(
            device,
            non_blocking=True
        )

        masks = masks.to(
            device,
            non_blocking=True
        )

        logits = model(
            images
        )

        predictions = (
            torch.sigmoid(logits)
            >= 0.5
        ).float()

        for i in range(
            images.shape[0]
        ):

            p = predictions[i]
            g = masks[i]

            tp = (
                p * g
            ).sum().item()

            fp = (
                p * (1 - g)
            ).sum().item()

            fn = (
                (1 - p) * g
            ).sum().item()

            tn = (
                (1 - p) *
                (1 - g)
            ).sum().item()

            dice = (
                2 * tp /
                (
                    2 * tp +
                    fp +
                    fn +
                    1e-7
                )
            )

            iou = (
                tp /
                (
                    tp +
                    fp +
                    fn +
                    1e-7
                )
            )

            precision = (
                tp /
                (
                    tp +
                    fp +
                    1e-7
                )
            )

            recall = (
                tp /
                (
                    tp +
                    fn +
                    1e-7
                )
            )

            accuracy = (
                (tp + tn) /
                (
                    tp + tn +
                    fp + fn +
                    1e-7
                )
            )

            dice_values.append(dice)
            iou_values.append(iou)
            precision_values.append(
                precision
            )
            recall_values.append(
                recall
            )
            accuracy_values.append(
                accuracy
            )

        if (batch_idx + 1) % 10 == 0:

            print(
                f"Processed "
                f"{min((batch_idx + 1) * 8, 169)}/169",
                flush=True
            )


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

results = {

    "model":
        "Step18 DeepLabV3Plus ResNet50",

    "checkpoint_epoch":
        int(checkpoint["epoch"]),

    "validation_dice":
        float(
            checkpoint["best_val_dice"]
        ),

    "test_images":
        169,

    "dice":
        float(np.mean(dice_values)),

    "iou":
        float(np.mean(iou_values)),

    "precision":
        float(
            np.mean(precision_values)
        ),

    "recall":
        float(
            np.mean(recall_values)
        ),

    "pixel_accuracy":
        float(
            np.mean(accuracy_values)
        ),

    "dice_std":
        float(np.std(dice_values)),

    "iou_std":
        float(np.std(iou_values)),

    "precision_std":
        float(
            np.std(precision_values)
        ),

    "recall_std":
        float(np.std(recall_values)),

    "pixel_accuracy_std":
        float(np.std(accuracy_values))
}


print("\n" + "=" * 70)
print("STEP18 BEACH TEST RESULTS")
print("=" * 70)

for key, value in results.items():
    print(
        f"{key}: {value}"
    )


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

SUMMARY = (
    OUT /
    "step18_beach_test_summary.json"
)

with open(
    SUMMARY,
    "w"
) as f:

    json.dump(
        results,
        f,
        indent=2
    )

print(
    "\nSaved:",
    SUMMARY
)

2c-test on wwtp

In [ ]:
# ============================================================
# STEP18 — WWTP EXTERNAL EVALUATION
# DeepLabV3+ + ResNet50
# ============================================================

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import segmentation_models_pytorch as smp
from PIL import Image


# ============================================================
# PATHS
# ============================================================

WWTP_ROOT = Path(
    "/kaggle/input/datasets/shanumkhsharma/"
    "kaggle-microplastic-bundle/data/wwtp"
)

CKPT = Path(
    "/kaggle/working/"
    "step18_deeplabv3plus_resnet50/"
    "deeplabv3plus_resnet50_best_dice.pth"
)

OUT = Path(
    "/kaggle/working/"
    "step18_deeplabv3plus_resnet50/"
    "evaluation/wwtp_full"
)

OUT.mkdir(
    parents=True,
    exist_ok=True
)

RESULTS_CSV = (
    OUT /
    "wwtp_step18_per_image.csv"
)

SUMMARY_JSON = (
    OUT /
    "wwtp_step18_summary.json"
)


# ============================================================
# CONFIGURATION
# ============================================================

TILE_SIZE = 512
STRIDE = 384
THRESHOLD = 0.5

OVERLAP = TILE_SIZE - STRIDE


print("=" * 70)
print("STEP18 — WWTP EXTERNAL EVALUATION")
print("=" * 70)

print(
    "Architecture:",
    "DeepLabV3+ + ResNet50"
)

print(
    "Checkpoint:",
    CKPT
)

print(
    "Tile size:",
    TILE_SIZE
)

print(
    "Stride:",
    STRIDE
)

print(
    "Overlap:",
    OVERLAP
)

print(
    "Threshold:",
    THRESHOLD
)


# ============================================================
# DEVICE
# ============================================================

device = torch.device(
    "cuda:0"
    if torch.cuda.is_available()
    else "cpu"
)

print(
    "\nDevice:",
    device
)

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )


# ============================================================
# LOAD CHECKPOINT
# ============================================================

checkpoint = torch.load(
    CKPT,
    map_location=device
)

print(
    "\nCheckpoint epoch:",
    checkpoint["epoch"]
)

print(
    "Best validation Dice:",
    checkpoint["best_val_dice"]
)


# ============================================================
# MODEL
# ============================================================

model = smp.DeepLabV3Plus(
    encoder_name="resnet50",
    encoder_weights=None,
    in_channels=3,
    classes=1,
    activation=None
).to(device)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

print(
    "Model loaded successfully."
)


# ============================================================
# WWTP DIRECTORIES
# ============================================================

image_dirs = {

    "train":
        WWTP_ROOT /
        "images" /
        "train",

    "validation":
        WWTP_ROOT /
        "images" /
        "validation",

    "test":
        WWTP_ROOT /
        "images" /
        "test"
}

mask_dirs = {

    "train":
        WWTP_ROOT /
        "masks" /
        "train",

    "validation":
        WWTP_ROOT /
        "masks" /
        "validation",

    "test":
        WWTP_ROOT /
        "masks" /
        "test"
}


# ============================================================
# COLLECT ALL 2010 IMAGES
# ============================================================

records = []

for split in [
    "train",
    "validation",
    "test"
]:

    image_paths = sorted(
        image_dirs[split].glob(
            "*.jpg"
        )
    )

    print(
        f"{split}: {len(image_paths)} images"
    )

    for image_path in image_paths:

        mask_path = (
            mask_dirs[split] /
            f"{image_path.stem}.png"
        )

        if not mask_path.exists():

            raise FileNotFoundError(
                f"Missing mask:\n{mask_path}"
            )

        records.append({

            "split":
                split,

            "image_path":
                image_path,

            "mask_path":
                mask_path
        })


print(
    "\nTotal images:",
    len(records)
)

assert len(records) == 2010


# ============================================================
# TILE POSITION FUNCTION
# ============================================================

def get_positions(
    length,
    tile_size=512,
    stride=384
):

    if length <= tile_size:

        return [0]

    positions = list(
        range(
            0,
            length - tile_size + 1,
            stride
        )
    )

    final_position = (
        length - tile_size
    )

    if positions[-1] != final_position:

        positions.append(
            final_position
        )

    return positions


# ============================================================
# METRICS
# ============================================================

def compute_metrics(
    prediction,
    target
):

    prediction = (
        prediction.astype(
            np.uint8
        )
    )

    target = (
        target.astype(
            np.uint8
        )
    )

    tp = np.logical_and(
        prediction == 1,
        target == 1
    ).sum()

    fp = np.logical_and(
        prediction == 1,
        target == 0
    ).sum()

    fn = np.logical_and(
        prediction == 0,
        target == 1
    ).sum()

    tn = np.logical_and(
        prediction == 0,
        target == 0
    ).sum()

    dice = (
        2.0 * tp /
        (
            2.0 * tp +
            fp +
            fn +
            1e-7
        )
    )

    iou = (
        tp /
        (
            tp +
            fp +
            fn +
            1e-7
        )
    )

    precision = (
        tp /
        (
            tp +
            fp +
            1e-7
        )
    )

    recall = (
        tp /
        (
            tp +
            fn +
            1e-7
        )
    )

    accuracy = (
        (tp + tn) /
        (
            tp +
            tn +
            fp +
            fn +
            1e-7
        )
    )

    return {

        "dice":
            float(dice),

        "iou":
            float(iou),

        "precision":
            float(precision),

        "recall":
            float(recall),

        "pixel_accuracy":
            float(accuracy),

        "tp":
            int(tp),

        "fp":
            int(fp),

        "fn":
            int(fn),

        "tn":
            int(tn)
    }


# ============================================================
# TILED INFERENCE
# ============================================================

def predict_tiled(
    image_np,
    model,
    device,
    tile_size=512,
    stride=384
):

    height, width = (
        image_np.shape[:2]
    )

    y_positions = get_positions(
        height,
        tile_size,
        stride
    )

    x_positions = get_positions(
        width,
        tile_size,
        stride
    )

    logits_sum = np.zeros(
        (height, width),
        dtype=np.float32
    )

    coverage = np.zeros(
        (height, width),
        dtype=np.float32
    )

    tiles_processed = 0

    with torch.inference_mode():

        for y in y_positions:

            for x in x_positions:

                tile = image_np[
                    y:y + tile_size,
                    x:x + tile_size
                ]

                # ------------------------------------------------
                # Safety padding for edge tiles
                # ------------------------------------------------

                if (
                    tile.shape[0] != tile_size
                    or
                    tile.shape[1] != tile_size
                ):

                    padded = np.zeros(
                        (
                            tile_size,
                            tile_size,
                            3
                        ),
                        dtype=np.uint8
                    )

                    padded[
                        :tile.shape[0],
                        :tile.shape[1]
                    ] = tile

                    tile = padded


                # ------------------------------------------------
                # Tensor
                # ------------------------------------------------

                tile_tensor = (
                    torch.from_numpy(
                        tile.astype(
                            np.float32
                        ) / 255.0
                    )
                    .permute(
                        2,
                        0,
                        1
                    )
                    .unsqueeze(0)
                    .to(
                        device,
                        non_blocking=True
                    )
                )


                # ------------------------------------------------
                # Model
                # ------------------------------------------------

                logits = model(
                    tile_tensor
                )

                logits = (
                    logits[
                        0,
                        0
                    ]
                    .detach()
                    .float()
                    .cpu()
                    .numpy()
                )


                # ------------------------------------------------
                # Actual image region
                # ------------------------------------------------

                actual_h = min(
                    tile_size,
                    height - y
                )

                actual_w = min(
                    tile_size,
                    width - x
                )


                logits_sum[
                    y:y + actual_h,
                    x:x + actual_w
                ] += (
                    logits[
                        :actual_h,
                        :actual_w
                    ]
                )

                coverage[
                    y:y + actual_h,
                    x:x + actual_w
                ] += 1.0

                tiles_processed += 1


    # ------------------------------------------------------------
    # Coverage verification
    # ------------------------------------------------------------

    assert np.all(
        coverage > 0
    ), (
        "Incomplete WWTP tile coverage."
    )


    # ------------------------------------------------------------
    # Average logits
    # ------------------------------------------------------------

    stitched_logits = (
        logits_sum /
        coverage
    )


    # ------------------------------------------------------------
    # Sigmoid
    # ------------------------------------------------------------

    probabilities = (
        1.0 /
        (
            1.0 +
            np.exp(
                -np.clip(
                    stitched_logits,
                    -50,
                    50
                )
            )
        )
    )


    # ------------------------------------------------------------
    # Threshold
    # ------------------------------------------------------------

    prediction = (
        probabilities >=
        THRESHOLD
    ).astype(
        np.uint8
    )


    return (
        prediction,
        probabilities,
        coverage,
        tiles_processed
    )


# ============================================================
# EVALUATION
# ============================================================

results = []

start_total = time.time()


for index, record in enumerate(
    records,
    start=1
):

    image_path = (
        record["image_path"]
    )

    mask_path = (
        record["mask_path"]
    )


    # --------------------------------------------------------
    # Load native-resolution image
    # --------------------------------------------------------

    image = np.array(
        Image.open(
            image_path
        ).convert("RGB")
    )


    # --------------------------------------------------------
    # Load native-resolution GT
    # --------------------------------------------------------

    target = np.array(
        Image.open(
            mask_path
        ).convert("L")
    )

    target = (
        target > 127
    ).astype(
        np.uint8
    )


    height, width = (
        image.shape[:2]
    )


    # --------------------------------------------------------
    # Tiled inference
    # --------------------------------------------------------

    (
        prediction,
        probabilities,
        coverage,
        tiles
    ) = predict_tiled(
        image,
        model,
        device,
        TILE_SIZE,
        STRIDE
    )


    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------

    metrics = compute_metrics(
        prediction,
        target
    )


    # --------------------------------------------------------
    # Foreground statistics
    # --------------------------------------------------------

    gt_fraction = float(
        target.mean()
    )

    predicted_fraction = float(
        prediction.mean()
    )


    if gt_fraction > 0:

        area_ratio = (
            predicted_fraction /
            gt_fraction
        )

    else:

        area_ratio = float("nan")


    # --------------------------------------------------------
    # Store result
    # --------------------------------------------------------

    results.append({

        "index":
            index,

        "split":
            record["split"],

        "image":
            image_path.name,

        "height":
            height,

        "width":
            width,

        "tiles":
            tiles,

        "dice":
            metrics["dice"],

        "iou":
            metrics["iou"],

        "precision":
            metrics["precision"],

        "recall":
            metrics["recall"],

        "pixel_accuracy":
            metrics[
                "pixel_accuracy"
            ],

        "gt_foreground_fraction":
            gt_fraction,

        "pred_foreground_fraction":
            predicted_fraction,

        "foreground_area_ratio":
            area_ratio,

        "tp":
            metrics["tp"],

        "fp":
            metrics["fp"],

        "fn":
            metrics["fn"],

        "tn":
            metrics["tn"]
    })


    # --------------------------------------------------------
    # Progress
    # --------------------------------------------------------

    if (
        index % 50 == 0
        or
        index == len(records)
    ):

        elapsed = (
            time.time()
            -
            start_total
        )

        rate = (
            index /
            max(
                elapsed,
                1e-6
            )
        )

        remaining = (
            len(records) -
            index
        ) / max(
            rate,
            1e-6
        )

        print(
            f"{index}/{len(records)} | "
            f"elapsed={elapsed/60:.1f} min | "
            f"ETA={remaining/60:.1f} min",
            flush=True
        )


# ============================================================
# DATAFRAME
# ============================================================

df = pd.DataFrame(
    results
)

df.to_csv(
    RESULTS_CSV,
    index=False
)


# ============================================================
# PRIMARY EVALUATION
# ============================================================

# Audited WWTP dataset:
#
# 2010 total
# 2004 annotated-positive
# 6 annotation-empty
#
# Primary evaluation:
# annotated-positive only.

primary = df[
    df[
        "gt_foreground_fraction"
    ] > 0
].copy()

secondary_empty = df[
    df[
        "gt_foreground_fraction"
    ] == 0
].copy()


# ============================================================
# SUMMARY
# ============================================================

summary = {

    "experiment":
        "Step18 DeepLabV3Plus ResNet50",

    "checkpoint_epoch":
        int(
            checkpoint["epoch"]
        ),

    "validation_dice":
        float(
            checkpoint[
                "best_val_dice"
            ]
        ),

    "protocol": {

        "native_resolution":
            [2160, 3840],

        "tile_size":
            TILE_SIZE,

        "stride":
            STRIDE,

        "overlap_pixels":
            OVERLAP,

        "stitching":
            "average overlapping logits",

        "activation":
            "sigmoid",

        "threshold":
            THRESHOLD
    },

    "total_images":
        int(len(df)),

    "primary_annotated_positive_images":
        int(len(primary)),

    "secondary_annotation_empty_images":
        int(
            len(
                secondary_empty
            )
        ),

    "primary_metrics": {

        "dice_mean":
            float(
                primary["dice"].mean()
            ),

        "dice_std":
            float(
                primary["dice"].std()
            ),

        "dice_median":
            float(
                primary["dice"].median()
            ),

        "iou_mean":
            float(
                primary["iou"].mean()
            ),

        "iou_std":
            float(
                primary["iou"].std()
            ),

        "iou_median":
            float(
                primary["iou"].median()
            ),

        "precision_mean":
            float(
                primary[
                    "precision"
                ].mean()
            ),

        "precision_std":
            float(
                primary[
                    "precision"
                ].std()
            ),

        "recall_mean":
            float(
                primary[
                    "recall"
                ].mean()
            ),

        "recall_std":
            float(
                primary[
                    "recall"
                ].std()
            ),

        "pixel_accuracy_mean":
            float(
                primary[
                    "pixel_accuracy"
                ].mean()
            ),

        "pixel_accuracy_std":
            float(
                primary[
                    "pixel_accuracy"
                ].std()
            ),

        "gt_foreground_fraction_mean":
            float(
                primary[
                    "gt_foreground_fraction"
                ].mean()
            ),

        "gt_foreground_fraction_median":
            float(
                primary[
                    "gt_foreground_fraction"
                ].median()
            ),

        "pred_foreground_fraction_mean":
            float(
                primary[
                    "pred_foreground_fraction"
                ].mean()
            ),

        "pred_foreground_fraction_median":
            float(
                primary[
                    "pred_foreground_fraction"
                ].median()
            ),

        "foreground_area_ratio_mean":
            float(
                primary[
                    "foreground_area_ratio"
                ].mean()
            ),

        "foreground_area_ratio_median":
            float(
                primary[
                    "foreground_area_ratio"
                ].median()
            )
        }
}


# ============================================================
# DICE BANDS
# ============================================================

bins = [
    -np.inf,
    0.01,
    0.05,
    0.10,
    0.25,
    0.50,
    np.inf
]

labels = [
    "<0.01",
    "0.01-0.05",
    "0.05-0.10",
    "0.10-0.25",
    "0.25-0.50",
    ">=0.50"
]

bands = pd.cut(
    primary["dice"],
    bins=bins,
    labels=labels,
    right=False
)

band_counts = (
    bands
    .value_counts()
    .sort_index()
)


# ============================================================
# PRINT RESULTS
# ============================================================

print(
    "\n" + "=" * 70
)

print(
    "STEP18 WWTP PRIMARY RESULTS"
)

print(
    "=" * 70
)

print(
    "Images:",
    len(primary)
)

print(
    "Dice:",
    summary[
        "primary_metrics"
    ]["dice_mean"]
)

print(
    "Dice std:",
    summary[
        "primary_metrics"
    ]["dice_std"]
)

print(
    "Dice median:",
    summary[
        "primary_metrics"
    ]["dice_median"]
)

print(
    "IoU:",
    summary[
        "primary_metrics"
    ]["iou_mean"]
)

print(
    "Precision:",
    summary[
        "primary_metrics"
    ]["precision_mean"]
)

print(
    "Recall:",
    summary[
        "primary_metrics"
    ]["recall_mean"]
)

print(
    "Pixel accuracy:",
    summary[
        "primary_metrics"
    ]["pixel_accuracy_mean"]
)

print(
    "GT foreground fraction:",
    summary[
        "primary_metrics"
    ]["gt_foreground_fraction_mean"]
)

print(
    "Predicted foreground fraction:",
    summary[
        "primary_metrics"
    ]["pred_foreground_fraction_mean"]
)

print(
    "Foreground area ratio:",
    summary[
        "primary_metrics"
    ]["foreground_area_ratio_mean"]
)


print("\nDice bands:")

for label in labels:

    count = int(
        band_counts[label]
    )

    percentage = (
        100.0 *
        count /
        len(primary)
    )

    print(
        f"{label}: "
        f"{count} "
        f"({percentage:.2f}%)"
    )


# ============================================================
# SAVE DICE BANDS
# ============================================================

summary["dice_bands"] = {

    str(label): {

        "count":
            int(
                band_counts[label]
            ),

        "percentage":
            float(
                100.0 *
                band_counts[label] /
                len(primary)
            )
    }

    for label in labels
}


# ============================================================
# SAVE
# ============================================================

with open(
    SUMMARY_JSON,
    "w"
) as f:

    json.dump(
        summary,
        f,
        indent=2
    )


print(
    "\nSaved:"
)

print(
    RESULTS_CSV
)

print(
    SUMMARY_JSON
)

print(
    "\nTotal runtime:",
    f"{(time.time() - start_total)/60:.2f} minutes"
)

Step 19 — UDA with domain-adversarial feature alignment

We will keep the original U-Net + ResNet-50 architecture so that any improvement can be attributed to domain adaptation rather than another architecture change.

The implementation sequence will be:

1. Build the labeled Beach source loader.
2. Build the unlabeled WWTP-train target loader.
3. Add a Gradient Reversal Layer.
4. Add a domain discriminator to the ResNet-50 encoder features.
5. Verify source segmentation loss and source/target domain loss on a diagnostic batch.
6. Train using:
   \[
   L = L_{\text{BCE+Dice}}^{Beach}+\lambda L_{\text{domain}}
   \]
7. Select the checkpoint using **Beach validation Dice only**.
8. Evaluate the frozen model on the untouched WWTP validation + test splits.
9. Compare directly against the source-only baseline (WWTP Dice 0.0604).
10. Analyze whether false-positive foreground prediction decreases.



#expeiment 3, Unsupervised Domain Adaptation

implementation

In [ ]:
import os
import json
import random
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import cv2
import segmentation_models_pytorch as smp

# ============================================================
# 1. Reproducibility
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


# ============================================================
# 2. Paths
# ============================================================

BEACH_ROOT = Path(
    "/kaggle/input/datasets/shanumkhsharma/"
    "microplastic-step16a-recovery/kaggle_bundle/data/beach"
)

WWTP_ROOT = Path(
    "/kaggle/input/datasets/shanumkhsharma/"
    "kaggle-microplastic-bundle/data/wwtp"
)

OUTPUT_ROOT = Path(
    "/kaggle/working/step19_dann_uda"
)

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("Beach root:", BEACH_ROOT)
print("WWTP root:", WWTP_ROOT)
print("Output:", OUTPUT_ROOT)

In [ ]:
optimizer.step()

print("Optimizer update completed successfully.")

In [ ]:
import torch
import torch.nn as nn
import segmentation_models_pytorch as smp


class GradientReversalFunction(torch.autograd.Function):

    @staticmethod
    def forward(ctx, x, lambda_value):
        ctx.lambda_value = lambda_value
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return (
            -ctx.lambda_value * grad_output,
            None
        )


class GradientReversal(nn.Module):

    def __init__(self):
        super().__init__()

    def forward(self, x, lambda_value):
        return GradientReversalFunction.apply(
            x,
            lambda_value
        )


class DANNUNetResNet50(nn.Module):

    def __init__(self):
        super().__init__()

        self.segmentation_model = smp.Unet(
            encoder_name="resnet50",
            encoder_weights="imagenet",
            in_channels=3,
            classes=1,
            activation=None
        )

        self.grl = GradientReversal()

        # ResNet50 deepest encoder feature = 2048 channels
        self.domain_discriminator = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),

            nn.Linear(2048, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),

            nn.Linear(512, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),

            nn.Linear(128, 1)
        )

    def encode(self, x):
        return self.segmentation_model.encoder(x)

    def segment_from_features(self, features):

        decoder_output = self.segmentation_model.decoder(
            *features
        )

        masks = self.segmentation_model.segmentation_head(
            decoder_output
        )

        return masks

    def forward(self, x, domain_lambda=0.0):

        features = self.encode(x)

        segmentation_logits = self.segment_from_features(
            features
        )

        deepest_feature = features[-1]

        reversed_feature = self.grl(
            deepest_feature,
            domain_lambda
        )

        domain_logits = self.domain_discriminator(
            reversed_feature
        )

        return segmentation_logits, domain_logits


print("DANNUNetResNet50 definition loaded.")

In [ ]:
model = DANNUNetResNet50().to(device)

print(
    "Total parameters:",
    sum(p.numel() for p in model.parameters())
)

print(
    "Trainable parameters:",
    sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )
)

In [ ]:
from pathlib import Path

BEACH_ROOT = Path(
    "/kaggle/input/datasets/shanumkhsharma/"
    "microplastic-step16a-recovery/kaggle_bundle"
)

print("Bundle exists:", BEACH_ROOT.exists())

for p in BEACH_ROOT.rglob("*"):
    if p.is_dir() and p.name in ["train", "val", "test", "images", "masks"]:
        print(p)

In [ ]:
print("Bundle contents:")
print(list(BEACH_ROOT.iterdir()))

In [ ]:
from pathlib import Path
import cv2
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

# ============================================================
# STEP 19 — CORRECT DATA SETUP
# ============================================================

BEACH_ROOT = Path(
    "/kaggle/input/datasets/shanumkhsharma/"
    "microplastic-step16a-recovery/kaggle_bundle/data"
)

WWTP_ROOT = Path(
    "/kaggle/input/datasets/shanumkhsharma/"
    "kaggle-microplastic-bundle/data/wwtp"
)

# ------------------------------------------------------------
# Dataset classes
# ------------------------------------------------------------

class BeachDataset(Dataset):

    def __init__(self, split):

        self.image_dir = BEACH_ROOT / split / "images"
        self.mask_dir = BEACH_ROOT / split / "masks"

        self.images = sorted(
            [
                p for p in self.image_dir.iterdir()
                if p.suffix.lower() in {
                    ".jpg", ".jpeg", ".png"
                }
            ]
        )

        self.pairs = []

        for img_path in self.images:

            mask_path = (
                self.mask_dir
                / f"{img_path.stem}.png"
            )

            if mask_path.exists():
                self.pairs.append(
                    (img_path, mask_path)
                )

        print(
            f"Beach {split}: "
            f"{len(self.pairs)} image/mask pairs"
        )

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):

        img_path, mask_path = self.pairs[idx]

        image = cv2.imread(
            str(img_path),
            cv2.IMREAD_COLOR
        )

        image = cv2.cvtColor(
            image,
            cv2.COLOR_BGR2RGB
        )

        mask = cv2.imread(
            str(mask_path),
            cv2.IMREAD_GRAYSCALE
        )

        image = cv2.resize(
            image,
            (512, 512),
            interpolation=cv2.INTER_LINEAR
        )

        mask = cv2.resize(
            mask,
            (512, 512),
            interpolation=cv2.INTER_NEAREST
        )

        image = (
            image.astype(np.float32)
            / 255.0
        )

        mask = (
            mask > 127
        ).astype(np.float32)

        image = torch.from_numpy(
            image.transpose(2, 0, 1)
        ).float()

        mask = torch.from_numpy(
            mask[None]
        ).float()

        return image, mask


class WWTPUnlabeledDataset(Dataset):

    def __init__(self):

        self.image_dir = (
            WWTP_ROOT / "images" / "train"
        )

        self.images = sorted(
            [
                p for p in self.image_dir.iterdir()
                if p.suffix.lower() in {
                    ".jpg", ".jpeg", ".png"
                }
            ]
        )

        print(
            "WWTP unlabeled train:",
            len(self.images)
        )

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):

        img_path = self.images[idx]

        image = cv2.imread(
            str(img_path),
            cv2.IMREAD_COLOR
        )

        image = cv2.cvtColor(
            image,
            cv2.COLOR_BGR2RGB
        )

        image = cv2.resize(
            image,
            (512, 512),
            interpolation=cv2.INTER_LINEAR
        )

        image = (
            image.astype(np.float32)
            / 255.0
        )

        image = torch.from_numpy(
            image.transpose(2, 0, 1)
        ).float()

        return image


# ------------------------------------------------------------
# Create datasets
# ------------------------------------------------------------

source_train = BeachDataset("train")
source_val = BeachDataset("val")
source_test = BeachDataset("test")

target_train = WWTPUnlabeledDataset()


# ------------------------------------------------------------
# DataLoaders
# ------------------------------------------------------------

BATCH_SIZE = 8

source_loader = DataLoader(
    source_train,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=True
)

target_loader = DataLoader(
    target_train,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=True
)

source_val_loader = DataLoader(
    source_val,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


# ------------------------------------------------------------
# Final verification
# ------------------------------------------------------------

print()
print("=" * 60)
print("DATA SETUP COMPLETE")
print("=" * 60)

print("Beach train:", len(source_train))
print("Beach val:", len(source_val))
print("Beach test:", len(source_test))
print("WWTP unlabeled train:", len(target_train))

source_images, source_masks = next(
    iter(source_loader)
)

target_images = next(
    iter(target_loader)
)

print()
print("Source images:", source_images.shape)
print("Source masks:", source_masks.shape)
print("Target images:", target_images.shape)

print(
    "Mask values:",
    torch.unique(source_masks).tolist()
)

In [ ]:
# ============================================================
# FIXED STEP 19 DANN MODEL
# ============================================================

class GradientReversalFunction(torch.autograd.Function):

    @staticmethod
    def forward(ctx, x, lambda_value):
        ctx.lambda_value = lambda_value
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambda_value * grad_output, None


class GradientReversal(nn.Module):

    def forward(self, x, lambda_value):
        return GradientReversalFunction.apply(
            x, lambda_value
        )


class DANNUNetResNet50(nn.Module):

    def __init__(self):
        super().__init__()

        self.segmentation_model = smp.Unet(
            encoder_name="resnet50",
            encoder_weights="imagenet",
            in_channels=3,
            classes=1,
            activation=None
        )

        self.grl = GradientReversal()

        self.domain_discriminator = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),

            nn.Linear(2048, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),

            nn.Linear(512, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),

            nn.Linear(128, 1)
        )

    def forward(self, x, domain_lambda=0.0):

        # Encoder
        features = self.segmentation_model.encoder(x)

        # IMPORTANT:
        # This SMP version expects the feature list as ONE argument.
        decoder_output = self.segmentation_model.decoder(
            features
        )

        segmentation_logits = (
            self.segmentation_model.segmentation_head(
                decoder_output
            )
        )

        # Deepest ResNet50 feature
        deepest_feature = features[-1]

        # Gradient reversal
        reversed_feature = self.grl(
            deepest_feature,
            domain_lambda
        )

        # Domain prediction
        domain_logits = self.domain_discriminator(
            reversed_feature
        )

        return segmentation_logits, domain_logits


# ------------------------------------------------------------
# Initialize fresh model
# ------------------------------------------------------------

model = DANNUNetResNet50().to(device)

print(
    "Total parameters:",
    sum(p.numel() for p in model.parameters())
)

print(
    "Trainable parameters:",
    sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )
)

In [ ]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

print(
    f"GPU allocated: "
    f"{torch.cuda.memory_allocated()/1024**3:.2f} GB"
)

print(
    f"GPU reserved: "
    f"{torch.cuda.memory_reserved()/1024**3:.2f} GB"
)

First, run this cell to clear the failed computation and reduce memory:

In [ ]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

BATCH_SIZE_UDA = 2

source_loader = DataLoader(
    source_train,
    batch_size=BATCH_SIZE_UDA,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=True
)

target_loader = DataLoader(
    target_train,
    batch_size=BATCH_SIZE_UDA,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=True
)

source_val_loader = DataLoader(
    source_val,
    batch_size=BATCH_SIZE_UDA,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("UDA batch size:", BATCH_SIZE_UDA)
print("CUDA memory cleared.")

Because, the gpu is strugging with batch size 8 so we changed to batch size 2

In [ ]:
# ============================================================
# STEP 19 — MEMORY-EFFICIENT DANN
# ============================================================

class GradientReversalFunction(torch.autograd.Function):

    @staticmethod
    def forward(ctx, x, lambda_value):
        ctx.lambda_value = lambda_value
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambda_value * grad_output, None


class GradientReversal(nn.Module):

    def forward(self, x, lambda_value):
        return GradientReversalFunction.apply(
            x, lambda_value
        )


class DANNUNetResNet50(nn.Module):

    def __init__(self):
        super().__init__()

        self.segmentation_model = smp.Unet(
            encoder_name="resnet50",
            encoder_weights="imagenet",
            in_channels=3,
            classes=1,
            activation=None
        )

        self.grl = GradientReversal()

        self.domain_discriminator = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),

            nn.Linear(2048, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),

            nn.Linear(512, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),

            nn.Linear(128, 1)
        )

    def source_forward(self, x, domain_lambda):

        features = self.segmentation_model.encoder(x)

        decoder_output = self.segmentation_model.decoder(
            features
        )

        segmentation_logits = (
            self.segmentation_model.segmentation_head(
                decoder_output
            )
        )

        deepest_feature = features[-1]

        domain_feature = self.grl(
            deepest_feature,
            domain_lambda
        )

        domain_logits = self.domain_discriminator(
            domain_feature
        )

        return segmentation_logits, domain_logits

    def target_domain_forward(self, x, domain_lambda):

        # Target needs only encoder features.
        # No U-Net decoder is executed.
        features = self.segmentation_model.encoder(x)

        deepest_feature = features[-1]

        domain_feature = self.grl(
            deepest_feature,
            domain_lambda
        )

        domain_logits = self.domain_discriminator(
            domain_feature
        )

        return domain_logits

We run this ocde to run the new initilaize the fresh model

In [ ]:
gc.collect()
torch.cuda.empty_cache()

model = DANNUNetResNet50().to(device)

print(
    "Total parameters:",
    sum(p.numel() for p in model.parameters())
)

print(
    "Trainable parameters:",
    sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )
)

#3

In [ ]:
# ============================================================
# STEP 19 — MEMORY-SAFE DIAGNOSTIC
# ============================================================

model.train()

source_images, source_masks = next(
    iter(source_loader)
)

target_images = next(
    iter(target_loader)
)

source_images = source_images.to(
    device,
    non_blocking=True
)

source_masks = source_masks.to(
    device,
    non_blocking=True
)

target_images = target_images.to(
    device,
    non_blocking=True
)

print("Source:", source_images.shape)
print("Target:", target_images.shape)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

optimizer.zero_grad(set_to_none=True)

with torch.cuda.amp.autocast(
    enabled=True
):

    # Source: segmentation + domain
    source_seg, source_domain = (
        model.source_forward(
            source_images,
            domain_lambda=0.1
        )
    )

    # Target: domain only
    target_domain = (
        model.target_domain_forward(
            target_images,
            domain_lambda=0.1
        )
    )

    # Segmentation loss
    seg_bce = F.binary_cross_entropy_with_logits(
        source_seg,
        source_masks
    )

    probs = torch.sigmoid(source_seg)

    probs_flat = probs.reshape(
        probs.size(0), -1
    )

    masks_flat = source_masks.reshape(
        source_masks.size(0), -1
    )

    intersection = (
        probs_flat * masks_flat
    ).sum(dim=1)

    dice = (
        2 * intersection + 1.0
    ) / (
        probs_flat.sum(dim=1)
        + masks_flat.sum(dim=1)
        + 1.0
    )

    seg_dice = 1.0 - dice.mean()

    seg_loss = (
        0.5 * seg_bce
        + 0.5 * seg_dice
    )

    # Domain labels
    source_labels = torch.zeros_like(
        source_domain
    )

    target_labels = torch.ones_like(
        target_domain
    )

    source_domain_loss = (
        F.binary_cross_entropy_with_logits(
            source_domain,
            source_labels
        )
    )

    target_domain_loss = (
        F.binary_cross_entropy_with_logits(
            target_domain,
            target_labels
        )
    )

    domain_loss = (
        source_domain_loss
        + target_domain_loss
    ) / 2.0

    total_loss = (
        seg_loss
        + 0.1 * domain_loss
    )

print()
print("Segmentation loss:", float(seg_loss))
print("Domain loss:", float(domain_loss))
print("Total loss:", float(total_loss))

total_loss.backward()

finite_gradients = True
nonzero_gradients = 0

for parameter in model.parameters():

    if parameter.grad is not None:

        if not torch.isfinite(
            parameter.grad
        ).all():

            finite_gradients = False

        if torch.count_nonzero(
            parameter.grad
        ).item() > 0:

            nonzero_gradients += 1

print()
print(
    "Gradients finite:",
    finite_gradients
)

print(
    "Nonzero-gradient parameters:",
    nonzero_gradients
)

optimizer.step()

print("Optimizer update: PASS")

gc.collect()
torch.cuda.empty_cache()

Do not use the WWTP masks anywhere in this training cell. The target branch receives only the 1,406 WWTP training images.

Do not evaluate WWTP yet. The WWTP validation/test labels remain untouched.

The T4 memory issue is addressed by:
- batch size 2
- mixed precision
- target encoder-only forward
- explicit CUDA-memory cleanup.

The expected runtime will be substantially longer than your previous source-only experiments because each iteration processes both Beach and WWTP.

In [ ]:
# ============================================================
# STEP 19 — FULL DANN UDA TRAINING
# ============================================================

import os
import json
import math
import time
import gc

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

MAX_EPOCHS = 50
PATIENCE = 10

LR = 1e-4
WEIGHT_DECAY = 1e-4

LAMBDA_MAX = 0.10

BEST_DICE = -1.0
BEST_EPOCH = -1
NO_IMPROVEMENT = 0

history = []

checkpoint_path = (
    OUTPUT_ROOT /
    "dann_unet_resnet50_best_dice.pth"
)

history_path = (
    OUTPUT_ROOT /
    "training_history.json"
)


# ------------------------------------------------------------
# Fresh model
# ------------------------------------------------------------

gc.collect()
torch.cuda.empty_cache()

model = DANNUNetResNet50().to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=torch.cuda.is_available()
)


# ------------------------------------------------------------
# Progressive GRL coefficient
# ------------------------------------------------------------

def dann_lambda(progress):

    return (
        LAMBDA_MAX *
        (
            2.0 /
            (1.0 + math.exp(-10.0 * progress))
            - 1.0
        )
    )


# ------------------------------------------------------------
# Beach validation
# ------------------------------------------------------------

@torch.no_grad()
def evaluate_beach(model, loader):

    model.eval()

    dice_scores = []
    iou_scores = []
    precision_scores = []
    recall_scores = []
    accuracy_scores = []

    for images, masks in loader:

        images = images.to(
            device,
            non_blocking=True
        )

        masks = masks.to(
            device,
            non_blocking=True
        )

        with torch.amp.autocast(
            "cuda",
            enabled=torch.cuda.is_available()
        ):

            logits, _ = model.source_forward(
                images,
                domain_lambda=0.0
            )

        probs = torch.sigmoid(logits)

        preds = (
            probs >= 0.5
        ).float()

        for i in range(images.size(0)):

            pred = preds[i].reshape(-1)
            target = masks[i].reshape(-1)

            tp = (
                (pred == 1) &
                (target == 1)
            ).sum().item()

            fp = (
                (pred == 1) &
                (target == 0)
            ).sum().item()

            fn = (
                (pred == 0) &
                (target == 1)
            ).sum().item()

            correct = (
                pred == target
            ).sum().item()

            total = target.numel()

            dice = (
                2 * tp + 1e-7
            ) / (
                2 * tp + fp + fn + 1e-7
            )

            iou = (
                tp + 1e-7
            ) / (
                tp + fp + fn + 1e-7
            )

            precision = (
                tp + 1e-7
            ) / (
                tp + fp + 1e-7
            )

            recall = (
                tp + 1e-7
            ) / (
                tp + fn + 1e-7
            )

            accuracy = correct / total

            dice_scores.append(dice)
            iou_scores.append(iou)
            precision_scores.append(precision)
            recall_scores.append(recall)
            accuracy_scores.append(accuracy)

    return {
        "dice": float(np.mean(dice_scores)),
        "iou": float(np.mean(iou_scores)),
        "precision": float(np.mean(precision_scores)),
        "recall": float(np.mean(recall_scores)),
        "pixel_accuracy": float(
            np.mean(accuracy_scores)
        )
    }


# ------------------------------------------------------------
# Training
# ------------------------------------------------------------

training_start = time.time()

for epoch in range(1, MAX_EPOCHS + 1):

    model.train()

    epoch_start = time.time()

    source_iter = iter(source_loader)
    target_iter = iter(target_loader)

    steps = min(
        len(source_loader),
        len(target_loader)
    )

    running_seg = 0.0
    running_domain = 0.0
    running_total = 0.0

    for step in range(steps):

        source_images, source_masks = next(
            source_iter
        )

        target_images = next(
            target_iter
        )

        source_images = source_images.to(
            device,
            non_blocking=True
        )

        source_masks = source_masks.to(
            device,
            non_blocking=True
        )

        target_images = target_images.to(
            device,
            non_blocking=True
        )

        # ----------------------------------------------------
        # Progressive domain coefficient
        # ----------------------------------------------------

        global_step = (
            (epoch - 1) * steps + step
        )

        total_steps = (
            MAX_EPOCHS * steps
        )

        progress = (
            global_step /
            max(total_steps - 1, 1)
        )

        lambda_grl = dann_lambda(
            progress
        )

        # ----------------------------------------------------
        # Domain labels
        # ----------------------------------------------------

        source_labels = torch.zeros(
            source_images.size(0),
            1,
            device=device
        )

        target_labels = torch.ones(
            target_images.size(0),
            1,
            device=device
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        # ----------------------------------------------------
        # Forward
        # ----------------------------------------------------

        with torch.amp.autocast(
            "cuda",
            enabled=torch.cuda.is_available()
        ):

            # Source:
            # segmentation + domain
            source_seg, source_domain = (
                model.source_forward(
                    source_images,
                    lambda_grl
                )
            )

            # Target:
            # domain only
            target_domain = (
                model.target_domain_forward(
                    target_images,
                    lambda_grl
                )
            )

            # ------------------------------------------------
            # Segmentation loss
            # ------------------------------------------------

            seg_bce = F.binary_cross_entropy_with_logits(
                source_seg,
                source_masks
            )

            probs = torch.sigmoid(
                source_seg
            )

            probs_flat = probs.reshape(
                probs.size(0),
                -1
            )

            masks_flat = source_masks.reshape(
                source_masks.size(0),
                -1
            )

            intersection = (
                probs_flat * masks_flat
            ).sum(dim=1)

            dice = (
                2 * intersection + 1.0
            ) / (
                probs_flat.sum(dim=1)
                + masks_flat.sum(dim=1)
                + 1.0
            )

            seg_dice = (
                1.0 - dice.mean()
            )

            seg_loss = (
                0.5 * seg_bce
                + 0.5 * seg_dice
            )

            # ------------------------------------------------
            # Domain loss
            # ------------------------------------------------

            source_domain_loss = (
                F.binary_cross_entropy_with_logits(
                    source_domain,
                    source_labels
                )
            )

            target_domain_loss = (
                F.binary_cross_entropy_with_logits(
                    target_domain,
                    target_labels
                )
            )

            domain_loss = (
                source_domain_loss
                + target_domain_loss
            ) / 2.0

            total_loss = (
                seg_loss
                + LAMBDA_MAX * domain_loss
            )

        # ----------------------------------------------------
        # Backpropagation
        # ----------------------------------------------------

        scaler.scale(
            total_loss
        ).backward()

        scaler.step(
            optimizer
        )

        scaler.update()

        running_seg += (
            seg_loss.detach().item()
        )

        running_domain += (
            domain_loss.detach().item()
        )

        running_total += (
            total_loss.detach().item()
        )

        # Release batch references
        del (
            source_images,
            source_masks,
            target_images,
            source_seg,
            source_domain,
            target_domain
        )

    # --------------------------------------------------------
    # Epoch averages
    # --------------------------------------------------------

    avg_seg = (
        running_seg / steps
    )

    avg_domain = (
        running_domain / steps
    )

    avg_total = (
        running_total / steps
    )

    # --------------------------------------------------------
    # Beach validation
    # --------------------------------------------------------

    val = evaluate_beach(
        model,
        source_val_loader
    )

    epoch_time = (
        time.time() - epoch_start
    )

    record = {
        "epoch": epoch,
        "train_segmentation_loss": avg_seg,
        "train_domain_loss": avg_domain,
        "train_total_loss": avg_total,
        "val_dice": val["dice"],
        "val_iou": val["iou"],
        "val_precision": val["precision"],
        "val_recall": val["recall"],
        "val_pixel_accuracy": val[
            "pixel_accuracy"
        ],
        "lambda_grl": lambda_grl,
        "epoch_time_sec": epoch_time
    }

    history.append(record)

    print(
        f"Epoch {epoch:02d}/{MAX_EPOCHS} | "
        f"Seg {avg_seg:.4f} | "
        f"Domain {avg_domain:.4f} | "
        f"Total {avg_total:.4f} | "
        f"Val Dice {val['dice']:.4f} | "
        f"Val IoU {val['iou']:.4f} | "
        f"λ {lambda_grl:.4f} | "
        f"{epoch_time:.1f}s"
    )

    # --------------------------------------------------------
    # Save best checkpoint
    # --------------------------------------------------------

    if val["dice"] > BEST_DICE:

        BEST_DICE = val["dice"]
        BEST_EPOCH = epoch
        NO_IMPROVEMENT = 0

        checkpoint = {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_val_dice": BEST_DICE,
            "history": history,
            "experiment": "step19_dann_uda",
            "architecture": "UNet_ResNet50",
            "source_domain": "Beach",
            "target_domain": "WWTP",
            "target_training_split": (
                "WWTP_train_unlabeled"
            ),
            "lambda_max": LAMBDA_MAX,
            "batch_size": BATCH_SIZE_UDA,
            "seed": 42
        }

        temp_path = checkpoint_path.with_suffix(
            ".tmp"
        )

        torch.save(
            checkpoint,
            temp_path
        )

        os.replace(
            temp_path,
            checkpoint_path
        )

        print(
            f"  BEST checkpoint saved — "
            f"Val Dice {BEST_DICE:.4f}"
        )

    else:

        NO_IMPROVEMENT += 1

    # --------------------------------------------------------
    # Early stopping
    # --------------------------------------------------------

    if NO_IMPROVEMENT >= PATIENCE:

        print(
            f"Early stopping after epoch {epoch}."
        )

        break

    gc.collect()
    torch.cuda.empty_cache()


# ------------------------------------------------------------
# Save history
# ------------------------------------------------------------

with open(history_path, "w") as f:

    json.dump(
        history,
        f,
        indent=2
    )

elapsed_minutes = (
    time.time() - training_start
) / 60.0

print()
print("=" * 60)
print("STEP 19 TRAINING COMPLETE")
print("=" * 60)
print("Best epoch:", BEST_EPOCH)
print(
    "Best Beach validation Dice:",
    BEST_DICE
)
print(
    "Checkpoint:",
    checkpoint_path
)
print(
    "History:",
    history_path
)
print(
    "Training time:",
    f"{elapsed_minutes:.2f} minutes"
)

Step 19 training completed successfully, and the checkpoint is valid. However, the Beach validation behavior is substantially weaker than the source-only models, so we should not assume DANN helped until we evaluate the untouched WWTP data.

In [ ]:
# ============================================================
# STEP 19M — DANN UDA: UNSEEN WWTP VALIDATION + TEST
# ============================================================

import os
import gc
import json
import time
import numpy as np
import pandas as pd
import cv2
import torch
from pathlib import Path

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

WWTP_ROOT = Path(
    "/kaggle/input/datasets/shanumkhsharma/"
    "kaggle-microplastic-bundle/data/wwtp"
)

CHECKPOINT = Path(
    "/kaggle/working/step19_dann_uda/"
    "dann_unet_resnet50_best_dice.pth"
)

EVAL_ROOT = Path(
    "/kaggle/working/step19_dann_uda/evaluation"
)

EVAL_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

print("Checkpoint exists:", CHECKPOINT.exists())
print("Checkpoint:", CHECKPOINT)


# ------------------------------------------------------------
# Load best checkpoint
# ------------------------------------------------------------

checkpoint = torch.load(
    CHECKPOINT,
    map_location=device,
    weights_only=False
)

model = DANNUNetResNet50().to(device)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

print("Loaded epoch:", checkpoint["epoch"])
print(
    "Best Beach validation Dice:",
    checkpoint["best_val_dice"]
)


# ------------------------------------------------------------
# WWTP paths
# ------------------------------------------------------------

IMAGE_ROOT = WWTP_ROOT / "images"
MASK_ROOT = WWTP_ROOT / "masks"

TILE_SIZE = 512
STRIDE = 384
THRESHOLD = 0.5


# ------------------------------------------------------------
# Tile positions
# ------------------------------------------------------------

def tile_positions(length, tile_size, stride):

    if length <= tile_size:
        return [0]

    positions = list(
        range(
            0,
            length - tile_size + 1,
            stride
        )
    )

    final_position = length - tile_size

    if positions[-1] != final_position:
        positions.append(final_position)

    return positions


# ------------------------------------------------------------
# WWTP inference
# ------------------------------------------------------------

@torch.no_grad()
def predict_wwtp(image):

    h, w = image.shape[:2]

    ys = tile_positions(
        h,
        TILE_SIZE,
        STRIDE
    )

    xs = tile_positions(
        w,
        TILE_SIZE,
        STRIDE
    )

    probability_sum = np.zeros(
        (h, w),
        dtype=np.float32
    )

    count_map = np.zeros(
        (h, w),
        dtype=np.float32
    )

    for y in ys:

        for x in xs:

            tile = image[
                y:y + TILE_SIZE,
                x:x + TILE_SIZE
            ]

            # RGB -> tensor
            tile_tensor = torch.from_numpy(
                tile.astype(np.float32)
                .transpose(2, 0, 1)
                / 255.0
            ).unsqueeze(0).to(
                device,
                non_blocking=True
            )

            with torch.amp.autocast(
                "cuda",
                enabled=torch.cuda.is_available()
            ):

                # Only segmentation output is needed.
                logits, _ = model.source_forward(
                    tile_tensor,
                    domain_lambda=0.0
                )

            probs = torch.sigmoid(
                logits
            )[0, 0].float().cpu().numpy()

            probability_sum[
                y:y + TILE_SIZE,
                x:x + TILE_SIZE
            ] += probs

            count_map[
                y:y + TILE_SIZE,
                x:x + TILE_SIZE
            ] += 1.0

            del tile_tensor
            del logits
            del probs

    probability_map = (
        probability_sum
        / np.maximum(count_map, 1.0)
    )

    prediction = (
        probability_map >= THRESHOLD
    ).astype(np.uint8)

    return (
        prediction,
        probability_map
    )


# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------

def calculate_metrics(pred, target):

    pred = pred.astype(bool)
    target = target.astype(bool)

    tp = np.logical_and(
        pred,
        target
    ).sum()

    fp = np.logical_and(
        pred,
        ~target
    ).sum()

    fn = np.logical_and(
        ~pred,
        target
    ).sum()

    correct = (
        pred == target
    ).sum()

    total = target.size

    dice = (
        2 * tp + 1e-7
    ) / (
        2 * tp + fp + fn + 1e-7
    )

    iou = (
        tp + 1e-7
    ) / (
        tp + fp + fn + 1e-7
    )

    precision = (
        tp + 1e-7
    ) / (
        tp + fp + 1e-7
    )

    recall = (
        tp + 1e-7
    ) / (
        tp + fn + 1e-7
    )

    accuracy = (
        correct / total
    )

    gt_fraction = (
        target.sum() / total
    )

    pred_fraction = (
        pred.sum() / total
    )

    area_ratio = (
        pred.sum() /
        max(target.sum(), 1)
    )

    return {
        "dice": float(dice),
        "iou": float(iou),
        "precision": float(precision),
        "recall": float(recall),
        "pixel_accuracy": float(accuracy),
        "gt_foreground_fraction": float(
            gt_fraction
        ),
        "pred_foreground_fraction": float(
            pred_fraction
        ),
        "foreground_area_ratio": float(
            area_ratio
        )
    }


# ------------------------------------------------------------
# Evaluate one WWTP split
# ------------------------------------------------------------

def evaluate_split(split):

    image_dir = IMAGE_ROOT / split
    mask_dir = MASK_ROOT / split

    image_paths = sorted(
        image_dir.glob("*.jpg")
    )

    rows = []

    print()
    print("=" * 60)
    print(f"Evaluating WWTP {split}")
    print("Images:", len(image_paths))
    print("=" * 60)

    start = time.time()

    for index, image_path in enumerate(
        image_paths,
        start=1
    ):

        mask_path = (
            mask_dir
            / f"{image_path.stem}.png"
        )

        image = cv2.imread(
            str(image_path),
            cv2.IMREAD_COLOR
        )

        image = cv2.cvtColor(
            image,
            cv2.COLOR_BGR2RGB
        )

        mask = cv2.imread(
            str(mask_path),
            cv2.IMREAD_GRAYSCALE
        )

        target = (
            mask > 127
        ).astype(np.uint8)

        # --------------------------------------------
        # Annotation-empty handling
        # --------------------------------------------

        target_pixels = int(
            target.sum()
        )

        annotation_status = (
            "annotation_empty"
            if target_pixels == 0
            else "annotated_positive"
        )

        prediction, probability = (
            predict_wwtp(image)
        )

        metrics = calculate_metrics(
            prediction,
            target
        )

        row = {
            "image": image_path.name,
            "split": split,
            "annotation_status": annotation_status,
            "target_pixels": target_pixels,
            "predicted_pixels": int(
                prediction.sum()
            ),
            "max_probability": float(
                probability.max()
            ),
            "mean_probability": float(
                probability.mean()
            ),
            **metrics
        }

        rows.append(row)

        if index % 25 == 0 or index == len(image_paths):

            print(
                f"{index}/{len(image_paths)} "
                f"completed"
            )

        del image
        del mask
        del target
        del prediction
        del probability

        gc.collect()
        torch.cuda.empty_cache()

    df = pd.DataFrame(rows)

    # --------------------------------------------------------
    # Primary metrics = annotated-positive images only
    # --------------------------------------------------------

    primary = df[
        df["annotation_status"]
        == "annotated_positive"
    ].copy()

    empty = df[
        df["annotation_status"]
        == "annotation_empty"
    ].copy()

    summary = {
        "experiment": "step19_dann_uda",
        "checkpoint_epoch": int(
            checkpoint["epoch"]
        ),
        "best_beach_validation_dice": float(
            checkpoint["best_val_dice"]
        ),
        "split": split,
        "total_images": int(len(df)),
        "annotated_positive_images": int(
            len(primary)
        ),
        "annotation_empty_images": int(
            len(empty)
        ),
        "tile_size": TILE_SIZE,
        "stride": STRIDE,
        "overlap_pixels": (
            TILE_SIZE - STRIDE
        ),
        "threshold": THRESHOLD,
        "metrics": {}
    }

    for metric in [
        "dice",
        "iou",
        "precision",
        "recall",
        "pixel_accuracy",
        "gt_foreground_fraction",
        "pred_foreground_fraction",
        "foreground_area_ratio"
    ]:

        if len(primary) > 0:

            summary["metrics"][metric] = {
                "mean": float(
                    primary[metric].mean()
                ),
                "std": float(
                    primary[metric].std()
                ),
                "median": float(
                    primary[metric].median()
                )
            }

    summary["runtime_minutes"] = (
        time.time() - start
    ) / 60.0

    return df, summary


# ------------------------------------------------------------
# Evaluate validation + test
# ------------------------------------------------------------

val_df, val_summary = evaluate_split(
    "validation"
)

test_df, test_summary = evaluate_split(
    "test"
)


# ------------------------------------------------------------
# Save outputs
# ------------------------------------------------------------

val_csv = (
    EVAL_ROOT /
    "wwtp_validation_per_image.csv"
)

test_csv = (
    EVAL_ROOT /
    "wwtp_test_per_image.csv"
)

val_json = (
    EVAL_ROOT /
    "wwtp_validation_summary.json"
)

test_json = (
    EVAL_ROOT /
    "wwtp_test_summary.json"
)

val_df.to_csv(
    val_csv,
    index=False
)

test_df.to_csv(
    test_csv,
    index=False
)

with open(val_json, "w") as f:
    json.dump(
        val_summary,
        f,
        indent=2
    )

with open(test_json, "w") as f:
    json.dump(
        test_summary,
        f,
        indent=2
    )


# ------------------------------------------------------------
# Final report
# ------------------------------------------------------------

print()
print("=" * 70)
print("STEP 19 WWTP UDA EVALUATION COMPLETE")
print("=" * 70)

for name, summary in [
    ("VALIDATION", val_summary),
    ("TEST", test_summary)
]:

    m = summary["metrics"]

    print()
    print(name)
    print(
        "Images:",
        summary["total_images"]
    )
    print(
        "Annotated-positive:",
        summary["annotated_positive_images"]
    )
    print(
        "Annotation-empty:",
        summary["annotation_empty_images"]
    )

    print(
        "Dice:",
        f"{m['dice']['mean']:.4f}"
    )

    print(
        "IoU:",
        f"{m['iou']['mean']:.4f}"
    )

    print(
        "Precision:",
        f"{m['precision']['mean']:.4f}"
    )

    print(
        "Recall:",
        f"{m['recall']['mean']:.4f}"
    )

    print(
        "Pixel accuracy:",
        f"{m['pixel_accuracy']['mean']:.4f}"
    )

    print(
        "GT foreground:",
        f"{m['gt_foreground_fraction']['mean']:.4f}"
    )

    print(
        "Pred foreground:",
        f"{m['pred_foreground_fraction']['mean']:.4f}"
    )

    print(
        "Foreground area ratio:",
        f"{m['foreground_area_ratio']['mean']:.2f}x"
    )

print()
print("Saved:")
print(val_csv)
print(test_csv)
print(val_json)
print(test_json)

Domain-adversarial adaptation was evaluated using unlabeled WWTP training images. Although the domain discriminator was driven toward domain confusion, the adapted model exhibited severe foreground overprediction on the held-out WWTP domain, resulting in substantially lower Dice and precision than the source-only baseline.

#visuvalization.

In [ ]:
import os

VIZ_ROOT = "/kaggle/working/microplastic_visualizations"

os.makedirs(VIZ_ROOT, exist_ok=True)

print("Visualization directory:", VIZ_ROOT)

Experiment 1 — Baseline U-Net + ResNet50

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file == "beach_baseline_training_history.json":
            print(os.path.join(root, file))

In [ ]:
history_path = "/kaggle/input/datasets/shanumkhsharma/resultsbaseline/beach_baseline_training_history.json"

In [ ]:
import json
import os

history_path = "/kaggle/input/datasets/shanumkhsharma/resultsbaseline/beach_baseline_training_history.json"

print("Exists:", os.path.exists(history_path))
print("Size:", os.path.getsize(history_path), "bytes")

with open(history_path, "r") as f:
    history = json.load(f)

print("\nType:", type(history))

if isinstance(history, dict):
    print("\nKeys:")
    print(history.keys())

elif isinstance(history, list):
    print("\nNumber of records:", len(history))
    print("\nFirst record:")
    print(history[0])

In [ ]:
print(json.dumps(history, indent=2)[:5000])

In [ ]:
import json
import matplotlib.pyplot as plt
import os

history_path = "/kaggle/input/datasets/shanumkhsharma/resultsbaseline/beach_baseline_training_history.json"

with open(history_path, "r") as f:
    history = json.load(f)

epochs = [x["epoch"] for x in history]
train_loss = [x["train_loss"] for x in history]
val_loss = [x["val_loss"] for x in history]

plt.figure(figsize=(8, 5))

plt.plot(
    epochs,
    train_loss,
    label="Training Loss",
    linewidth=2
)

plt.plot(
    epochs,
    val_loss,
    label="Validation Loss",
    linewidth=2
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Baseline U-Net ResNet50: Training and Validation Loss")

plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()

save_path = "/kaggle/working/baseline_training_validation_loss.png"

plt.savefig(
    save_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Saved:", save_path)

In [ ]:
val_dice = [x["val_dice"] for x in history]
val_iou = [x["val_iou"] for x in history]

best_dice_idx = val_dice.index(max(val_dice))

best_epoch = epochs[best_dice_idx]
best_dice = val_dice[best_dice_idx]
best_iou_at_best_dice = val_iou[best_dice_idx]

plt.figure(figsize=(8, 5))

plt.plot(
    epochs,
    val_dice,
    label="Validation Dice",
    linewidth=2
)

plt.plot(
    epochs,
    val_iou,
    label="Validation IoU",
    linewidth=2
)

plt.scatter(
    best_epoch,
    best_dice,
    s=70,
    zorder=5,
    label=f"Best Dice = {best_dice:.3f} (Epoch {best_epoch})"
)

plt.xlabel("Epoch")
plt.ylabel("Score")
plt.title("Baseline U-Net ResNet50: Validation Performance")

plt.ylim(0, 1)

plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()

save_path = "/kaggle/working/baseline_validation_metrics.png"

plt.savefig(
    save_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print(f"Best validation Dice : {best_dice:.6f}")
print(f"Best epoch           : {best_epoch}")
print(f"Validation IoU       : {best_iou_at_best_dice:.6f}")
print("Saved:", save_path)

In [ ]:
epoch_time = [x["epoch_time_sec"] for x in history]

plt.figure(figsize=(8, 5))

plt.plot(
    epochs,
    epoch_time,
    linewidth=2
)

plt.xlabel("Epoch")
plt.ylabel("Time (seconds)")
plt.title("Baseline Training Time per Epoch")

plt.grid(alpha=0.3)

plt.tight_layout()

save_path = "/kaggle/working/baseline_epoch_training_time.png"

plt.savefig(
    save_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print(f"Mean epoch time: {sum(epoch_time)/len(epoch_time):.2f} seconds")
print(f"Total training time: {sum(epoch_time)/3600:.2f} hours")

In [ ]:
fig, ax1 = plt.subplots(figsize=(9, 5.5))

# Loss
ax1.plot(
    epochs,
    train_loss,
    label="Training Loss",
    linewidth=2
)

ax1.plot(
    epochs,
    val_loss,
    label="Validation Loss",
    linewidth=2
)

ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")

# Second axis
ax2 = ax1.twinx()

ax2.plot(
    epochs,
    val_dice,
    label="Validation Dice",
    linewidth=2
)

ax2.plot(
    epochs,
    val_iou,
    label="Validation IoU",
    linewidth=2
)

ax2.set_ylabel("Segmentation Score")
ax2.set_ylim(0, 1)

# Best epoch
ax2.axvline(
    best_epoch,
    linestyle="--",
    linewidth=1.5,
    label=f"Best epoch = {best_epoch}"
)

plt.title(
    "Baseline U-Net ResNet50 Training on Beach Dataset"
)

# Combine legends from both axes
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()

ax1.legend(
    lines1 + lines2,
    labels1 + labels2,
    loc="center right"
)

ax1.grid(alpha=0.3)

fig.tight_layout()

save_path = "/kaggle/working/baseline_training_summary.png"

plt.savefig(
    save_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Saved:", save_path)

In [ ]:
best = max(history, key=lambda x: x["val_dice"])

print("BEST BASELINE CHECKPOINT")
print("------------------------")
print("Epoch       :", best["epoch"])
print("Train Loss  :", best["train_loss"])
print("Val Loss    :", best["val_loss"])
print("Val Dice    :", best["val_dice"])
print("Val IoU     :", best["val_iou"])
print("Epoch Time  :", best["epoch_time_sec"], "sec")

Experiment 2 — Step16A: Strong Augmentation.

In [ ]:
import os

print("Searching for Step16A / strong augmentation history files...\n")

for base in ["/kaggle/input", "/kaggle/working"]:
    for root, dirs, files in os.walk(base):
        for file in files:
            name = file.lower()

            if (
                "history" in name
                or "strong_aug" in name
                or "strong_augmentation" in name
            ):
                print(os.path.join(root, file))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

experiments = [
    "Baseline\nU-Net",
    "Strong\nAugmentation",
    "Appearance\nAugmentation",
    "DeepLabV3+\nResNet50",
    "DANN\nUDA"
]

beach_dice = [
    0.8323,
    0.7895,
    0.7769,
    0.8258,
    0.6767   # DANN best validation Dice, NOT Beach test
]

wwtp_dice = [
    0.0604,
    0.0475,
    0.0631,
    0.0566,
    0.0163   # approximate combined validation/test mean
]

x = np.arange(len(experiments))
width = 0.36

fig, ax = plt.subplots(figsize=(12, 6))

bars1 = ax.bar(
    x - width/2,
    beach_dice,
    width,
    label="Beach"
)

bars2 = ax.bar(
    x + width/2,
    wwtp_dice,
    width,
    label="WWTP"
)

ax.set_ylabel("Dice Score")
ax.set_title("Segmentation Performance Across Experiments")
ax.set_xticks(x)
ax.set_xticklabels(experiments)
ax.set_ylim(0, 0.9)
ax.legend()

for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width()/2,
            height + 0.015,
            f"{height:.3f}",
            ha="center",
            va="bottom",
            fontsize=9
        )

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

experiments = [
    "Baseline",
    "Strong Aug.",
    "Appearance Aug.",
    "DeepLabV3+"
]

beach = np.array([
    0.8323,
    0.7895,
    0.7769,
    0.8258
])

wwtp = np.array([
    0.0604,
    0.0475,
    0.0631,
    0.0566
])

domain_gap = beach - wwtp

fig, ax = plt.subplots(figsize=(11, 6))

x = np.arange(len(experiments))

bars = ax.bar(x, domain_gap)

ax.set_xticks(x)
ax.set_xticklabels(experiments)
ax.set_ylabel("Dice Difference: Beach − WWTP")
ax.set_title("Cross-Domain Performance Gap")
ax.set_ylim(0, 0.85)

for bar, value in zip(bars, domain_gap):
    ax.text(
        bar.get_x() + bar.get_width()/2,
        value + 0.015,
        f"{value:.3f}",
        ha="center",
        fontsize=10
    )

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

experiments = [
    "Baseline",
    "Strong Aug.",
    "Appearance Aug.",
    "DeepLabV3+",
    "DANN-UDA"
]

precision = [
    0.0393,
    0.0383,
    0.0492,
    0.0730,
    0.0084
]

recall = [
    0.4751,
    0.2261,
    0.5181,
    0.3298,
    0.7388
]

fig, ax = plt.subplots(figsize=(9, 7))

for name, p, r in zip(experiments, precision, recall):
    ax.scatter(p, r, s=100)
    ax.annotate(
        name,
        (p, r),
        xytext=(8, 5),
        textcoords="offset points"
    )

ax.set_xlabel("Precision")
ax.set_ylabel("Recall")
ax.set_title("WWTP Precision–Recall Behavior")
ax.set_xlim(0, 0.09)
ax.set_ylim(0, 0.85)
ax.grid(alpha=0.25)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

experiments = [
    "Baseline",
    "Strong Aug.",
    "Appearance Aug.",
    "DeepLabV3+",
    "DANN-UDA"
]

gt_fraction = np.array([
    0.008906,
    0.008906,
    0.008906,
    0.008906,
    0.01065   # approximate validation/test combined GT fraction
])

pred_fraction = np.array([
    0.2443,
    0.2370,
    0.5345,
    0.3239,
    0.907
])

x = np.arange(len(experiments))
width = 0.36

fig, ax = plt.subplots(figsize=(12, 6))

ax.bar(
    x - width/2,
    gt_fraction,
    width,
    label="Ground Truth"
)

ax.bar(
    x + width/2,
    pred_fraction,
    width,
    label="Prediction"
)

ax.set_xticks(x)
ax.set_xticklabels(experiments)
ax.set_ylabel("Foreground Pixel Fraction")
ax.set_title("WWTP Foreground Area: Ground Truth vs Prediction")
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
import json
import matplotlib.pyplot as plt

history_path = "/kaggle/input/datasets/shanumkhsharma/resultsbaseline/beach_baseline_training_history.json"

with open(history_path, "r") as f:
    history = json.load(f)

epochs = [x["epoch"] for x in history]
train_loss = [x["train_loss"] for x in history]
val_loss = [x["val_loss"] for x in history]
val_dice = [x["val_dice"] for x in history]
val_iou = [x["val_iou"] for x in history]

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(epochs, train_loss, label="Training Loss")
ax.plot(epochs, val_loss, label="Validation Loss")

ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Baseline Training and Validation Loss")
ax.legend()
ax.grid(alpha=0.25)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(epochs, val_dice, label="Validation Dice")
ax.plot(epochs, val_iou, label="Validation IoU")

ax.set_xlabel("Epoch")
ax.set_ylabel("Score")
ax.set_title("Baseline Validation Performance")
ax.set_ylim(0, 1)
ax.legend()
ax.grid(alpha=0.25)

plt.tight_layout()
plt.show()

1. Experiment 3: Beach vs WWTP Dice comparison

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os

out_dir = "/kaggle/working/experiment3_visualizations"
os.makedirs(out_dir, exist_ok=True)

models = [
    "Baseline\nU-Net",
    "Strong\nAugmentation",
    "Appearance\nAugmentation",
    "DeepLabV3+"
]

beach_dice = [
    0.8323,
    0.7895,
    0.7769,
    0.8258
]

wwtp_dice = [
    0.0604,
    0.0475,
    0.0631,
    0.0566
]

x = np.arange(len(models))
width = 0.36

fig, ax = plt.subplots(figsize=(10, 6))

ax.bar(x - width/2, beach_dice, width, label="Beach test")
ax.bar(x + width/2, wwtp_dice, width, label="WWTP external")

ax.set_ylabel("Dice coefficient")
ax.set_xlabel("Experiment")
ax.set_title("Experiment 3: In-domain vs cross-domain segmentation performance")
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.set_ylim(0, 0.9)
ax.legend()

for i, value in enumerate(beach_dice):
    ax.text(i - width/2, value + 0.02, f"{value:.3f}",
            ha="center", fontsize=9)

for i, value in enumerate(wwtp_dice):
    ax.text(i + width/2, value + 0.02, f"{value:.3f}",
            ha="center", fontsize=9)

plt.tight_layout()
plt.savefig(
    f"{out_dir}/experiment3_beach_vs_wwtp_dice.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

In [ ]:
domain_gap = np.array(beach_dice) - np.array(wwtp_dice)

fig, ax = plt.subplots(figsize=(9, 6))

bars = ax.bar(models, domain_gap)

ax.set_ylabel("Dice reduction")
ax.set_xlabel("Experiment")
ax.set_title("Cross-domain performance gap")

ax.set_ylim(0, 0.85)

for bar, value in zip(bars, domain_gap):
    ax.text(
        bar.get_x() + bar.get_width()/2,
        value + 0.02,
        f"{value:.3f}",
        ha="center",
        fontsize=10
    )

plt.tight_layout()

plt.savefig(
    f"{out_dir}/experiment3_domain_gap.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
precision = [
    0.0393,
    0.0383,
    0.0492,
    0.0730
]

recall = [
    0.4751,
    0.2261,
    0.5181,
    0.3298
]

fig, ax = plt.subplots(figsize=(9, 6))

x = np.arange(len(models))
width = 0.36

ax.bar(x - width/2, precision, width, label="Precision")
ax.bar(x + width/2, recall, width, label="Recall")

ax.set_ylabel("Score")
ax.set_xlabel("Experiment")
ax.set_title("WWTP external evaluation: precision vs recall")
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.set_ylim(0, 0.6)
ax.legend()

for i, value in enumerate(precision):
    ax.text(
        i - width/2,
        value + 0.015,
        f"{value:.3f}",
        ha="center",
        fontsize=9
    )

for i, value in enumerate(recall):
    ax.text(
        i + width/2,
        value + 0.015,
        f"{value:.3f}",
        ha="center",
        fontsize=9
    )

plt.tight_layout()

plt.savefig(
    f"{out_dir}/experiment3_wwtp_precision_recall.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
predicted_foreground = [
    0.2443,
    0.2370,
    0.5345,
    0.3239
]

gt_foreground = 0.008906

fig, ax = plt.subplots(figsize=(9, 6))

x = np.arange(len(models))
width = 0.36

ax.bar(
    x - width/2,
    predicted_foreground,
    width,
    label="Predicted foreground"
)

ax.bar(
    x + width/2,
    [gt_foreground] * len(models),
    width,
    label="Ground-truth foreground"
)

ax.set_ylabel("Foreground pixel fraction")
ax.set_xlabel("Experiment")
ax.set_title("WWTP foreground-area mismatch")
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.legend()

for i, value in enumerate(predicted_foreground):
    ax.text(
        i - width/2,
        value + 0.015,
        f"{value:.3f}",
        ha="center",
        fontsize=9
    )

plt.tight_layout()

plt.savefig(
    f"{out_dir}/experiment3_foreground_mismatch.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
uda_models = [
    "Source-only\nBaseline",
    "DANN UDA"
]

uda_wwtp_dice = [
    0.0604,
    0.0138
]

fig, ax = plt.subplots(figsize=(7, 5))

bars = ax.bar(uda_models, uda_wwtp_dice)

ax.set_ylabel("Dice coefficient")
ax.set_xlabel("Training strategy")
ax.set_title("WWTP external performance: source-only vs DANN UDA")
ax.set_ylim(0, 0.07)

for bar, value in zip(bars, uda_wwtp_dice):
    ax.text(
        bar.get_x() + bar.get_width()/2,
        value + 0.002,
        f"{value:.4f}",
        ha="center",
        fontsize=10
    )

plt.tight_layout()

plt.savefig(
    f"{out_dir}/experiment3_dann_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import pandas as pd

results = pd.DataFrame({
    "Experiment": [
        "Baseline U-Net",
        "Strong Augmentation",
        "Appearance Augmentation",
        "DeepLabV3+"
    ],
    "Beach_Dice": [
        0.8323,
        0.7895,
        0.7769,
        0.8258
    ],
    "WWTP_Dice": [
        0.0604,
        0.0475,
        0.0631,
        0.0566
    ],
    "WWTP_Precision": [
        0.0393,
        0.0383,
        0.0492,
        0.0730
    ],
    "WWTP_Recall": [
        0.4751,
        0.2261,
        0.5181,
        0.3298
    ],
    "WWTP_Predicted_Foreground": [
        0.2443,
        0.2370,
        0.5345,
        0.3239
    ],
    "WWTP_GT_Foreground": [
        0.008906,
        0.008906,
        0.008906,
        0.008906
    ]
})

results["Dice_Gap"] = (
    results["Beach_Dice"] -
    results["WWTP_Dice"]
)

csv_path = f"{out_dir}/experiment3_results.csv"

results.to_csv(csv_path, index=False)

display(results)

print(f"\nSaved to: {csv_path}")

1. Beach validation Dice during DANN training

In [ ]:
import json
import matplotlib.pyplot as plt
import os

step19_dir = "/kaggle/working/step19_dann_uda"

with open(
    os.path.join(step19_dir, "training_history.json"),
    "r"
) as f:
    history = json.load(f)

# Your history is a list of dictionaries
print("Number of epochs:", len(history))
print("First epoch record:")
print(history[0])

# Extract values from each epoch
epochs = [row["epoch"] for row in history]
val_dice = [row["val_dice"] for row in history]
val_iou = [row["val_iou"] for row in history]

# Find best validation Dice
best_idx = val_dice.index(max(val_dice))
best_epoch = epochs[best_idx]
best_dice = val_dice[best_idx]

# Plot
fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(
    epochs,
    val_dice,
    marker="o",
    label="Beach validation Dice"
)

ax.plot(
    epochs,
    val_iou,
    marker="s",
    label="Beach validation IoU"
)

ax.scatter(
    best_epoch,
    best_dice,
    s=100,
    label=f"Best checkpoint: epoch {best_epoch}"
)

ax.set_xlabel("Epoch")
ax.set_ylabel("Score")
ax.set_title("DANN UDA: Source-Domain Validation Performance")
ax.set_ylim(0, 1)
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()

plt.savefig(
    os.path.join(
        step19_dir,
        "experiment4_dann_validation_curve.png"
    ),
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print(f"Best epoch: {best_epoch}")
print(f"Best validation Dice: {best_dice:.4f}")
print(f"Best validation IoU: {val_iou[best_idx]:.4f}")

In [ ]:
import json

history_path = "/kaggle/working/step19_dann_uda/training_history.json"

with open(history_path, "r") as f:
    history = json.load(f)

print("History type:", type(history))
print("Number of records:", len(history))

print("\nFirst record:")
print(history[0])

print("\nExact keys:")
print(list(history[0].keys()))

In [ ]:
for i, row in enumerate(history[:3]):
    print(f"\n--- Epoch record {i+1} ---")
    for key, value in row.items():
        print(f"{key}: {value}")

In [ ]:
import json
import matplotlib.pyplot as plt
import os

step19_dir = "/kaggle/working/step19_dann_uda"

with open(
    os.path.join(step19_dir, "training_history.json"),
    "r"
) as f:
    history = json.load(f)

epochs = [row["epoch"] for row in history]
val_dice = [row["val_dice"] for row in history]
val_iou = [row["val_iou"] for row in history]

best_idx = val_dice.index(max(val_dice))
best_epoch = epochs[best_idx]
best_dice = val_dice[best_idx]

fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(
    epochs,
    val_dice,
    marker="o",
    label="Beach validation Dice"
)

ax.plot(
    epochs,
    val_iou,
    marker="s",
    label="Beach validation IoU"
)

ax.scatter(
    best_epoch,
    best_dice,
    s=100,
    label=f"Best checkpoint: Epoch {best_epoch}"
)

ax.set_xlabel("Epoch")
ax.set_ylabel("Score")
ax.set_title("DANN UDA: Beach Validation Performance")
ax.set_ylim(0, 1)
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()

plt.savefig(
    os.path.join(
        step19_dir,
        "experiment4_dann_validation_performance.png"
    ),
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print(f"Best epoch: {best_epoch}")
print(f"Best validation Dice: {best_dice:.4f}")
print(f"Best validation IoU: {val_iou[best_idx]:.4f}")

In [ ]:
epochs = [row["epoch"] for row in history]

seg_loss = [
    row["train_segmentation_loss"]
    for row in history
]

domain_loss = [
    row["train_domain_loss"]
    for row in history]

total_loss = [
    row["train_total_loss"]
    for row in history
]

fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(
    epochs,
    seg_loss,
    marker="o",
    label="Segmentation loss"
)

ax.plot(
    epochs,
    domain_loss,
    marker="s",
    label="Domain loss"
)

ax.plot(
    epochs,
    total_loss,
    marker="^",
    label="Total loss"
)

ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("DANN Training Losses")
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()

plt.savefig(
    os.path.join(
        step19_dir,
        "experiment4_dann_training_losses.png"
    ),
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
epochs = [row["epoch"] for row in history]

lambda_grl = [
    row["lambda_grl"]
    for row in history
]

fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(
    epochs,
    lambda_grl,
    marker="o"
)

ax.set_xlabel("Epoch")
ax.set_ylabel("GRL λ")
ax.set_title("DANN: Gradient Reversal Layer Weight")
ax.grid(alpha=0.3)

plt.tight_layout()

plt.savefig(
    os.path.join(
        step19_dir,
        "experiment4_dann_grl_lambda.png"
    ),
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
epochs = [row["epoch"] for row in history]

val_dice = [row["val_dice"] for row in history]
val_precision = [row["val_precision"] for row in history]
val_recall = [row["val_recall"] for row in history]

fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(
    epochs,
    val_dice,
    marker="o",
    label="Dice"
)

ax.plot(
    epochs,
    val_precision,
    marker="s",
    label="Precision"
)

ax.plot(
    epochs,
    val_recall,
    marker="^",
    label="Recall"
)

ax.set_xlabel("Epoch")
ax.set_ylabel("Score")
ax.set_title("DANN UDA: Beach Validation Metrics")
ax.set_ylim(0, 1)
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()

plt.savefig(
    os.path.join(
        step19_dir,
        "experiment4_dann_validation_metrics.png"
    ),
    dpi=300,
    bbox_inches="tight"
)

plt.show()

DANN successfully promoted domain-invariant feature learning, as indicated by the domain-classification loss approaching the random-guessing regime. However, this alignment did not translate into consistent improvement in target-domain segmentation. The high precision but comparatively lower recall indicates conservative foreground prediction and substantial false-negative errors, suggesting that domain alignment alone is insufficient to bridge the semantic and visual gap between the source and target datasets.

1. Baseline — U-Net + ResNet50, Beach → unseen WWTP
2. Domain Generalization — augmentation + architecture robustness
3. DANN UDA — domain-adversarial adaptation
4. Confidence-Controlled Pseudo-Label UDA — next
After Experiment 4, I would recommend 2 more substantive experiments, not a large number of additional variants:
5. Teacher–Student / EMA Self-Training UDA
   - Use a teacher model to generate more stable WWTP pseudo-labels.
   - Student learns from Beach ground truth + confident WWTP pseudo-labels.
   - This is the stronger version of confidence-controlled pseudo-labeling.
6. Final Hybrid / Best-Strategy Experiment
   - Combine the best-performing ingredients from Experiments 2–5.
   - For example: strongest source-domain augmentation + confidence-controlled target adaptation + EMA teacher.
   - Evaluate once on the untouched WWTP test set.
   - This becomes your final proposed model.

#Experiment 4 — Confidence-Controlled Pseudo-Label UDA

In [ ]:
import os

BASE = "/kaggle/input/datasets/shanumkhsharma"

print("Available datasets:\n")

for item in sorted(os.listdir(BASE)):
    path = os.path.join(BASE, item)
    if os.path.isdir(path):
        print("📁", item)

print("\nSearching for .pth checkpoints...\n")

found = []

for root, dirs, files in os.walk(BASE):
    for f in files:
        if f.endswith(".pth"):
            path = os.path.join(root, f)
            size_mb = os.path.getsize(path) / (1024 ** 2)
            found.append((path, size_mb))

if found:
    for path, size_mb in found:
        print(f"{size_mb:8.2f} MB  {path}")
else:
    print("No .pth checkpoints found.")

In [ ]:
pip install segmentation_models_pytorch

#exp4

In [ ]:
import torch
import segmentation_models_pytorch as smp

SOURCE_CHECKPOINT = (
    "/kaggle/input/datasets/shanumkhsharma/"
    "kaggle-microplastic-bundle/checkpoints/"
    "beach_unet_resnet50_strong_aug_best_dice.pth"
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = smp.Unet(
    encoder_name="resnet50",
    encoder_weights=None,
    in_channels=3,
    classes=1,
    activation=None
).to(device)

checkpoint = torch.load(
    SOURCE_CHECKPOINT,
    map_location=device
)

print("Checkpoint type:", type(checkpoint))

if isinstance(checkpoint, dict):
    print("Checkpoint keys:")
    print(checkpoint.keys())

#loading block

In [ ]:
if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    state_dict = checkpoint["model_state_dict"]
elif isinstance(checkpoint, dict) and "state_dict" in checkpoint:
    state_dict = checkpoint["state_dict"]
else:
    state_dict = checkpoint

model.load_state_dict(state_dict, strict=True)

print("Checkpoint loaded successfully.")
print("Model:", "U-Net + ResNet50")
print("Parameters:", sum(p.numel() for p in model.parameters()))

Build an unlabeled WWTP tile dataset

In [ ]:
import os
import glob
import random
import numpy as np
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader

WWTP_TRAIN_IMAGES = (
    "/kaggle/input/datasets/shanumkhsharma/"
    "kaggle-microplastic-bundle/data/wwtp/images/train"
)

TILE_SIZE = 512
STRIDE = 384


class WWTPUnlabeledTileDataset(Dataset):

    def __init__(
        self,
        image_dir,
        tile_size=512,
        stride=384
    ):
        self.image_dir = image_dir
        self.tile_size = tile_size
        self.stride = stride

        self.image_paths = sorted(
            glob.glob(os.path.join(image_dir, "*.jpg"))
        )

        self.tiles = []

        for image_path in self.image_paths:

            with Image.open(image_path) as img:
                width, height = img.size

            x_positions = list(
                range(
                    0,
                    max(1, width - tile_size + 1),
                    stride
                )
            )

            y_positions = list(
                range(
                    0,
                    max(1, height - tile_size + 1),
                    stride
                )
            )

            # Force final tile to image boundary
            if x_positions[-1] != width - tile_size:
                x_positions.append(width - tile_size)

            if y_positions[-1] != height - tile_size:
                y_positions.append(height - tile_size)

            for y in y_positions:
                for x in x_positions:
                    self.tiles.append(
                        {
                            "image_path": image_path,
                            "x": x,
                            "y": y
                        }
                    )

    def __len__(self):
        return len(self.tiles)

    def __getitem__(self, idx):

        item = self.tiles[idx]

        image = Image.open(
            item["image_path"]
        ).convert("RGB")

        tile = image.crop(
            (
                item["x"],
                item["y"],
                item["x"] + self.tile_size,
                item["y"] + self.tile_size
            )
        )

        tile = np.asarray(
            tile,
            dtype=np.float32
        ) / 255.0

        tile = torch.from_numpy(
            tile
        ).permute(2, 0, 1)

        return tile, item["image_path"], item["x"], item["y"]


target_dataset = WWTPUnlabeledTileDataset(
    WWTP_TRAIN_IMAGES,
    tile_size=TILE_SIZE,
    stride=STRIDE
)

print("WWTP train images:", len(target_dataset.image_paths))
print("Total 512×512 tiles:", len(target_dataset))

target_loader = DataLoader(
    target_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("DataLoader created.")

Run the pseudo-label diagnostic

In [ ]:
import torch
import numpy as np

model.eval()

FG_THRESHOLD = 0.99
BG_THRESHOLD = 0.05

total_pixels = 0
confident_pixels = 0
foreground_pixels = 0
background_pixels = 0

max_batches = 100   # diagnostic only

with torch.no_grad():

    for batch_idx, batch in enumerate(target_loader):

        if batch_idx >= max_batches:
            break

        images = batch[0].to(
            device,
            non_blocking=True
        )

        # Updated AMP syntax
        with torch.amp.autocast(
            device_type="cuda",
            enabled=torch.cuda.is_available()
        ):
            logits = model(images)

        probabilities = torch.sigmoid(logits)

        fg_mask = probabilities >= FG_THRESHOLD
        bg_mask = probabilities <= BG_THRESHOLD
        confidence_mask = fg_mask | bg_mask

        total_pixels += probabilities.numel()
        confident_pixels += confidence_mask.sum().item()
        foreground_pixels += fg_mask.sum().item()
        background_pixels += bg_mask.sum().item()

        del images
        del logits
        del probabilities
        del fg_mask
        del bg_mask
        del confidence_mask

# Calculate statistics
confident_fraction = (
    confident_pixels / total_pixels
)

foreground_fraction = (
    foreground_pixels / total_pixels
)

background_fraction = (
    background_pixels / total_pixels
)

print("Pseudo-label diagnostic")
print("=" * 50)

print(f"Diagnostic batches:       {max_batches}")
print(f"Pixels examined:          {total_pixels:,}")
print(f"Confident pixels:         {confident_pixels:,}")
print(f"Confident coverage:       {confident_fraction:.4%}")
print(f"Confident foreground:     {foreground_fraction:.4%}")
print(f"Confident background:     {background_fraction:.4%}")

print("\nThresholds")
print("-" * 50)
print(f"Foreground: p >= {FG_THRESHOLD}")
print(f"Background: p <= {BG_THRESHOLD}")
print(f"Uncertain:  {BG_THRESHOLD} < p < {FG_THRESHOLD}")

Also check prediction statistics

In [ ]:
print("\nInterpretation")
print("=" * 50)

if foreground_fraction > 0.30:
    print("WARNING: Very large confident foreground region.")
elif foreground_fraction > 0.10:
    print("CAUTION: Confident foreground is relatively large.")
else:
    print("Confident foreground fraction is relatively controlled.")

if confident_fraction < 0.05:
    print("WARNING: Very few pixels are confident.")
elif confident_fraction < 0.20:
    print("CAUTION: Pseudo-label coverage is low.")
else:
    print("Pseudo-label coverage appears usable.")

In [ ]:
import torch
import numpy as np

model.eval()

sample_probs = []

max_batches = 100

with torch.no_grad():

    for batch_idx, batch in enumerate(target_loader):

        if batch_idx >= max_batches:
            break

        images = batch[0].to(
            device,
            non_blocking=True
        )

        with torch.amp.autocast(
            device_type="cuda",
            enabled=torch.cuda.is_available()
        ):
            logits = model(images)

        probs = torch.sigmoid(logits)

        # Randomly sample probabilities to keep memory small
        flat = probs.detach().float().flatten()

        if flat.numel() > 10000:
            indices = torch.randperm(
                flat.numel(),
                device=flat.device
            )[:10000]

            flat = flat[indices]

        sample_probs.append(
            flat.cpu().numpy()
        )

        del images, logits, probs

sample_probs = np.concatenate(sample_probs)

print("Probability distribution")
print("=" * 50)

print(f"Samples: {len(sample_probs):,}")

for q in [0, 1, 5, 10, 25, 50, 75, 90, 95, 99, 100]:
    print(
        f"Percentile {q:>3}%: "
        f"{np.percentile(sample_probs, q):.6f}"
    )

In [ ]:
thresholds_bg = [
    0.50,
    0.30,
    0.20,
    0.10,
    0.05,
    0.01
]

print("\nBackground coverage by threshold")
print("=" * 50)

for threshold in thresholds_bg:

    coverage = (
        sample_probs <= threshold
    ).mean()

    print(
        f"p <= {threshold:.2f}: "
        f"{coverage:.4%}"
    )

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

model.eval()

# Select 10 tiles from the WWTP training dataset
indices = np.linspace(
    0,
    len(target_dataset) - 1,
    10,
    dtype=int
)

fig, axes = plt.subplots(
    10,
    4,
    figsize=(14, 30)
)

with torch.no_grad():

    for row, idx in enumerate(indices):

        image, image_path, x, y = target_dataset[idx]

        input_tensor = image.unsqueeze(0).to(device)

        with torch.amp.autocast(
            device_type="cuda",
            enabled=torch.cuda.is_available()
        ):
            logits = model(input_tensor)

        probability = torch.sigmoid(
            logits
        )[0, 0].float().cpu().numpy()

        image_np = image.permute(
            1, 2, 0
        ).numpy()

        fg_mask = probability >= 0.99
        bg_mask = probability <= 0.30

        axes[row, 0].imshow(image_np)
        axes[row, 0].set_title(
            f"Input\n{os.path.basename(image_path)}"
        )

        axes[row, 1].imshow(
            probability,
            vmin=0,
            vmax=1
        )
        axes[row, 1].set_title(
            "Prediction probability"
        )

        axes[row, 2].imshow(
            fg_mask,
            vmin=0,
            vmax=1
        )
        axes[row, 2].set_title(
            "Foreground p ≥ 0.99"
        )

        axes[row, 3].imshow(
            bg_mask,
            vmin=0,
            vmax=1
        )
        axes[row, 3].set_title(
            "Background p ≤ 0.30"
        )

        for col in range(4):
            axes[row, col].axis("off")

        del input_tensor, logits

plt.tight_layout()

output_path = (
    "/kaggle/working/"
    "experiment4_pseudolabel_diagnostic.png"
)

plt.savefig(
    output_path,
    dpi=200,
    bbox_inches="tight"
)

plt.show()

print("Saved:", output_path)

The next step is to calibrate the pseudo-label confidence thresholds using the Beach validation set only. We will not use any WWTP masks.
The rule will be:
- Foreground threshold = the lowest threshold that achieves ≥95% foreground precision on Beach validation, because lower thresholds give more pseudo-label coverage.
- Background threshold = the highest threshold that achieves ≥95% background precision on Beach validation.
- Then freeze those thresholds and apply them to unlabeled WWTP training data.
This is a much cleaner Experiment 4 protocol.

Beach validation confidence calibration

In [ ]:
#Step 1 — Beach validation confidence calibration
import os
import glob
import numpy as np
import torch

from PIL import Image
from torch.utils.data import Dataset, DataLoader


# ============================================================
# PATHS
# ============================================================

BEACH_VAL_IMAGES = (
    "/kaggle/input/datasets/shanumkhsharma/"
    "microplastic-step16a-recovery/kaggle_bundle/data/val/images"
)

BEACH_VAL_MASKS = (
    "/kaggle/input/datasets/shanumkhsharma/"
    "microplastic-step16a-recovery/kaggle_bundle/data/val/masks"
)


# ============================================================
# DATASET
# ============================================================

class BeachValidationDataset(Dataset):

    def __init__(self, image_dir, mask_dir):
        self.image_paths = sorted(
            glob.glob(os.path.join(image_dir, "*"))
        )

        self.mask_dir = mask_dir

        # Keep only files that have corresponding masks
        valid = []

        for image_path in self.image_paths:

            stem = os.path.splitext(
                os.path.basename(image_path)
            )[0]

            mask_path = os.path.join(
                mask_dir,
                stem + ".png"
            )

            if os.path.exists(mask_path):
                valid.append(
                    (image_path, mask_path)
                )

        self.samples = valid

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):

        image_path, mask_path = self.samples[idx]

        image = Image.open(
            image_path
        ).convert("RGB")

        mask = Image.open(
            mask_path
        ).convert("L")

        # Beach images are originally 256x256.
        # Model input is 512x512.
        image = image.resize(
            (512, 512),
            Image.Resampling.BILINEAR
        )

        mask = mask.resize(
            (512, 512),
            Image.Resampling.NEAREST
        )

        image = np.asarray(
            image,
            dtype=np.float32
        ) / 255.0

        mask = np.asarray(
            mask,
            dtype=np.uint8
        )

        mask = (mask > 127).astype(
            np.float32
        )

        image = torch.from_numpy(
            image
        ).permute(2, 0, 1)

        mask = torch.from_numpy(
            mask
        ).unsqueeze(0)

        return image, mask


beach_val_dataset = BeachValidationDataset(
    BEACH_VAL_IMAGES,
    BEACH_VAL_MASKS
)

beach_val_loader = DataLoader(
    beach_val_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Beach validation images:", len(beach_val_dataset))

In [ ]:
# ============================================================
# CLEAN AGGREGATION
# ============================================================

total_validation_pixels = (
    len(beach_val_dataset) * 512 * 512
)

fg_table = []

for threshold in thresholds:

    rows = [
        r for r in fg_results
        if r[0] == threshold
    ]

    total_tp = sum(r[1] for r in rows)
    total_pred = sum(r[2] for r in rows)

    precision = (
        total_tp / total_pred
        if total_pred > 0
        else 0.0
    )

    coverage = (
        total_pred /
        total_validation_pixels
    )

    fg_table.append(
        (threshold, precision, coverage)
    )


bg_table = []

for threshold in thresholds:

    rows = [
        r for r in bg_results
        if r[0] == threshold
    ]

    total_tn = sum(r[1] for r in rows)
    total_pred = sum(r[2] for r in rows)

    precision = (
        total_tn / total_pred
        if total_pred > 0
        else 0.0
    )

    coverage = (
        total_pred /
        total_validation_pixels
    )

    bg_table.append(
        (threshold, precision, coverage)
    )


print("\nFOREGROUND CONFIDENCE")
print("=" * 65)

for threshold, precision, coverage in fg_table:
    print(
        f"Threshold {threshold:.3f} | "
        f"Precision {precision:.4%} | "
        f"Coverage {coverage:.4%}"
    )


print("\nBACKGROUND CONFIDENCE")
print("=" * 65)

for threshold, precision, coverage in bg_table:
    print(
        f"Threshold {threshold:.3f} | "
        f"Precision {precision:.4%} | "
        f"Coverage {coverage:.4%}"
    )

In [ ]:
# ============================================================
# SELECT THRESHOLDS
# ============================================================

MIN_PRECISION = 0.95

valid_fg = [
    row for row in fg_table
    if row[1] >= MIN_PRECISION
]

valid_bg = [
    row for row in bg_table
    if row[1] >= MIN_PRECISION
]

if len(valid_fg) == 0:
    print(
        "No foreground threshold reached "
        "95% precision."
    )
else:

    # Lowest threshold satisfying precision requirement
    selected_fg = min(
        valid_fg,
        key=lambda x: x[0]
    )

    print(
        "\nSelected foreground threshold:"
    )

    print(
        f"Threshold = {selected_fg[0]:.3f}"
    )

    print(
        f"Precision = {selected_fg[1]:.4%}"
    )

    print(
        f"Coverage   = {selected_fg[2]:.4%}"
    )


if len(valid_bg) == 0:
    print(
        "No background threshold reached "
        "95% precision."
    )
else:

    # Highest threshold satisfying precision requirement
    selected_bg = max(
        valid_bg,
        key=lambda x: x[0]
    )

    print(
        "\nSelected background threshold:"
    )

    print(
        f"Threshold = {selected_bg[0]:.3f}"
    )

    print(
        f"Precision = {selected_bg[1]:.4%}"
    )

    print(
        f"Coverage   = {selected_bg[2]:.4%}"
    )

In [ ]:
import numpy as np
import torch

model.eval()

FG_THRESHOLD = 0.95

beach_fg_fractions = []

with torch.no_grad():

    for images, masks in beach_val_loader:

        images = images.to(
            device,
            non_blocking=True
        )

        with torch.amp.autocast(
            device_type="cuda",
            enabled=torch.cuda.is_available()
        ):
            logits = model(images)

        probabilities = torch.sigmoid(logits)

        fg_mask = probabilities >= FG_THRESHOLD

        # Calculate foreground fraction PER IMAGE
        fractions = (
            fg_mask.flatten(1)
            .float()
            .mean(dim=1)
            .cpu()
            .numpy()
        )

        beach_fg_fractions.extend(
            fractions.tolist()
        )

        del images
        del logits
        del probabilities

beach_fg_fractions = np.array(
    beach_fg_fractions
)

print("Beach validation foreground coverage")
print("=" * 55)

for percentile in [50, 75, 90, 95, 99, 100]:

    value = np.percentile(
        beach_fg_fractions,
        percentile
    )

    print(
        f"{percentile:>3}th percentile: "
        f"{value:.4%}"
    )

print(
    "\nMean:",
    f"{beach_fg_fractions.mean():.4%}"
)

print(
    "Median:",
    f"{np.median(beach_fg_fractions):.4%}"
)

This value is derived only from the labeled Beach validation set. We will not use WWTP masks to tune it.
One correction to the earlier wording: this makes Experiment 4 a confidence-controlled self-training UDA experiment, not an EMA teacher experiment. The teacher/student EMA mechanism remains Experiment 5.
Experiment 4 — Training
The training objective is:
\[
L = L_{source} + \lambda_t L_{pseudo}
\]
where:
\[
L_{source}=0.5L_{BCE}+0.5L_{Dice}
\]
and L_pseudo is a confidence-masked BCE loss on the unlabeled WWTP tiles.
Run the following cells.

configure

In [ ]:
import os
import random
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from PIL import Image
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp


# ============================================================
# EXPERIMENT 4 CONFIGURATION
# ============================================================

SEED = 42

SOURCE_CHECKPOINT = (
    "/kaggle/input/datasets/shanumkhsharma/"
    "kaggle-microplastic-bundle/checkpoints/"
    "beach_unet_resnet50_strong_aug_best_dice.pth"
)

BEACH_ROOT = (
    "/kaggle/input/datasets/shanumkhsharma/"
    "microplastic-step16a-recovery/kaggle_bundle/data"
)

WWTP_ROOT = (
    "/kaggle/input/datasets/shanumkhsharma/"
    "kaggle-microplastic-bundle/data/wwtp"
)

OUTPUT_DIR = "/kaggle/working/experiment4_confidence_pseudo_uda"
os.makedirs(OUTPUT_DIR, exist_ok=True)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Model
IMAGE_SIZE = 512

# Optimization
LR = 1e-4
WEIGHT_DECAY = 1e-4
BATCH_SIZE = 2
MAX_EPOCHS = 50
PATIENCE = 10

# Pseudo-label configuration
FG_THRESHOLD = 0.95
BG_THRESHOLD = 0.50

# Derived only from Beach validation
MAX_PSEUDO_FG_FRACTION = 0.119705

MAX_TARGET_LOSS_WEIGHT = 0.25

# WWTP tiling
TILE_SIZE = 512
STRIDE = 384
OVERLAP = 128

# AMP
USE_AMP = True

# Reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("Experiment 4 configuration loaded.")
print("Device:", DEVICE)
print("Output:", OUTPUT_DIR)

loss

In [ ]:
def dice_loss_from_logits(logits, targets, eps=1e-7):

    probs = torch.sigmoid(logits)

    probs = probs.reshape(
        probs.shape[0], -1
    )

    targets = targets.reshape(
        targets.shape[0], -1
    )

    intersection = (
        probs * targets
    ).sum(dim=1)

    denominator = (
        probs.sum(dim=1) +
        targets.sum(dim=1)
    )

    dice = (
        (2.0 * intersection + eps) /
        (denominator + eps)
    )

    return 1.0 - dice.mean()


def source_segmentation_loss(logits, masks):

    bce = F.binary_cross_entropy_with_logits(
        logits,
        masks
    )

    dice = dice_loss_from_logits(
        logits,
        masks
    )

    total = (
        0.5 * bce +
        0.5 * dice
    )

    return total, bce, dice


def masked_pseudo_bce(
    logits,
    pseudo_labels,
    confidence_mask
):

    bce = F.binary_cross_entropy_with_logits(
        logits,
        pseudo_labels,
        reduction="none"
    )

    # Only confident pixels contribute
    bce = bce * confidence_mask.float()

    valid_pixels = confidence_mask.sum()

    if valid_pixels == 0:
        return logits.sum() * 0.0

    return (
        bce.sum() /
        valid_pixels.float()
    )

Beach dataset

In [ ]:
class BeachDataset(Dataset):

    def __init__(
        self,
        image_dir,
        mask_dir
    ):

        self.image_dir = image_dir
        self.mask_dir = mask_dir

        image_files = sorted(
            [
                f for f in os.listdir(image_dir)
                if f.lower().endswith(
                    (".png", ".jpg", ".jpeg")
                )
            ]
        )

        self.samples = []

        for filename in image_files:

            stem = os.path.splitext(
                filename
            )[0]

            mask_path = os.path.join(
                mask_dir,
                stem + ".png"
            )

            if os.path.exists(mask_path):

                self.samples.append(
                    (
                        os.path.join(
                            image_dir,
                            filename
                        ),
                        mask_path
                    )
                )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):

        image_path, mask_path = self.samples[idx]

        image = Image.open(
            image_path
        ).convert("RGB")

        mask = Image.open(
            mask_path
        ).convert("L")

        image = image.resize(
            (IMAGE_SIZE, IMAGE_SIZE),
            Image.Resampling.BILINEAR
        )

        mask = mask.resize(
            (IMAGE_SIZE, IMAGE_SIZE),
            Image.Resampling.NEAREST
        )

        image = np.asarray(
            image,
            dtype=np.float32
        ) / 255.0

        mask = np.asarray(
            mask,
            dtype=np.float32
        ) / 255.0

        mask = (
            mask > 0.5
        ).astype(np.float32)

        image = torch.from_numpy(
            image
        ).permute(2, 0, 1)

        mask = torch.from_numpy(
            mask
        ).unsqueeze(0)

        return image, mask

In [ ]:
source_train_dataset = BeachDataset(
    os.path.join(BEACH_ROOT, "train/images"),
    os.path.join(BEACH_ROOT, "train/masks")
)

source_val_dataset = BeachDataset(
    os.path.join(BEACH_ROOT, "val/images"),
    os.path.join(BEACH_ROOT, "val/masks")
)

print("Beach train:", len(source_train_dataset))
print("Beach val:", len(source_val_dataset))

source_train_loader = DataLoader(
    source_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=True
)

source_val_loader = DataLoader(
    source_val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

WWTP unlabeld adpation dataset

In [ ]:
class WWTPUnlabeledRandomTileDataset(Dataset):

    def __init__(
        self,
        image_dir,
        tile_size=512
    ):

        self.image_paths = sorted(
            [
                os.path.join(image_dir, f)
                for f in os.listdir(image_dir)
                if f.lower().endswith(
                    (".jpg", ".jpeg", ".png")
                )
            ]
        )

        self.tile_size = tile_size

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):

        image_path = self.image_paths[idx]

        image = Image.open(
            image_path
        ).convert("RGB")

        width, height = image.size

        # All WWTP images are 3840x2160,
        # but this remains general.
        max_x = width - self.tile_size
        max_y = height - self.tile_size

        if max_x < 0 or max_y < 0:
            raise ValueError(
                f"Image smaller than tile: "
                f"{image_path}, size={image.size}"
            )

        # Random tile location
        x = random.randint(
            0,
            max_x
        )

        y = random.randint(
            0,
            max_y
        )

        tile = image.crop(
            (
                x,
                y,
                x + self.tile_size,
                y + self.tile_size
            )
        )

        tile = np.asarray(
            tile,
            dtype=np.float32
        ) / 255.0

        tile = torch.from_numpy(
            tile
        ).permute(2, 0, 1)

        return tile


target_train_dataset = WWTPUnlabeledRandomTileDataset(
    os.path.join(
        WWTP_ROOT,
        "images/train"
    )
)

target_train_loader = DataLoader(
    target_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=True
)

print(
    "WWTP unlabeled adaptation images:",
    len(target_train_dataset)
)

load the source model

In [ ]:
model = smp.Unet(
    encoder_name="resnet50",
    encoder_weights=None,
    in_channels=3,
    classes=1,
    activation=None
).to(DEVICE)

checkpoint = torch.load(
    SOURCE_CHECKPOINT,
    map_location=DEVICE
)

model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

print(
    "Loaded source checkpoint."
)

print(
    "Checkpoint epoch:",
    checkpoint.get("epoch")
)

print(
    "Source best validation Dice:",
    checkpoint.get("best_val_dice")
)

print(
    "Parameters:",
    sum(
        p.numel()
        for p in model.parameters()
    )
)

optimizer and amp

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=USE_AMP and DEVICE.type == "cuda"
)

print("Optimizer:", optimizer.__class__.__name__)
print("LR:", LR)
print("Weight decay:", WEIGHT_DECAY)

validation

In [ ]:
@torch.no_grad()
def evaluate_source_validation(
    model,
    loader
):

    model.eval()

    dice_scores = []
    iou_scores = []
    precision_scores = []
    recall_scores = []
    accuracy_scores = []

    for images, masks in loader:

        images = images.to(
            DEVICE,
            non_blocking=True
        )

        masks = masks.to(
            DEVICE,
            non_blocking=True
        )

        with torch.amp.autocast(
            device_type="cuda",
            enabled=USE_AMP and DEVICE.type == "cuda"
        ):
            logits = model(images)

        probs = torch.sigmoid(logits)

        preds = (
            probs >= 0.5
        ).float()

        for pred, target in zip(
            preds,
            masks
        ):

            pred = pred.flatten()
            target = target.flatten()

            tp = (
                (pred == 1) &
                (target == 1)
            ).sum().item()

            fp = (
                (pred == 1) &
                (target == 0)
            ).sum().item()

            fn = (
                (pred == 0) &
                (target == 1)
            ).sum().item()

            tn = (
                (pred == 0) &
                (target == 0)
            ).sum().item()

            dice = (
                2 * tp /
                max(
                    2 * tp + fp + fn,
                    1
                )
            )

            iou = (
                tp /
                max(
                    tp + fp + fn,
                    1
                )
            )

            precision = (
                tp /
                max(
                    tp + fp,
                    1
                )
            )

            recall = (
                tp /
                max(
                    tp + fn,
                    1
                )
            )

            accuracy = (
                (tp + tn) /
                max(
                    tp + tn + fp + fn,
                    1
                )
            )

            dice_scores.append(dice)
            iou_scores.append(iou)
            precision_scores.append(precision)
            recall_scores.append(recall)
            accuracy_scores.append(accuracy)

    return {
        "dice": float(np.mean(dice_scores)),
        "iou": float(np.mean(iou_scores)),
        "precision": float(np.mean(precision_scores)),
        "recall": float(np.mean(recall_scores)),
        "pixel_accuracy": float(np.mean(accuracy_scores))
    }

Experiment 4 training loop

In [ ]:
history = []

best_val_dice = -1.0
best_epoch = -1
epochs_without_improvement = 0

best_checkpoint_path = os.path.join(
    OUTPUT_DIR,
    "experiment4_confidence_pseudo_uda_best.pth"
)

for epoch in range(1, MAX_EPOCHS + 1):

    model.train()

    epoch_source_loss = []
    epoch_pseudo_loss = []
    epoch_total_loss = []

    epoch_confident_fraction = []
    epoch_pseudo_fg_fraction = []

    target_iterator = iter(
        target_train_loader
    )

    for source_images, source_masks in source_train_loader:

        # ----------------------------------------------------
        # Get target batch
        # ----------------------------------------------------

        try:
            target_images = next(
                target_iterator
            )
        except StopIteration:

            target_iterator = iter(
                target_train_loader
            )

            target_images = next(
                target_iterator
            )

        source_images = source_images.to(
            DEVICE,
            non_blocking=True
        )

        source_masks = source_masks.to(
            DEVICE,
            non_blocking=True
        )

        target_images = target_images.to(
            DEVICE,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        # ----------------------------------------------------
        # Forward pass
        # ----------------------------------------------------

        with torch.amp.autocast(
            device_type="cuda",
            enabled=USE_AMP and DEVICE.type == "cuda"
        ):

            # Source supervised segmentation
            source_logits = model(
                source_images
            )

            source_loss, _, _ = (
                source_segmentation_loss(
                    source_logits,
                    source_masks
                )
            )

            # ------------------------------------------------
            # Target pseudo-label generation
            # ------------------------------------------------
            #
            # IMPORTANT:
            # Pseudo-labels are detached from the graph.
            #

            target_logits_for_labels = model(
                target_images
            )

            target_probs = torch.sigmoid(
                target_logits_for_labels
            ).detach()

            # Candidate foreground
            fg_candidate = (
                target_probs >= FG_THRESHOLD
            )

            # Candidate background
            bg_candidate = (
                target_probs <= BG_THRESHOLD
            )

            # ------------------------------------------------
            # Foreground-area constraint
            # ------------------------------------------------

            final_fg = torch.zeros_like(
                fg_candidate
            )

            for b in range(
                target_images.shape[0]
            ):

                candidate = (
                    fg_candidate[b, 0]
                )

                candidate_count = (
                    candidate.sum().item()
                )

                max_allowed = int(
                    MAX_PSEUDO_FG_FRACTION *
                    candidate.numel()
                )

                if candidate_count <= max_allowed:

                    final_fg[b, 0] = candidate

                else:

                    # Keep the highest-confidence
                    # candidate foreground pixels.
                    candidate_probs = (
                        target_probs[b, 0]
                    )

                    candidate_indices = (
                        torch.where(candidate)
                    )

                    candidate_values = (
                        candidate_probs[
                            candidate_indices
                        ]
                    )

                    top_k = min(
                        max_allowed,
                        candidate_values.numel()
                    )

                    if top_k > 0:

                        top_indices = torch.topk(
                            candidate_values,
                            k=top_k,
                            largest=True
                        ).indices

                        selected_y = (
                            candidate_indices[0][
                                top_indices
                            ]
                        )

                        selected_x = (
                            candidate_indices[1][
                                top_indices
                            ]
                        )

                        final_fg[
                            b,
                            0,
                            selected_y,
                            selected_x
                        ] = True

            # ------------------------------------------------
            # Background candidate
            #
            # Never allow a pixel to be both FG and BG.
            # ------------------------------------------------

            final_bg = (
                bg_candidate &
                (~final_fg)
            )

            confidence_mask = (
                final_fg |
                final_bg
            )

            pseudo_labels = (
                final_fg.float()
            )

            # Target pseudo-label loss
            pseudo_loss = masked_pseudo_bce(
                target_logits_for_labels,
                pseudo_labels,
                confidence_mask
            )

            # ------------------------------------------------
            # Ramp target loss
            # ------------------------------------------------

            progress = (
                (epoch - 1) /
                max(MAX_EPOCHS - 1, 1)
            )

            target_weight = (
                MAX_TARGET_LOSS_WEIGHT *
                progress
            )

            total_loss = (
                source_loss +
                target_weight * pseudo_loss
            )

        # ----------------------------------------------------
        # Backward
        # ----------------------------------------------------

        scaler.scale(
            total_loss
        ).backward()

        scaler.step(
            optimizer
        )

        scaler.update()

        # ----------------------------------------------------
        # Statistics
        # ----------------------------------------------------

        epoch_source_loss.append(
            source_loss.detach().item()
        )

        epoch_pseudo_loss.append(
            pseudo_loss.detach().item()
        )

        epoch_total_loss.append(
            total_loss.detach().item()
        )

        epoch_confident_fraction.append(
            confidence_mask.float()
            .mean()
            .item()
        )

        epoch_pseudo_fg_fraction.append(
            final_fg.float()
            .mean()
            .item()
        )

        del (
            source_images,
            source_masks,
            target_images,
            source_logits,
            target_logits_for_labels,
            target_probs,
            fg_candidate,
            bg_candidate,
            final_fg,
            final_bg,
            confidence_mask,
            pseudo_labels
        )

    # --------------------------------------------------------
    # Beach validation
    # --------------------------------------------------------

    val_metrics = evaluate_source_validation(
        model,
        source_val_loader
    )

    mean_source_loss = np.mean(
        epoch_source_loss
    )

    mean_pseudo_loss = np.mean(
        epoch_pseudo_loss
    )

    mean_total_loss = np.mean(
        epoch_total_loss
    )

    mean_confidence = np.mean(
        epoch_confident_fraction
    )

    mean_pseudo_fg = np.mean(
        epoch_pseudo_fg_fraction
    )

    record = {
        "epoch": epoch,
        "train_source_loss": float(
            mean_source_loss
        ),
        "train_pseudo_loss": float(
            mean_pseudo_loss
        ),
        "train_total_loss": float(
            mean_total_loss
        ),
        "target_loss_weight": float(
            target_weight
        ),
        "pseudo_confident_fraction": float(
            mean_confidence
        ),
        "pseudo_foreground_fraction": float(
            mean_pseudo_fg
        ),
        "val_dice": val_metrics["dice"],
        "val_iou": val_metrics["iou"],
        "val_precision": val_metrics["precision"],
        "val_recall": val_metrics["recall"],
        "val_pixel_accuracy": val_metrics[
            "pixel_accuracy"
        ]
    }

    history.append(record)

    print(
        f"Epoch {epoch:02d} | "
        f"Src {mean_source_loss:.4f} | "
        f"Pseudo {mean_pseudo_loss:.4f} | "
        f"λ {target_weight:.4f} | "
        f"Conf {mean_confidence:.4%} | "
        f"PseudoFG {mean_pseudo_fg:.4%} | "
        f"Val Dice {val_metrics['dice']:.4f} | "
        f"Val IoU {val_metrics['iou']:.4f}"
    )

    # --------------------------------------------------------
    # Checkpoint based ONLY on Beach validation Dice
    # --------------------------------------------------------

    if val_metrics["dice"] > best_val_dice:

        best_val_dice = (
            val_metrics["dice"]
        )

        best_epoch = epoch
        epochs_without_improvement = 0

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_val_dice": best_val_dice,
                "history": history,
                "experiment": (
                    "Experiment 4 - "
                    "Confidence-Controlled "
                    "Pseudo-Label UDA"
                ),
                "seed": SEED,
                "fg_threshold": FG_THRESHOLD,
                "bg_threshold": BG_THRESHOLD,
                "max_pseudo_fg_fraction": (
                    MAX_PSEUDO_FG_FRACTION
                ),
                "max_target_loss_weight": (
                    MAX_TARGET_LOSS_WEIGHT
                )
            },
            best_checkpoint_path
        )

        print(
            f"  ✓ New best checkpoint "
            f"(epoch {epoch})"
        )

    else:

        epochs_without_improvement += 1

        if (
            epochs_without_improvement >=
            PATIENCE
        ):

            print(
                f"\nEarly stopping at epoch {epoch}."
            )

            break

In [ ]:
history_path = os.path.join(
    OUTPUT_DIR,
    "training_history.json"
)

with open(
    history_path,
    "w"
) as f:

    json.dump(
        history,
        f,
        indent=2
    )

config = {
    "experiment": (
        "Experiment 4 - "
        "Confidence-Controlled "
        "Pseudo-Label UDA"
    ),
    "architecture": "U-Net",
    "encoder": "ResNet50",
    "encoder_weights": "Strong-Augmentation checkpoint",
    "input_size": "512x512",
    "source_domain": "Beach",
    "target_domain": "WWTP",
    "target_training_split": "WWTP train only",
    "target_labels_used": False,
    "learning_rate": LR,
    "weight_decay": WEIGHT_DECAY,
    "batch_size": BATCH_SIZE,
    "max_epochs": MAX_EPOCHS,
    "early_stopping_patience": PATIENCE,
    "seed": SEED,
    "fg_threshold": FG_THRESHOLD,
    "bg_threshold": BG_THRESHOLD,
    "max_pseudo_fg_fraction": (
        MAX_PSEUDO_FG_FRACTION
    ),
    "max_target_loss_weight": (
        MAX_TARGET_LOSS_WEIGHT
    ),
    "tile_size": TILE_SIZE,
    "stride": STRIDE,
    "model_selection": (
        "Beach validation Dice"
    )
}

with open(
    os.path.join(
        OUTPUT_DIR,
        "experiment4_config.json"
    ),
    "w"
) as f:

    json.dump(
        config,
        f,
        indent=2
    )

print("\nExperiment 4 training artifacts saved.")
print("Best epoch:", best_epoch)
print("Best Beach validation Dice:", best_val_dice)
print("Checkpoint:", best_checkpoint_path)

In [ ]:
# Load best Experiment 4 checkpoint for final evaluation

BEST_E4 = (
    "/kaggle/working/experiment4_confidence_pseudo_uda/"
    "experiment4_confidence_pseudo_uda_best.pth"
)

checkpoint_e4 = torch.load(
    BEST_E4,
    map_location=DEVICE
)

model.load_state_dict(
    checkpoint_e4["model_state_dict"],
    strict=True
)

model.eval()

print("Loaded Experiment 4 best checkpoint")
print("Epoch:", checkpoint_e4["epoch"])
print("Best Beach Val Dice:", checkpoint_e4["best_val_dice"])

se the same protocol as the previous experiments:
- Native WWTP image: 3840×2160
- Tile: 512×512
- Stride: 384
- Overlap: 128 px
- Average overlapping logits
- Sigmoid
- Threshold: 0.5
- Evaluate WWTP validation and test separately
- Do not use WWTP labels for any model selection
Run this evaluation code.

wwtp evaluation dataset

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import torch

from PIL import Image
from tqdm.auto import tqdm


WWTP_ROOT = (
    "/kaggle/input/datasets/shanumkhsharma/"
    "kaggle-microplastic-bundle/data/wwtp"
)

TILE_SIZE = 512
STRIDE = 384
PRED_THRESHOLD = 0.5


def get_positions(length, tile_size, stride):

    positions = list(
        range(
            0,
            max(1, length - tile_size + 1),
            stride
        )
    )

    final_position = length - tile_size

    if positions[-1] != final_position:
        positions.append(final_position)

    return positions


def load_wwtp_pairs(split):

    image_dir = os.path.join(
        WWTP_ROOT,
        "images",
        split
    )

    mask_dir = os.path.join(
        WWTP_ROOT,
        "masks",
        split
    )

    image_paths = sorted(
        glob.glob(
            os.path.join(
                image_dir,
                "*"
            )
        )
    )

    pairs = []

    for image_path in image_paths:

        stem = os.path.splitext(
            os.path.basename(image_path)
        )[0]

        mask_path = os.path.join(
            mask_dir,
            stem + ".png"
        )

        if os.path.exists(mask_path):

            pairs.append(
                (image_path, mask_path)
            )

    return pairs


validation_pairs = load_wwtp_pairs(
    "validation"
)

test_pairs = load_wwtp_pairs(
    "test"
)

print(
    "WWTP validation pairs:",
    len(validation_pairs)
)

print(
    "WWTP test pairs:",
    len(test_pairs)
)

evalaution fucntion

In [ ]:
@torch.no_grad()
def evaluate_wwtp_split(
    model,
    pairs,
    split_name
):

    model.eval()

    results = []

    for image_path, mask_path in tqdm(
        pairs,
        desc=f"WWTP {split_name}"
    ):

        image = Image.open(
            image_path
        ).convert("RGB")

        gt_mask = Image.open(
            mask_path
        ).convert("L")

        image_np = np.asarray(
            image,
            dtype=np.float32
        ) / 255.0

        gt_np = np.asarray(
            gt_mask,
            dtype=np.uint8
        )

        height, width = gt_np.shape

        x_positions = get_positions(
            width,
            TILE_SIZE,
            STRIDE
        )

        y_positions = get_positions(
            height,
            TILE_SIZE,
            STRIDE
        )

        # Accumulate logits from overlapping tiles
        logit_sum = np.zeros(
            (height, width),
            dtype=np.float32
        )

        count_map = np.zeros(
            (height, width),
            dtype=np.float32
        )

        for y in y_positions:

            for x in x_positions:

                tile = image_np[
                    y:y + TILE_SIZE,
                    x:x + TILE_SIZE
                ]

                tile_tensor = torch.from_numpy(
                    tile
                ).permute(
                    2, 0, 1
                ).unsqueeze(0).to(
                    DEVICE,
                    non_blocking=True
                )

                with torch.amp.autocast(
                    device_type="cuda",
                    enabled=(
                        USE_AMP and
                        DEVICE.type == "cuda"
                    )
                ):

                    logits = model(
                        tile_tensor
                    )

                logits_np = (
                    logits[0, 0]
                    .float()
                    .cpu()
                    .numpy()
                )

                logit_sum[
                    y:y + TILE_SIZE,
                    x:x + TILE_SIZE
                ] += logits_np

                count_map[
                    y:y + TILE_SIZE,
                    x:x + TILE_SIZE
                ] += 1

        # Average overlapping logits
        averaged_logits = (
            logit_sum /
            np.maximum(
                count_map,
                1e-8
            )
        )

        probabilities = 1.0 / (
            1.0 +
            np.exp(
                -averaged_logits
            )
        )

        prediction = (
            probabilities >=
            PRED_THRESHOLD
        )

        target = (
            gt_np > 127
        )

        tp = np.logical_and(
            prediction,
            target
        ).sum()

        fp = np.logical_and(
            prediction,
            ~target
        ).sum()

        fn = np.logical_and(
            ~prediction,
            target
        ).sum()

        tn = np.logical_and(
            ~prediction,
            ~target
        ).sum()

        dice = (
            2.0 * tp /
            max(
                2.0 * tp + fp + fn,
                1
            )
        )

        iou = (
            tp /
            max(
                tp + fp + fn,
                1
            )
        )

        precision = (
            tp /
            max(
                tp + fp,
                1
            )
        )

        recall = (
            tp /
            max(
                tp + fn,
                1
            )
        )

        accuracy = (
            (tp + tn) /
            max(
                tp + tn + fp + fn,
                1
            )
        )

        gt_fraction = (
            target.mean()
        )

        predicted_fraction = (
            prediction.mean()
        )

        results.append(
            {
                "image": os.path.basename(
                    image_path
                ),
                "split": split_name,
                "dice": float(dice),
                "iou": float(iou),
                "precision": float(precision),
                "recall": float(recall),
                "pixel_accuracy": float(
                    accuracy
                ),
                "gt_foreground_fraction": float(
                    gt_fraction
                ),
                "pred_foreground_fraction": float(
                    predicted_fraction
                ),
                "foreground_area_ratio": float(
                    predicted_fraction /
                    max(
                        gt_fraction,
                        1e-12
                    )
                ),
                "gt_foreground_pixels": int(
                    target.sum()
                ),
                "pred_foreground_pixels": int(
                    prediction.sum()
                )
            }
        )

    return pd.DataFrame(results)

run wwtp vlaidation

In [ ]:
wwtp_val_results = evaluate_wwtp_split(
    model,
    validation_pairs,
    "validation"
)

print("\nWWTP VALIDATION")
print("=" * 60)

print(
    wwtp_val_results[
        [
            "dice",
            "iou",
            "precision",
            "recall",
            "pixel_accuracy",
            "gt_foreground_fraction",
            "pred_foreground_fraction",
            "foreground_area_ratio"
        ]
    ].mean()
)

run wwtp test

In [ ]:
wwtp_test_results = evaluate_wwtp_split(
    model,
    test_pairs,
    "test"
)

print("\nWWTP TEST")
print("=" * 60)

print(
    wwtp_test_results[
        [
            "dice",
            "iou",
            "precision",
            "recall",
            "pixel_accuracy",
            "gt_foreground_fraction",
            "pred_foreground_fraction",
            "foreground_area_ratio"
        ]
    ].mean()
)

In [ ]:
evaluation_dir = os.path.join(
    OUTPUT_DIR,
    "evaluation"
)

os.makedirs(
    evaluation_dir,
    exist_ok=True
)

wwtp_val_results.to_csv(
    os.path.join(
        evaluation_dir,
        "wwtp_validation_per_image.csv"
    ),
    index=False
)

wwtp_test_results.to_csv(
    os.path.join(
        evaluation_dir,
        "wwtp_test_per_image.csv"
    ),
    index=False
)

print(
    "Saved evaluation results to:",
    evaluation_dir
)

In [ ]:
def summarize_results(df):

    metrics = [
        "dice",
        "iou",
        "precision",
        "recall",
        "pixel_accuracy",
        "gt_foreground_fraction",
        "pred_foreground_fraction",
        "foreground_area_ratio"
    ]

    return {
        metric: {
            "mean": float(
                df[metric].mean()
            ),
            "std": float(
                df[metric].std()
            ),
            "median": float(
                df[metric].median()
            )
        }
        for metric in metrics
    }


val_summary = summarize_results(
    wwtp_val_results
)

test_summary = summarize_results(
    wwtp_test_results
)

summary = {
    "experiment": (
        "Experiment 4 - "
        "Confidence-Controlled "
        "Pseudo-Label UDA"
    ),
    "checkpoint_epoch": 1,
    "beach_val_dice": 0.6703659201339194,
    "foreground_threshold": 0.95,
    "background_threshold": 0.50,
    "max_pseudo_foreground_fraction": 0.119705,
    "wwtp_validation": val_summary,
    "wwtp_test": test_summary
}

with open(
    os.path.join(
        evaluation_dir,
        "experiment4_wwtp_summary.json"
    ),
    "w"
) as f:

    json.dump(
        summary,
        f,
        indent=2
    )

print(
    json.dumps(
        summary,
        indent=2
    )
)

Confidence-controlled pseudo-label UDA did not improve cross-domain segmentation. Although recall increased to 0.7690, precision decreased to 0.0100 and predicted foreground expanded to 93.7% of the WWTP test image area. This indicates severe confirmation bias and foreground propagation from unreliable target-domain pseudo-labels.

In [ ]:
import json
import os

EXP4_DIR = "/kaggle/working/experiment4_confidence_pseudo_uda"

with open(os.path.join(EXP4_DIR, "training_history.json"), "r") as f:
    history = json.load(f)

print(history[0].keys())
print(history[0])

In [ ]:
import json
import os
import matplotlib.pyplot as plt

EXP4_DIR = "/kaggle/working/experiment4_confidence_pseudo_uda"

with open(os.path.join(EXP4_DIR, "training_history.json"), "r") as f:
    history = json.load(f)

epochs = [h["epoch"] for h in history]
val_dice = [h["val_dice"] for h in history]

plt.figure(figsize=(8,5))
plt.plot(epochs, val_dice, marker="o")

plt.xlabel("Epoch")
plt.ylabel("Beach Validation Dice")
plt.title("Experiment 4: Beach Validation Dice")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import json
import os
import numpy as np
import matplotlib.pyplot as plt

EXP4_DIR = "/kaggle/working/experiment4_confidence_pseudo_uda"

with open(os.path.join(EXP4_DIR, "training_history.json"), "r") as f:
    history = json.load(f)

epochs = [h["epoch"] for h in history]

source_loss = [h["train_source_loss"] for h in history]
pseudo_loss = [h["train_pseudo_loss"] for h in history]
total_loss = [h["train_total_loss"] for h in history]

target_weight = [h["target_loss_weight"] for h in history]
confidence = [h["pseudo_confident_fraction"] for h in history]
pseudo_fg = [h["pseudo_foreground_fraction"] for h in history]

val_dice = [h["val_dice"] for h in history]
val_iou = [h["val_iou"] for h in history]
val_precision = [h["val_precision"] for h in history]
val_recall = [h["val_recall"] for h in history]
val_accuracy = [h["val_pixel_accuracy"] for h in history]

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(epochs, val_dice, marker="o", label="Validation Dice")

best_idx = np.argmax(val_dice)

plt.scatter(
    epochs[best_idx],
    val_dice[best_idx],
    s=100,
    zorder=5,
    label=f"Best = {val_dice[best_idx]:.4f} (Epoch {epochs[best_idx]})"
)

plt.xlabel("Epoch")
plt.ylabel("Dice")
plt.title("Experiment 4: Beach Validation Dice")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(epochs, source_loss, marker="o", label="Source segmentation loss")
plt.plot(epochs, pseudo_loss, marker="o", label="Pseudo-label loss")
plt.plot(epochs, total_loss, marker="o", label="Total loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Experiment 4: Training Losses")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(
    epochs,
    target_weight,
    marker="o"
)

plt.xlabel("Epoch")
plt.ylabel("Target Loss Weight")
plt.title("Experiment 4: Target Pseudo-Label Loss Weight")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(
    epochs,
    np.array(confidence) * 100,
    marker="o"
)

plt.xlabel("Epoch")
plt.ylabel("Confident Target Pixels (%)")
plt.title("Experiment 4: Pseudo-Label Confidence Coverage")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(
    epochs,
    np.array(pseudo_fg) * 100,
    marker="o",
    label="Pseudo-foreground fraction"
)

plt.axhline(
    11.9705,
    linestyle="--",
    label="Maximum allowed = 11.97%"
)

plt.xlabel("Epoch")
plt.ylabel("Pseudo-foreground (%)")
plt.title("Experiment 4: Target Pseudo-Label Foreground Fraction")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9,5))

plt.plot(epochs, val_dice, marker="o", label="Dice")
plt.plot(epochs, val_iou, marker="o", label="IoU")
plt.plot(epochs, val_precision, marker="o", label="Precision")
plt.plot(epochs, val_recall, marker="o", label="Recall")

plt.xlabel("Epoch")
plt.ylabel("Score")
plt.title("Experiment 4: Beach Validation Metrics")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

VAL_CSV = os.path.join(
    EXP4_DIR,
    "evaluation",
    "wwtp_validation_per_image.csv"
)

TEST_CSV = os.path.join(
    EXP4_DIR,
    "evaluation",
    "wwtp_test_per_image.csv"
)

val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)

print("Validation:")
print("GT:", val_df["gt_foreground_fraction"].mean() * 100)
print("Pred:", val_df["pred_foreground_fraction"].mean() * 100)

print("\nTest:")
print("GT:", test_df["gt_foreground_fraction"].mean() * 100)
print("Pred:", test_df["pred_foreground_fraction"].mean() * 100)

In [ ]:
plt.figure(figsize=(8,5))

plt.bar(
    ["Ground Truth", "Prediction"],
    [
        test_df["gt_foreground_fraction"].mean() * 100,
        test_df["pred_foreground_fraction"].mean() * 100
    ]
)

plt.ylabel("Foreground Area (%)")
plt.title("Experiment 4: WWTP Test Foreground Area")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7,5))

values = [
    test_df["precision"].mean(),
    test_df["recall"].mean()
]

bars = plt.bar(
    ["Precision", "Recall"],
    values
)

plt.ylabel("Score")
plt.ylim(0, 1)
plt.title("Experiment 4: WWTP Test Precision vs Recall")
plt.grid(axis="y", alpha=0.3)

for bar, value in zip(bars, values):
    plt.text(
        bar.get_x() + bar.get_width()/2,
        value + 0.02,
        f"{value:.3f}",
        ha="center"
    )

plt.tight_layout()
plt.show()

In [ ]:
models = [
    "Baseline\nU-Net",
    "Strong\nAug.",
    "Appearance\nAug.",
    "DeepLabV3+",
    "DANN",
    "Pseudo-label\nUDA"
]

dice = [
    0.0604,
    0.0475,
    0.0631,
    0.0566,
    0.0188,
    0.01915
]

plt.figure(figsize=(10,6))

bars = plt.bar(models, dice)

plt.ylabel("WWTP Dice")
plt.title("Unseen WWTP Dice Across Experiments")
plt.ylim(0, 0.07)
plt.grid(axis="y", alpha=0.3)

for bar, value in zip(bars, dice):
    plt.text(
        bar.get_x() + bar.get_width()/2,
        value + 0.001,
        f"{value:.3f}",
        ha="center"
    )

plt.tight_layout()
plt.show()

#Experiment 5:

EMA Teacher–Student UDA.
Build student + teacher U-Net ResNet50.
Initialize both from the same source checkpoint.
Freeze teacher gradients.
Train student on Beach supervised loss.
Run teacher on unlabeled WWTP train tiles.
Generate confidence-filtered teacher pseudo-labels.
Add target consistency loss to student.
Update teacher with EMA after every student optimizer step.
Select checkpoint using Beach validation Dice.
Evaluate once on WWTP validation/test using the same 512×512, stride-384 tiling protocol.

Next implementation sequence:
1. Build student + teacher U-Net ResNet50.
2. Initialize both from the same source checkpoint.
3. Freeze teacher gradients.
4. Train student on Beach supervised loss.
5. Run teacher on unlabeled WWTP train tiles.
6. Generate confidence-filtered teacher pseudo-labels.
7. Add target consistency loss to student.
8. Update teacher with EMA after every student optimizer step.
9. Select checkpoint using Beach validation Dice.
10. Evaluate once on WWTP validation/test using the same 512×512, stride-384 tiling protocol.

1. Configuration and paths

In [ ]:
import os
import json
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import segmentation_models_pytorch as smp

from torch.utils.data import Dataset, DataLoader

SEED = 42

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

BEACH_ROOT = (
    "/kaggle/input/datasets/shanumkhsharma/"
    "microplastic-step16a-recovery/kaggle_bundle/data"
)

WWTP_ROOT = (
    "/kaggle/input/datasets/shanumkhsharma/"
    "kaggle-microplastic-bundle/data/wwtp"
)

# Use the same available source checkpoint as Experiment 4.
INIT_CHECKPOINT = (
    "/kaggle/input/datasets/shanumkhsharma/"
    "kaggle-microplastic-bundle/checkpoints/"
    "beach_unet_resnet50_strong_aug_best_dice.pth"
)

OUT_DIR = "/kaggle/working/experiment5_ema_teacher_student"

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUT_DIR, "evaluation"), exist_ok=True)

print("Checkpoint exists:", os.path.exists(INIT_CHECKPOINT))

2. Beach supervised dataset

In [ ]:
from PIL import Image

class BeachDataset(Dataset):
    def __init__(self, root, split):
        self.image_dir = os.path.join(root, split, "images")
        self.mask_dir = os.path.join(root, split, "masks")

        self.images = sorted([
            f for f in os.listdir(self.image_dir)
            if f.lower().endswith((".png", ".jpg", ".jpeg"))
        ])

        self.images = [
            f for f in self.images
            if os.path.exists(
                os.path.join(
                    self.mask_dir,
                    os.path.splitext(f)[0] + ".png"
                )
            )
        ]

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        name = self.images[idx]

        image = Image.open(
            os.path.join(self.image_dir, name)
        ).convert("RGB")

        mask = Image.open(
            os.path.join(
                self.mask_dir,
                os.path.splitext(name)[0] + ".png"
            )
        ).convert("L")

        image = image.resize((512, 512), Image.BILINEAR)
        mask = mask.resize((512, 512), Image.NEAREST)

        image = np.asarray(image, dtype=np.float32) / 255.0
        mask = (np.asarray(mask) > 127).astype(np.float32)

        image = torch.from_numpy(image).permute(2, 0, 1)
        mask = torch.from_numpy(mask).unsqueeze(0)

        return image, mask

check loaders

In [ ]:
train_ds = BeachDataset(BEACH_ROOT, "train")
val_ds   = BeachDataset(BEACH_ROOT, "val")
test_ds  = BeachDataset(BEACH_ROOT, "test")

train_loader = DataLoader(
    train_ds,
    batch_size=2,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=2,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_ds,
    batch_size=2,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Beach train:", len(train_ds))
print("Beach val:", len(val_ds))
print("Beach test:", len(test_ds))

WWTP unlabeled dataset

In [ ]:
class WWTPUnlabeledRandomTileDataset(Dataset):
    def __init__(
        self,
        root,
        tile_size=512,
        stride=384
    ):
        self.image_dir = os.path.join(root, "images", "train")

        self.images = sorted([
            f for f in os.listdir(self.image_dir)
            if f.lower().endswith((".jpg", ".jpeg", ".png"))
        ])

        self.tile_size = tile_size
        self.stride = stride

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        name = self.images[idx]

        image = Image.open(
            os.path.join(self.image_dir, name)
        ).convert("RGB")

        image = np.asarray(image)

        H, W = image.shape[:2]

        if H < self.tile_size or W < self.tile_size:
            raise ValueError(
                f"Image too small: {name}, shape={image.shape}"
            )

        ys = list(range(0, H - self.tile_size + 1, self.stride))
        xs = list(range(0, W - self.tile_size + 1, self.stride))

        if ys[-1] != H - self.tile_size:
            ys.append(H - self.tile_size)

        if xs[-1] != W - self.tile_size:
            xs.append(W - self.tile_size)

        y = random.choice(ys)
        x = random.choice(xs)

        tile = image[
            y:y+self.tile_size,
            x:x+self.tile_size
        ]

        tile = tile.astype(np.float32) / 255.0

        tile = torch.from_numpy(tile).permute(2, 0, 1)

        return tile

In [ ]:
target_ds = WWTPUnlabeledRandomTileDataset(WWTP_ROOT)

target_loader = DataLoader(
    target_ds,
    batch_size=2,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=True
)

print("WWTP unlabeled training images:", len(target_ds))

Define student and teacher

In [ ]:
def create_model():
    return smp.Unet(
        encoder_name="resnet50",
        encoder_weights=None,
        in_channels=3,
        classes=1,
        activation=None
    ).to(DEVICE)

In [ ]:
checkpoint = torch.load(
    INIT_CHECKPOINT,
    map_location=DEVICE
)

print(checkpoint.keys())

student = create_model()
teacher = create_model()

student.load_state_dict(checkpoint["model_state_dict"])
teacher.load_state_dict(checkpoint["model_state_dict"])

student = student.to(DEVICE)
teacher = teacher.to(DEVICE)

teacher.eval()

for p in teacher.parameters():
    p.requires_grad = False

print("Student parameters:", sum(p.numel() for p in student.parameters()))
print("Teacher parameters:", sum(p.numel() for p in teacher.parameters()))

EMA update

eacher_new = 0.99 × Teacher_old + 0.01 × Student_current

In [ ]:
EMA_DECAY = 0.99

@torch.no_grad()
def update_teacher(student, teacher, decay=EMA_DECAY):
    student_state = student.state_dict()
    teacher_state = teacher.state_dict()

    for key in teacher_state.keys():
        # EMA only for floating-point parameters/buffers
        if teacher_state[key].is_floating_point():
            teacher_state[key].mul_(decay)
            teacher_state[key].add_(
                student_state[key],
                alpha=1.0 - decay
            )
        else:
            # Integer buffers such as BatchNorm num_batches_tracked
            teacher_state[key].copy_(student_state[key])

Loss functions

In [ ]:
bce_loss = nn.BCEWithLogitsLoss()

def dice_loss(logits, targets, eps=1e-6):
    probs = torch.sigmoid(logits)

    intersection = (probs * targets).sum(dim=(1,2,3))
    denominator = (
        probs.sum(dim=(1,2,3))
        + targets.sum(dim=(1,2,3))
    )

    dice = (2.0 * intersection + eps) / (
        denominator + eps
    )

    return 1.0 - dice.mean()


def source_segmentation_loss(logits, masks):
    bce = bce_loss(logits, masks)
    dice = dice_loss(logits, masks)

    return 0.5 * bce + 0.5 * dice

Confidence-controlled teacher pseudo-labels

In [ ]:
FG_THRESHOLD = 0.95
BG_THRESHOLD = 0.50

MAX_PSEUDO_FG_FRACTION = 0.119705

Generating teacher pseudo-labels

In [ ]:
@torch.no_grad()
def make_teacher_pseudo_labels(logits):
    probs = torch.sigmoid(logits)

    fg = probs >= FG_THRESHOLD
    bg = probs <= BG_THRESHOLD

    # Limit excessive pseudo-foreground.
    B, _, H, W = probs.shape

    final_fg = torch.zeros_like(fg)

    max_pixels = int(
        MAX_PSEUDO_FG_FRACTION * H * W
    )

    for b in range(B):

        candidate = fg[b, 0]

        n_candidate = candidate.sum().item()

        if n_candidate > max_pixels:
            candidate_probs = probs[b, 0].clone()

            candidate_probs[~candidate] = -1

            flat = candidate_probs.flatten()

            _, indices = torch.topk(
                flat,
                k=max_pixels
            )

            selected = torch.zeros_like(flat, dtype=torch.bool)
            selected[indices] = True

            final_fg[b, 0] = selected.reshape(H, W)

        else:
            final_fg[b, 0] = candidate

    # Background cannot overlap foreground.
    final_bg = bg & (~final_fg)

    valid = final_fg | final_bg

    pseudo = torch.zeros_like(probs)

    pseudo[final_fg] = 1.0
    pseudo[final_bg] = 0.0

    return pseudo, valid, probs

Masked pseudo-label loss

In [ ]:
def masked_pseudo_loss(student_logits, pseudo, valid):
    loss_map = F.binary_cross_entropy_with_logits(
        student_logits,
        pseudo,
        reduction="none"
    )

    valid = valid.float()

    denominator = valid.sum().clamp_min(1.0)

    return (loss_map * valid).sum() / denominator

Optimizer and AMP

In [ ]:
optimizer = torch.optim.AdamW(
    student.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

scaler = torch.cuda.amp.GradScaler(
    enabled=torch.cuda.is_available()
)

Validation function

In [ ]:
@torch.no_grad()
def evaluate_beach(model, loader):
    model.eval()

    dices = []
    ious = []
    precisions = []
    recalls = []
    accuracies = []

    for images, masks in loader:

        images = images.to(
            DEVICE,
            non_blocking=True
        )

        masks = masks.to(
            DEVICE,
            non_blocking=True
        )

        logits = model(images)
        probs = torch.sigmoid(logits)

        preds = (probs >= 0.5).float()

        tp = (preds * masks).sum(dim=(1,2,3))
        fp = (preds * (1-masks)).sum(dim=(1,2,3))
        fn = ((1-preds) * masks).sum(dim=(1,2,3))
        tn = ((1-preds) * (1-masks)).sum(dim=(1,2,3))

        dice = (2*tp + 1e-6) / (
            2*tp + fp + fn + 1e-6
        )

        iou = (tp + 1e-6) / (
            tp + fp + fn + 1e-6
        )

        precision = (tp + 1e-6) / (
            tp + fp + 1e-6
        )

        recall = (tp + 1e-6) / (
            tp + fn + 1e-6
        )

        accuracy = (tp + tn + 1e-6) / (
            tp + tn + fp + fn + 1e-6
        )

        dices.extend(dice.cpu().numpy())
        ious.extend(iou.cpu().numpy())
        precisions.extend(precision.cpu().numpy())
        recalls.extend(recall.cpu().numpy())
        accuracies.extend(accuracy.cpu().numpy())

    return {
        "dice": float(np.mean(dices)),
        "iou": float(np.mean(ious)),
        "precision": float(np.mean(precisions)),
        "recall": float(np.mean(recalls)),
        "pixel_accuracy": float(np.mean(accuracies))
    }

First diagnostic run

In [ ]:
student.train()
teacher.eval()

source_iter = iter(train_loader)
target_iter = iter(target_loader)

source_images, source_masks = next(source_iter)
target_images = next(target_iter)

source_images = source_images.to(DEVICE)
source_masks = source_masks.to(DEVICE)
target_images = target_images.to(DEVICE)

optimizer.zero_grad(set_to_none=True)

with torch.amp.autocast(
    "cuda",
    enabled=torch.cuda.is_available()
):
    source_logits = student(source_images)

    source_loss = source_segmentation_loss(
        source_logits,
        source_masks
    )

    with torch.no_grad():
        teacher_logits = teacher(target_images)

    pseudo, valid, teacher_probs = make_teacher_pseudo_labels(
        teacher_logits
    )

    student_target_logits = student(target_images)

    pseudo_loss = masked_pseudo_loss(
        student_target_logits,
        pseudo,
        valid
    )

    total_loss = source_loss + 0.25 * pseudo_loss

scaler.scale(total_loss).backward()

scaler.unscale_(optimizer)

grad_finite = all(
    p.grad is None or torch.isfinite(p.grad).all()
    for p in student.parameters()
)

scaler.step(optimizer)
scaler.update()

update_teacher(student, teacher)

print("Source loss:", source_loss.item())
print("Pseudo loss:", pseudo_loss.item())
print("Total loss:", total_loss.item())
print("Confident fraction:", valid.float().mean().item())
print("Pseudo foreground:", pseudo.mean().item())
print("Gradients finite:", grad_finite)

Experiment 5 training

In [ ]:
import time

MAX_EPOCHS = 50
PATIENCE = 10

BEST_DICE = -1.0
BEST_EPOCH = 0
PATIENCE_COUNTER = 0

history_ema = []

for epoch in range(1, MAX_EPOCHS + 1):

    epoch_start = time.time()

    student.train()
    teacher.eval()

    source_iter = iter(train_loader)
    target_iter = iter(target_loader)

    num_steps = min(
        len(train_loader),
        len(target_loader)
    )

    running_source = 0.0
    running_pseudo = 0.0
    running_total = 0.0
    running_conf = 0.0
    running_fg = 0.0

    for step in range(num_steps):

        source_images, source_masks = next(source_iter)
        target_images = next(target_iter)

        source_images = source_images.to(
            DEVICE,
            non_blocking=True
        )

        source_masks = source_masks.to(
            DEVICE,
            non_blocking=True
        )

        target_images = target_images.to(
            DEVICE,
            non_blocking=True
        )

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(
            "cuda",
            enabled=torch.cuda.is_available()
        ):

            # -------------------------
            # 1. Source supervised loss
            # -------------------------
            source_logits = student(source_images)

            source_loss = source_segmentation_loss(
                source_logits,
                source_masks
            )

            # -------------------------
            # 2. Teacher target prediction
            # -------------------------
            with torch.no_grad():

                teacher_logits = teacher(
                    target_images
                )

                pseudo, valid, teacher_probs = (
                    make_teacher_pseudo_labels(
                        teacher_logits
                    )
                )

            # -------------------------
            # 3. Student target prediction
            # -------------------------
            student_target_logits = student(
                target_images
            )

            pseudo_loss = masked_pseudo_loss(
                student_target_logits,
                pseudo,
                valid
            )

            # -------------------------
            # 4. Ramp target loss
            # -------------------------
            target_weight = min(
                0.25,
                0.25 * epoch / 10.0
            )

            total_loss = (
                source_loss
                + target_weight * pseudo_loss
            )

        # -------------------------
        # 5. Student update
        # -------------------------
        scaler.scale(total_loss).backward()

        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            student.parameters(),
            max_norm=5.0
        )

        scaler.step(optimizer)
        scaler.update()

        # -------------------------
        # 6. EMA teacher update
        # -------------------------
        update_teacher(
            student,
            teacher,
            decay=EMA_DECAY
        )

        running_source += source_loss.item()
        running_pseudo += pseudo_loss.item()
        running_total += total_loss.item()
        running_conf += valid.float().mean().item()
        running_fg += pseudo.mean().item()

    # -----------------------------
    # Average training statistics
    # -----------------------------

    train_source = running_source / num_steps
    train_pseudo = running_pseudo / num_steps
    train_total = running_total / num_steps
    pseudo_conf = running_conf / num_steps
    pseudo_fg = running_fg / num_steps

    # -----------------------------
    # Beach validation
    # -----------------------------

    val_metrics = evaluate_beach(
        student,
        val_loader
    )

    epoch_time = time.time() - epoch_start

    record = {
        "epoch": epoch,
        "train_source_loss": train_source,
        "train_pseudo_loss": train_pseudo,
        "train_total_loss": train_total,
        "target_loss_weight": target_weight,
        "pseudo_confident_fraction": pseudo_conf,
        "pseudo_foreground_fraction": pseudo_fg,
        "val_dice": val_metrics["dice"],
        "val_iou": val_metrics["iou"],
        "val_precision": val_metrics["precision"],
        "val_recall": val_metrics["recall"],
        "val_pixel_accuracy": val_metrics["pixel_accuracy"],
        "epoch_time_sec": epoch_time
    }

    history_ema.append(record)

    print(
        f"Epoch {epoch:02d} | "
        f"Src {train_source:.4f} | "
        f"Pseudo {train_pseudo:.4f} | "
        f"Total {train_total:.4f} | "
        f"λ {target_weight:.4f} | "
        f"Conf {pseudo_conf*100:.2f}% | "
        f"PseudoFG {pseudo_fg*100:.2f}% | "
        f"Val Dice {val_metrics['dice']:.4f} | "
        f"Val IoU {val_metrics['iou']:.4f}"
    )

    # -----------------------------
    # Best checkpoint
    # -----------------------------

    if val_metrics["dice"] > BEST_DICE:

        BEST_DICE = val_metrics["dice"]
        BEST_EPOCH = epoch
        PATIENCE_COUNTER = 0

        torch.save(
            {
                "epoch": epoch,
                "student_state_dict": student.state_dict(),
                "teacher_state_dict": teacher.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_val_dice": BEST_DICE,
                "history": history_ema,
                "experiment":
                    "Experiment 5 - EMA Teacher-Student UDA",
                "ema_decay": EMA_DECAY,
                "seed": SEED
            },
            os.path.join(
                OUT_DIR,
                "experiment5_ema_best.pth"
            )
        )

        print(
            f"  >>> BEST CHECKPOINT "
            f"(epoch {epoch}, Dice {BEST_DICE:.4f})"
        )

    else:
        PATIENCE_COUNTER += 1

    # -----------------------------
    # Save history every epoch
    # -----------------------------

    with open(
        os.path.join(
            OUT_DIR,
            "training_history.json"
        ),
        "w"
    ) as f:
        json.dump(
            history_ema,
            f,
            indent=2
        )

    # -----------------------------
    # Early stopping
    # -----------------------------

    if PATIENCE_COUNTER >= PATIENCE:

        print(
            f"\nEarly stopping at epoch {epoch}."
        )
        print(
            f"Best epoch: {BEST_EPOCH}"
        )
        print(
            f"Best Beach Val Dice: {BEST_DICE:.4f}"
        )

        break

print("\nExperiment 5 training complete.")
print("Best epoch:", BEST_EPOCH)
print("Best Beach Val Dice:", BEST_DICE)

Run this checkpoint-loading cell first:

In [ ]:
BEST_CKPT = os.path.join(
    OUT_DIR,
    "experiment5_ema_best.pth"
)

ckpt = torch.load(
    BEST_CKPT,
    map_location=DEVICE
)

print("Checkpoint epoch:", ckpt["epoch"])
print("Best Beach Val Dice:", ckpt["best_val_dice"])

student.load_state_dict(
    ckpt["student_state_dict"]
)

teacher.load_state_dict(
    ckpt["teacher_state_dict"]
)

student.eval()
teacher.eval()

print("Student and teacher checkpoint loaded.")

Evaluate EMA Teacher on WWTP

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import torch
from PIL import Image

# --------------------------------------------------
# WWTP paths
# --------------------------------------------------

WWTP_ROOT = (
    "/kaggle/input/datasets/shanumkhsharma/"
    "kaggle-microplastic-bundle/data/wwtp"
)

EVAL_DIR = os.path.join(
    OUT_DIR,
    "evaluation"
)

os.makedirs(EVAL_DIR, exist_ok=True)

TILE_SIZE = 512
STRIDE = 384
THRESHOLD = 0.5


# --------------------------------------------------
# Load image-mask pairs
# --------------------------------------------------

def load_wwtp_pairs(split):

    image_dir = os.path.join(
        WWTP_ROOT,
        "images",
        split
    )

    mask_dir = os.path.join(
        WWTP_ROOT,
        "masks",
        split
    )

    pairs = []

    for name in sorted(os.listdir(image_dir)):

        if not name.lower().endswith(
            (".jpg", ".jpeg", ".png")
        ):
            continue

        image_path = os.path.join(
            image_dir,
            name
        )

        mask_path = os.path.join(
            mask_dir,
            os.path.splitext(name)[0] + ".png"
        )

        if os.path.exists(mask_path):
            pairs.append(
                (name, image_path, mask_path)
            )

    return pairs


val_pairs = load_wwtp_pairs("validation")
test_pairs = load_wwtp_pairs("test")

print("WWTP validation pairs:", len(val_pairs))
print("WWTP test pairs:", len(test_pairs))

In [ ]:
# --------------------------------------------------
# Native WWTP tiled inference
# --------------------------------------------------

@torch.no_grad()
def predict_wwtp_native(model, image):

    H, W = image.shape[:2]

    ys = list(
        range(
            0,
            H - TILE_SIZE + 1,
            STRIDE
        )
    )

    xs = list(
        range(
            0,
            W - TILE_SIZE + 1,
            STRIDE
        )
    )

    if ys[-1] != H - TILE_SIZE:
        ys.append(H - TILE_SIZE)

    if xs[-1] != W - TILE_SIZE:
        xs.append(W - TILE_SIZE)

    logit_sum = np.zeros(
        (H, W),
        dtype=np.float32
    )

    count = np.zeros(
        (H, W),
        dtype=np.float32
    )

    for y in ys:
        for x in xs:

            tile = image[
                y:y+TILE_SIZE,
                x:x+TILE_SIZE
            ]

            tile = tile.astype(
                np.float32
            ) / 255.0

            tensor = torch.from_numpy(
                tile
            ).permute(2, 0, 1).unsqueeze(0)

            tensor = tensor.to(
                DEVICE,
                non_blocking=True
            )

            logits = model(
                tensor
            )

            logits = logits[
                0, 0
            ].float().cpu().numpy()

            logit_sum[
                y:y+TILE_SIZE,
                x:x+TILE_SIZE
            ] += logits

            count[
                y:y+TILE_SIZE,
                x:x+TILE_SIZE
            ] += 1

    logits = logit_sum / np.maximum(
        count,
        1
    )

    probs = 1.0 / (
        1.0 + np.exp(-logits)
    )

    pred = (
        probs >= THRESHOLD
    ).astype(np.uint8)

    return pred, probs

Metrices

In [ ]:
def calculate_metrics(pred, gt):

    pred = pred.astype(bool)
    gt = gt.astype(bool)

    tp = np.logical_and(
        pred,
        gt
    ).sum()

    fp = np.logical_and(
        pred,
        ~gt
    ).sum()

    fn = np.logical_and(
        ~pred,
        gt
    ).sum()

    tn = np.logical_and(
        ~pred,
        ~gt
    ).sum()

    dice = (
        2 * tp
        / max(
            2 * tp + fp + fn,
            1
        )
    )

    iou = (
        tp
        / max(
            tp + fp + fn,
            1
        )
    )

    precision = (
        tp
        / max(
            tp + fp,
            1
        )
    )

    recall = (
        tp
        / max(
            tp + fn,
            1
        )
    )

    accuracy = (
        tp + tn
    ) / max(
        tp + tn + fp + fn,
        1
    )

    gt_fraction = gt.mean()
    pred_fraction = pred.mean()

    if gt_fraction > 0:
        area_ratio = (
            pred_fraction /
            gt_fraction
        )
    else:
        area_ratio = np.nan

    return {
        "dice": dice,
        "iou": iou,
        "precision": precision,
        "recall": recall,
        "pixel_accuracy": accuracy,
        "gt_foreground_fraction": gt_fraction,
        "pred_foreground_fraction": pred_fraction,
        "foreground_area_ratio": area_ratio
    }

Evalution of validation

In [ ]:
def evaluate_wwtp_split(
    model,
    pairs,
    split_name
):

    model.eval()

    results = []

    for i, (name, image_path, mask_path) in enumerate(pairs):

        image = np.asarray(
            Image.open(
                image_path
            ).convert("RGB")
        )

        gt = np.asarray(
            Image.open(
                mask_path
            ).convert("L")
        )

        gt = (
            gt > 127
        ).astype(np.uint8)

        pred, probs = predict_wwtp_native(
            model,
            image
        )

        metrics = calculate_metrics(
            pred,
            gt
        )

        metrics["image"] = name
        metrics["split"] = split_name

        results.append(metrics)

        if (i + 1) % 25 == 0:
            print(
                f"{split_name}: "
                f"{i+1}/{len(pairs)}"
            )

    return pd.DataFrame(results)

RUN the teacher

In [ ]:
teacher.eval()

ema_val_df = evaluate_wwtp_split(
    teacher,
    val_pairs,
    "validation"
)

ema_test_df = evaluate_wwtp_split(
    teacher,
    test_pairs,
    "test"
)

In [ ]:
ema_val_path = os.path.join(
    EVAL_DIR,
    "ema_teacher_wwtp_validation_per_image.csv"
)

ema_test_path = os.path.join(
    EVAL_DIR,
    "ema_teacher_wwtp_test_per_image.csv"
)

ema_val_df.to_csv(
    ema_val_path,
    index=False
)

ema_test_df.to_csv(
    ema_test_path,
    index=False
)

print("Saved:")
print(ema_val_path)
print(ema_test_path)

In [ ]:
def summarize_results(df):

    metrics = [
        "dice",
        "iou",
        "precision",
        "recall",
        "pixel_accuracy",
        "gt_foreground_fraction",
        "pred_foreground_fraction",
        "foreground_area_ratio"
    ]

    summary = {}

    for metric in metrics:

        summary[metric] = {
            "mean": float(
                df[metric].mean()
            ),
            "std": float(
                df[metric].std()
            ),
            "median": float(
                df[metric].median()
            )
        }

    return summary


val_summary = summarize_results(
    ema_val_df
)

test_summary = summarize_results(
    ema_test_df
)

print("\nWWTP VALIDATION")
print("=" * 60)

for k, v in val_summary.items():
    print(
        f"{k:28s} "
        f"{v['mean']:.6f}"
    )

print("\nWWTP TEST")
print("=" * 60)

for k, v in test_summary.items():
    print(
        f"{k:28s} "
        f"{v['mean']:.6f}"
    )

#This evaluation is for the EMA teacher, not the student. That is intentional because Experiment 5's hypothesis is that the EMA teacher provides the more stable target-domain representation.

In [ ]:
summary = {
    "experiment":
        "Experiment 5 - EMA Teacher-Student UDA",
    "checkpoint_epoch":
        int(ckpt["epoch"]),
    "beach_val_dice":
        float(ckpt["best_val_dice"]),
    "ema_decay":
        EMA_DECAY,
    "foreground_threshold":
        FG_THRESHOLD,
    "background_threshold":
        BG_THRESHOLD,
    "max_pseudo_foreground_fraction":
        MAX_PSEUDO_FG_FRACTION,
    "wwtp_validation":
        val_summary,
    "wwtp_test":
        test_summary
}

summary_path = os.path.join(
    EVAL_DIR,
    "ema_teacher_wwtp_summary.json"
)

with open(summary_path, "w") as f:
    json.dump(
        summary,
        f,
        indent=2
    )

print("\nSummary saved to:")
print(summary_path)

EMA teacher–student UDA partially reduced foreground overprediction compared with DANN and direct pseudo-label UDA, but did not recover useful cross-domain segmentation performance. The model still predicted approximately 70.6% of the WWTP test image as foreground, compared with only 1.22% ground-truth foreground.

In [ ]:
# ============================================================
# EXPERIMENT 5 — EMA TEACHER-STUDENT UDA
# ALL VISUALIZATIONS IN ONE CELL
# ============================================================

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

EXP5_DIR = "/kaggle/working/experiment5_ema_teacher_student"
EVAL_DIR = os.path.join(EXP5_DIR, "evaluation")

HISTORY_PATH = os.path.join(
    EXP5_DIR,
    "training_history.json"
)

VAL_CSV = os.path.join(
    EVAL_DIR,
    "ema_teacher_wwtp_validation_per_image.csv"
)

TEST_CSV = os.path.join(
    EVAL_DIR,
    "ema_teacher_wwtp_test_per_image.csv"
)

# ------------------------------------------------------------
# Load Experiment 5 training history
# ------------------------------------------------------------

with open(HISTORY_PATH, "r") as f:
    history = json.load(f)

epochs = [h["epoch"] for h in history]

source_loss = [
    h["train_source_loss"] for h in history
]

pseudo_loss = [
    h["train_pseudo_loss"] for h in history
]

total_loss = [
    h["train_total_loss"] for h in history
]

target_weight = [
    h["target_loss_weight"] for h in history
]

confidence = [
    h["pseudo_confident_fraction"] for h in history
]

pseudo_fg = [
    h["pseudo_foreground_fraction"] for h in history
]

val_dice = [
    h["val_dice"] for h in history
]

val_iou = [
    h["val_iou"] for h in history
]

val_precision = [
    h["val_precision"] for h in history
]

val_recall = [
    h["val_recall"] for h in history
]

# ------------------------------------------------------------
# Load WWTP results
# ------------------------------------------------------------

val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)

# ============================================================
# 1. BEACH VALIDATION DICE
# ============================================================

plt.figure(figsize=(8,5))

plt.plot(
    epochs,
    val_dice,
    marker="o",
    label="Beach Validation Dice"
)

best_idx = int(np.argmax(val_dice))

plt.scatter(
    epochs[best_idx],
    val_dice[best_idx],
    s=100,
    zorder=5,
    label=(
        f"Best: {val_dice[best_idx]:.4f} "
        f"(Epoch {epochs[best_idx]})"
    )
)

plt.xlabel("Epoch")
plt.ylabel("Dice")
plt.title("Experiment 5: Beach Validation Dice")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


# ============================================================
# 2. TRAINING LOSSES
# ============================================================

plt.figure(figsize=(8,5))

plt.plot(
    epochs,
    source_loss,
    marker="o",
    label="Source segmentation loss"
)

plt.plot(
    epochs,
    pseudo_loss,
    marker="o",
    label="Teacher pseudo-label loss"
)

plt.plot(
    epochs,
    total_loss,
    marker="o",
    label="Total loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Experiment 5: Training Losses")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


# ============================================================
# 3. EMA TARGET CONFIDENCE
# ============================================================

plt.figure(figsize=(8,5))

plt.plot(
    epochs,
    np.array(confidence) * 100,
    marker="o"
)

plt.xlabel("Epoch")
plt.ylabel("Confident Pixels (%)")
plt.title("Experiment 5: EMA Teacher Confidence Coverage")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


# ============================================================
# 4. PSEUDO-FOREGROUND FRACTION
# ============================================================

plt.figure(figsize=(8,5))

plt.plot(
    epochs,
    np.array(pseudo_fg) * 100,
    marker="o",
    label="Pseudo-foreground"
)

plt.axhline(
    11.9705,
    linestyle="--",
    label="Maximum allowed = 11.97%"
)

plt.xlabel("Epoch")
plt.ylabel("Pseudo-foreground (%)")
plt.title("Experiment 5: Teacher Pseudo-Label Foreground")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


# ============================================================
# 5. BEACH VALIDATION METRICS
# ============================================================

plt.figure(figsize=(9,5))

plt.plot(
    epochs,
    val_dice,
    marker="o",
    label="Dice"
)

plt.plot(
    epochs,
    val_iou,
    marker="o",
    label="IoU"
)

plt.plot(
    epochs,
    val_precision,
    marker="o",
    label="Precision"
)

plt.plot(
    epochs,
    val_recall,
    marker="o",
    label="Recall"
)

plt.xlabel("Epoch")
plt.ylabel("Score")
plt.title("Experiment 5: Beach Validation Metrics")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


# ============================================================
# 6. WWTP TEST — GT VS PREDICTED FOREGROUND
# ============================================================

gt_fg = (
    test_df["gt_foreground_fraction"].mean() * 100
)

pred_fg = (
    test_df["pred_foreground_fraction"].mean() * 100
)

plt.figure(figsize=(7,5))

bars = plt.bar(
    ["Ground Truth", "EMA Prediction"],
    [gt_fg, pred_fg]
)

plt.ylabel("Foreground Area (%)")
plt.title("Experiment 5: WWTP Test Foreground Area")
plt.grid(axis="y", alpha=0.3)

for bar, value in zip(
    bars,
    [gt_fg, pred_fg]
):
    plt.text(
        bar.get_x() + bar.get_width()/2,
        value + 1,
        f"{value:.2f}%",
        ha="center"
    )

plt.tight_layout()
plt.show()


# ============================================================
# 7. WWTP TEST — PRECISION VS RECALL
# ============================================================

precision = test_df["precision"].mean()
recall = test_df["recall"].mean()

plt.figure(figsize=(7,5))

bars = plt.bar(
    ["Precision", "Recall"],
    [precision, recall]
)

plt.ylabel("Score")
plt.ylim(0, 1)
plt.title("Experiment 5: WWTP Test Precision vs Recall")
plt.grid(axis="y", alpha=0.3)

for bar, value in zip(
    bars,
    [precision, recall]
):
    plt.text(
        bar.get_x() + bar.get_width()/2,
        value + 0.02,
        f"{value:.3f}",
        ha="center"
    )

plt.tight_layout()
plt.show()


# ============================================================
# 8. WWTP TEST DICE DISTRIBUTION
# ============================================================

plt.figure(figsize=(8,5))

plt.hist(
    test_df["dice"],
    bins=30,
    edgecolor="black"
)

mean_dice = test_df["dice"].mean()

plt.axvline(
    mean_dice,
    linestyle="--",
    label=f"Mean Dice = {mean_dice:.4f}"
)

plt.xlabel("Dice")
plt.ylabel("Number of Images")
plt.title("Experiment 5: WWTP Test Dice Distribution")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


# ============================================================
# 9. ALL EXPERIMENTS — WWTP DICE
# ============================================================

models = [
    "Baseline\nU-Net",
    "Strong\nAug.",
    "Appearance\nAug.",
    "DeepLabV3+",
    "DANN",
    "Pseudo-label\nUDA",
    "EMA Teacher"
]

dice_values = [
    0.0604,
    0.0475,
    0.0631,
    0.0566,
    0.0188,
    0.0192,
    0.0197
]

plt.figure(figsize=(11,6))

bars = plt.bar(
    models,
    dice_values
)

plt.ylabel("WWTP Test Dice")
plt.title("Cross-Domain WWTP Performance Across Experiments")
plt.ylim(0, 0.07)
plt.grid(axis="y", alpha=0.3)

for bar, value in zip(
    bars,
    dice_values
):
    plt.text(
        bar.get_x() + bar.get_width()/2,
        value + 0.001,
        f"{value:.3f}",
        ha="center"
    )

plt.tight_layout()
plt.show()


# ============================================================
# 10. UDA METHODS — PRECISION / RECALL
# ============================================================

uda_models = [
    "DANN",
    "Pseudo-label UDA",
    "EMA Teacher"
]

uda_precision = [
    0.0098,
    0.0100,
    precision
]

uda_recall = [
    0.7537,
    0.7690,
    recall
]

x = np.arange(len(uda_models))
width = 0.35

plt.figure(figsize=(9,5))

plt.bar(
    x - width/2,
    uda_precision,
    width,
    label="Precision"
)

plt.bar(
    x + width/2,
    uda_recall,
    width,
    label="Recall"
)

plt.xticks(
    x,
    uda_models
)

plt.ylabel("Score")
plt.ylim(0, 1)
plt.title("UDA Methods: WWTP Test Precision vs Recall")
plt.legend()
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


# ============================================================
# 11. UDA METHODS — FOREGROUND OVERPREDICTION
# ============================================================

uda_fg = [
    92.4,
    93.7,
    pred_fg
]

plt.figure(figsize=(9,5))

bars = plt.bar(
    uda_models,
    uda_fg
)

plt.axhline(
    1.223,
    linestyle="--",
    label="WWTP Test GT = 1.22%"
)

plt.ylabel("Predicted Foreground (%)")
plt.title("UDA Methods: WWTP Foreground Overprediction")
plt.legend()
plt.grid(axis="y", alpha=0.3)

for bar, value in zip(
    bars,
    uda_fg
):
    plt.text(
        bar.get_x() + bar.get_width()/2,
        value + 1,
        f"{value:.1f}%",
        ha="center"
    )

plt.tight_layout()
plt.show()


# ============================================================
# FINAL NUMERICAL SUMMARY
# ============================================================

print("\n" + "="*65)
print("EXPERIMENT 5 — EMA TEACHER-STUDENT UDA")
print("="*65)

print(
    f"Best Epoch:              {epochs[best_idx]}"
)

print(
    f"Best Beach Val Dice:     {val_dice[best_idx]:.4f}"
)

print("\nWWTP Validation:")
print(
    f"Dice:                    "
    f"{val_df['dice'].mean():.4f}"
)
print(
    f"IoU:                     "
    f"{val_df['iou'].mean():.4f}"
)
print(
    f"Precision:               "
    f"{val_df['precision'].mean():.4f}"
)
print(
    f"Recall:                  "
    f"{val_df['recall'].mean():.4f}"
)
print(
    f"Predicted foreground:    "
    f"{val_df['pred_foreground_fraction'].mean()*100:.2f}%"
)

print("\nWWTP Test:")
print(
    f"Dice:                    "
    f"{test_df['dice'].mean():.4f}"
)
print(
    f"IoU:                     "
    f"{test_df['iou'].mean():.4f}"
)
print(
    f"Precision:               "
    f"{test_df['precision'].mean():.4f}"
)
print(
    f"Recall:                  "
    f"{test_df['recall'].mean():.4f}"
)
print(
    f"GT foreground:           "
    f"{test_df['gt_foreground_fraction'].mean()*100:.2f}%"
)
print(
    f"Predicted foreground:    "
    f"{test_df['pred_foreground_fraction'].mean()*100:.2f}%"
)
print(
    f"Foreground area ratio:   "
    f"{test_df['foreground_area_ratio'].median():.2f}x "
    f"(median)"
)

print("="*65)

#Experiment 6 — Hybrid Robust Source-Domain Training

The core idea is:
- U-Net + ResNet50
- Appearance-focused augmentation
- Geometric augmentation
- BCE + Dice
- No DANN
- No pseudo-label UDA
- No EMA target adaptation
- WWTP remains completely unseen during training/model selection

Experiment 6 — Hybrid Source-Domain Robustness

Architecture:

    U-Net + ResNet50

Input:

    512 × 512 RGB

Beach:

    Train = 790
    Validation = 169
    Test = 169

WWTP:

    Completely unseen during training
    Completely unseen during model selection

Augmentation:

    Horizontal flip p=0.5
    Rotation ±20°
    Scale 0.90–1.10
    Translation ±10%
    Brightness/contrast ±25%
    Hue ±30
    Saturation ±30
    Value ±20
    Gamma 70–130
    Gaussian blur p=0.25

Loss:

    0.5 BCE + 0.5 Dice

Optimizer:

    AdamW
    LR = 1e-4
    Weight decay = 1e-4

Batch:

    8 if GPU memory permits

Epochs:

    maximum 50

Early stopping:

    patience 10

Seed:

    42

Model selection:

    Beach validation Dice only

Final evaluation:

    WWTP validation + test
    512×512 tiles
    stride 384
    average overlapping logits
    threshold 0.5

Imports and configuration

In [ ]:
import os
import json
import random
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import segmentation_models_pytorch as smp

from PIL import Image
from torch.utils.data import Dataset, DataLoader

SEED = 42

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)

BEACH_ROOT = (
    "/kaggle/input/datasets/shanumkhsharma/"
    "microplastic-step16a-recovery/kaggle_bundle/data"
)

WWTP_ROOT = (
    "/kaggle/input/datasets/shanumkhsharma/"
    "kaggle-microplastic-bundle/data/wwtp"
)

OUT_DIR = (
    "/kaggle/working/"
    "experiment6_hybrid_source_robustness"
)

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(
    os.path.join(OUT_DIR, "evaluation"),
    exist_ok=True
)

print("Beach root exists:", os.path.exists(BEACH_ROOT))
print("WWTP root exists:", os.path.exists(WWTP_ROOT))

Augmentation pipeline

In [ ]:
import albumentations as A

train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),

    A.Affine(
        scale=(0.90, 1.10),
        translate_percent=(-0.10, 0.10),
        rotate=(-20, 20),
        p=0.8
    ),

    A.RandomBrightnessContrast(
        brightness_limit=0.25,
        contrast_limit=0.25,
        p=0.6
    ),

    A.HueSaturationValue(
        hue_shift_limit=30,
        sat_shift_limit=30,
        val_shift_limit=20,
        p=0.5
    ),

    A.RandomGamma(
        gamma_limit=(70, 130),
        p=0.4
    ),

    A.GaussianBlur(
        blur_limit=(3, 7),
        p=0.25
    )
])
class BeachHybridDataset(Dataset):

    def __init__(
        self,
        root,
        split,
        transform=None
    ):
        self.image_dir = os.path.join(
            root,
            split,
            "images"
        )

        self.mask_dir = os.path.join(
            root,
            split,
            "masks"
        )

        self.transform = transform

        self.images = sorted([
            f for f in os.listdir(self.image_dir)
            if f.lower().endswith(
                (".png", ".jpg", ".jpeg")
            )
        ])

        self.images = [
            f for f in self.images
            if os.path.exists(
                os.path.join(
                    self.mask_dir,
                    os.path.splitext(f)[0] + ".png"
                )
            )
        ]

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):

        name = self.images[idx]

        image = np.asarray(
            Image.open(
                os.path.join(
                    self.image_dir,
                    name
                )
            ).convert("RGB")
        )

        mask = np.asarray(
            Image.open(
                os.path.join(
                    self.mask_dir,
                    os.path.splitext(name)[0] + ".png"
                )
            ).convert("L")
        )

        if self.transform is not None:

            transformed = self.transform(
                image=image,
                mask=mask
            )

            image = transformed["image"]
            mask = transformed["mask"]

        image = np.asarray(
            Image.fromarray(image).resize(
                (512, 512),
                Image.BILINEAR
            ),
            dtype=np.float32
        ) / 255.0

        mask = np.asarray(
            Image.fromarray(mask).resize(
                (512, 512),
                Image.NEAREST
            ),
            dtype=np.float32
        )

        mask = (
            mask > 127
        ).astype(np.float32)

        image = torch.from_numpy(
            image
        ).permute(2, 0, 1)

        mask = torch.from_numpy(
            mask
        ).unsqueeze(0)

        return image, mask


In [ ]:
# ============================================================
# EXPERIMENT 6 — CORRECTED DATALOADERS
# ============================================================

import os
import numpy as np
import albumentations as A

from PIL import Image
from torch.utils.data import Dataset, DataLoader

# ------------------------------------------------------------
# 1. Define transforms
# ------------------------------------------------------------

# Training: hybrid geometric + appearance augmentation
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),

    A.Affine(
        scale=(0.90, 1.10),
        translate_percent=(-0.10, 0.10),
        rotate=(-20, 20),
        p=0.8
    ),

    A.RandomBrightnessContrast(
        brightness_limit=0.25,
        contrast_limit=0.25,
        p=0.6
    ),

    A.HueSaturationValue(
        hue_shift_limit=30,
        sat_shift_limit=30,
        val_shift_limit=20,
        p=0.5
    ),

    A.RandomGamma(
        gamma_limit=(70, 130),
        p=0.4
    ),

    A.GaussianBlur(
        blur_limit=(3, 7),
        p=0.25
    )
])

# Validation/test: NO augmentation
val_transform = A.Compose([])

# ------------------------------------------------------------
# 2. Dataset
# ------------------------------------------------------------

class BeachHybridDataset(Dataset):

    def __init__(
        self,
        root,
        split,
        transform=None
    ):
        self.image_dir = os.path.join(
            root,
            split,
            "images"
        )

        self.mask_dir = os.path.join(
            root,
            split,
            "masks"
        )

        self.transform = transform

        self.images = sorted([
            f for f in os.listdir(self.image_dir)
            if f.lower().endswith(
                (".png", ".jpg", ".jpeg")
            )
        ])

        self.images = [
            f for f in self.images
            if os.path.exists(
                os.path.join(
                    self.mask_dir,
                    os.path.splitext(f)[0] + ".png"
                )
            )
        ]

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):

        name = self.images[idx]

        image = np.asarray(
            Image.open(
                os.path.join(
                    self.image_dir,
                    name
                )
            ).convert("RGB")
        )

        mask = np.asarray(
            Image.open(
                os.path.join(
                    self.mask_dir,
                    os.path.splitext(name)[0] + ".png"
                )
            ).convert("L")
        )

        # Apply augmentation before resizing
        if self.transform is not None:

            transformed = self.transform(
                image=image,
                mask=mask
            )

            image = transformed["image"]
            mask = transformed["mask"]

        # Standardize to 512 × 512
        image = np.asarray(
            Image.fromarray(image).resize(
                (512, 512),
                Image.BILINEAR
            ),
            dtype=np.float32
        ) / 255.0

        mask = np.asarray(
            Image.fromarray(mask).resize(
                (512, 512),
                Image.NEAREST
            ),
            dtype=np.float32
        )

        mask = (
            mask > 127
        ).astype(np.float32)

        image = torch.from_numpy(
            image
        ).permute(2, 0, 1)

        mask = torch.from_numpy(
            mask
        ).unsqueeze(0)

        return image, mask


# ------------------------------------------------------------
# 3. Create datasets
# ------------------------------------------------------------

train_ds = BeachHybridDataset(
    BEACH_ROOT,
    "train",
    transform=train_transform
)

val_ds = BeachHybridDataset(
    BEACH_ROOT,
    "val",
    transform=val_transform
)

test_ds = BeachHybridDataset(
    BEACH_ROOT,
    "test",
    transform=val_transform
)


# ------------------------------------------------------------
# 4. Create DataLoaders
# ------------------------------------------------------------

train_loader = DataLoader(
    train_ds,
    batch_size=8,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=8,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_ds,
    batch_size=8,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


# ------------------------------------------------------------
# 5. Verify
# ------------------------------------------------------------

print("Beach train:", len(train_ds))
print("Beach val:", len(val_ds))
print("Beach test:", len(test_ds))

images, masks = next(iter(train_loader))

print("\nTraining batch:")
print("Images:", images.shape)
print("Masks:", masks.shape)
print("Image dtype:", images.dtype)
print("Mask dtype:", masks.dtype)
print("Image range:", images.min().item(), images.max().item())
print("Mask values:", torch.unique(masks))

In [ ]:
model = smp.Unet(
    encoder_name="resnet50",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
    activation=None
).to(DEVICE)

print(
    "Parameters:",
    sum(p.numel() for p in model.parameters())
)
bce_loss = nn.BCEWithLogitsLoss()

def dice_loss(
    logits,
    targets,
    eps=1e-6
):
    probs = torch.sigmoid(logits)

    intersection = (
        probs * targets
    ).sum(dim=(1,2,3))

    denominator = (
        probs.sum(dim=(1,2,3))
        + targets.sum(dim=(1,2,3))
    )

    dice = (
        2 * intersection + eps
    ) / (
        denominator + eps
    )

    return 1 - dice.mean()


def segmentation_loss(
    logits,
    masks
):
    bce = bce_loss(
        logits,
        masks
    )

    dice = dice_loss(
        logits,
        masks
    )

    return 0.5 * bce + 0.5 * dice
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=torch.cuda.is_available()
)
@torch.no_grad()
def evaluate_beach(
    model,
    loader
):

    model.eval()

    dices = []
    ious = []
    precisions = []
    recalls = []
    accuracies = []

    for images, masks in loader:

        images = images.to(
            DEVICE,
            non_blocking=True
        )

        masks = masks.to(
            DEVICE,
            non_blocking=True
        )

        with torch.amp.autocast(
            "cuda",
            enabled=torch.cuda.is_available()
        ):
            logits = model(images)

        probs = torch.sigmoid(logits)

        preds = (
            probs >= 0.5
        ).float()

        tp = (
            preds * masks
        ).sum(dim=(1,2,3))

        fp = (
            preds * (1-masks)
        ).sum(dim=(1,2,3))

        fn = (
            (1-preds) * masks
        ).sum(dim=(1,2,3))

        tn = (
            (1-preds) * (1-masks)
        ).sum(dim=(1,2,3))

        dice = (
            2*tp + 1e-6
        ) / (
            2*tp + fp + fn + 1e-6
        )

        iou = (
            tp + 1e-6
        ) / (
            tp + fp + fn + 1e-6
        )

        precision = (
            tp + 1e-6
        ) / (
            tp + fp + 1e-6
        )

        recall = (
            tp + 1e-6
        ) / (
            tp + fn + 1e-6
        )

        accuracy = (
            tp + tn + 1e-6
        ) / (
            tp + tn + fp + fn + 1e-6
        )

        dices.extend(
            dice.cpu().numpy()
        )

        ious.extend(
            iou.cpu().numpy()
        )

        precisions.extend(
            precision.cpu().numpy()
        )

        recalls.extend(
            recall.cpu().numpy()
        )

        accuracies.extend(
            accuracy.cpu().numpy()
        )

    return {
        "dice": float(np.mean(dices)),
        "iou": float(np.mean(ious)),
        "precision": float(np.mean(precisions)),
        "recall": float(np.mean(recalls)),
        "pixel_accuracy": float(
            np.mean(accuracies)
        )
    }

In [ ]:
MAX_EPOCHS = 50
PATIENCE = 10

best_dice = -1
best_epoch = 0
patience_counter = 0

history = []

for epoch in range(
    1,
    MAX_EPOCHS + 1
):

    start_time = time.time()

    model.train()

    running_loss = 0.0

    for images, masks in train_loader:

        images = images.to(
            DEVICE,
            non_blocking=True
        )

        masks = masks.to(
            DEVICE,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.amp.autocast(
            "cuda",
            enabled=torch.cuda.is_available()
        ):

            logits = model(images)

            loss = segmentation_loss(
                logits,
                masks
            )

        scaler.scale(loss).backward()

        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=5.0
        )

        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()

    train_loss = (
        running_loss /
        len(train_loader)
    )

    val_metrics = evaluate_beach(
        model,
        val_loader
    )

    epoch_time = (
        time.time() - start_time
    )

    record = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_dice": val_metrics["dice"],
        "val_iou": val_metrics["iou"],
        "val_precision": val_metrics["precision"],
        "val_recall": val_metrics["recall"],
        "val_pixel_accuracy":
            val_metrics["pixel_accuracy"],
        "epoch_time_sec": epoch_time
    }

    history.append(record)

    print(
        f"Epoch {epoch:02d} | "
        f"Train {train_loss:.4f} | "
        f"Val Dice {val_metrics['dice']:.4f} | "
        f"Val IoU {val_metrics['iou']:.4f}"
    )

    if (
        val_metrics["dice"]
        > best_dice
    ):

        best_dice = (
            val_metrics["dice"]
        )

        best_epoch = epoch
        patience_counter = 0

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict":
                    model.state_dict(),
                "optimizer_state_dict":
                    optimizer.state_dict(),
                "best_val_dice":
                    best_dice,
                "history": history,
                "experiment":
                    "Experiment 6 - Hybrid Source-Domain Robustness",
                "seed": SEED
            },
            os.path.join(
                OUT_DIR,
                "experiment6_hybrid_best.pth"
            )
        )

        print(
            f"  >>> BEST CHECKPOINT "
            f"(epoch {epoch}, "
            f"Dice {best_dice:.4f})"
        )

    else:

        patience_counter += 1

    with open(
        os.path.join(
            OUT_DIR,
            "training_history.json"
        ),
        "w"
    ) as f:

        json.dump(
            history,
            f,
            indent=2
        )

    if patience_counter >= PATIENCE:

        print(
            f"\nEarly stopping at "
            f"epoch {epoch}"
        )

        break

print("\nTraining complete.")
print("Best epoch:", best_epoch)
print("Best Beach Val Dice:", best_dice)

In [ ]:
# ============================================================
# EXPERIMENT 6 — WWTP EXTERNAL EVALUATION
# Hybrid Source-Domain Robustness
# Best checkpoint = epoch 21 selected ONLY by Beach validation Dice
# ============================================================

import os, json, glob
import numpy as np
import pandas as pd
from PIL import Image

import torch
import segmentation_models_pytorch as smp
from tqdm.auto import tqdm

# ------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------

CKPT = "/kaggle/working/experiment6_hybrid_source_robustness/experiment6_hybrid_best.pth"

WWTP_ROOT = (
    "/kaggle/input/datasets/shanumkhsharma/"
    "kaggle-microplastic-bundle/data/wwtp"
)

OUT_DIR = "/kaggle/working/experiment6_hybrid_source_robustness/evaluation"
os.makedirs(OUT_DIR, exist_ok=True)

assert os.path.exists(CKPT), f"Checkpoint not found: {CKPT}"
assert os.path.exists(WWTP_ROOT), f"WWTP root not found: {WWTP_ROOT}"

# ------------------------------------------------------------
# 2. DEVICE
# ------------------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ------------------------------------------------------------
# 3. MODEL
# ------------------------------------------------------------

model = smp.Unet(
    encoder_name="resnet50",
    encoder_weights=None,
    in_channels=3,
    classes=1,
    activation=None
).to(device)

checkpoint = torch.load(CKPT, map_location=device)

# Handle checkpoint format
if "model_state_dict" in checkpoint:
    state_dict = checkpoint["model_state_dict"]
else:
    state_dict = checkpoint

model.load_state_dict(state_dict)
model.eval()

print("Checkpoint:", CKPT)
print("Checkpoint epoch:", checkpoint.get("epoch", "unknown"))
print(
    "Checkpoint best Beach Val Dice:",
    checkpoint.get("best_val_dice", "unknown")
)

# ------------------------------------------------------------
# 4. WWTP TILING
# ------------------------------------------------------------

TILE = 512
STRIDE = 384   # 128 px overlap
THRESHOLD = 0.5

def get_positions(length, tile=TILE, stride=STRIDE):
    """
    Generate complete tile positions while forcing the final
    tile to reach the image boundary.
    """
    if length <= tile:
        return [0]

    positions = list(range(0, length - tile + 1, stride))

    final_pos = length - tile

    if positions[-1] != final_pos:
        positions.append(final_pos)

    return positions


@torch.no_grad()
def predict_native(image_np):
    """
    Native-resolution WWTP inference:
      3840x2160
      -> overlapping 512x512 tiles
      -> average logits
      -> sigmoid
      -> threshold
    """

    H, W = image_np.shape[:2]

    ys = get_positions(H)
    xs = get_positions(W)

    logit_sum = np.zeros((H, W), dtype=np.float32)
    count_map = np.zeros((H, W), dtype=np.float32)

    for y in ys:
        for x in xs:

            tile = image_np[y:y+TILE, x:x+TILE]

            # Safety for any edge case
            if tile.shape[0] != TILE or tile.shape[1] != TILE:
                padded = np.zeros(
                    (TILE, TILE, 3),
                    dtype=np.uint8
                )
                padded[:tile.shape[0], :tile.shape[1]] = tile
                tile = padded

            tile_tensor = (
                torch.from_numpy(tile)
                .permute(2, 0, 1)
                .float()
                / 255.0
            ).unsqueeze(0).to(device)

            logits = model(tile_tensor)

            logits = logits[0, 0].float().cpu().numpy()

            h, w = min(TILE, H-y), min(TILE, W-x)

            logit_sum[y:y+h, x:x+w] += logits[:h, :w]
            count_map[y:y+h, x:x+w] += 1.0

    avg_logits = logit_sum / np.maximum(count_map, 1e-8)

    probability = 1.0 / (1.0 + np.exp(-avg_logits))

    prediction = (probability >= THRESHOLD).astype(np.uint8)

    return prediction, probability


# ------------------------------------------------------------
# 5. METRICS
# ------------------------------------------------------------

def compute_metrics(pred, gt):
    pred = pred.astype(bool)
    gt = gt.astype(bool)

    intersection = np.logical_and(pred, gt).sum()
    pred_area = pred.sum()
    gt_area = gt.sum()

    union = np.logical_or(pred, gt).sum()

    dice = (
        2.0 * intersection / (pred_area + gt_area)
        if (pred_area + gt_area) > 0 else 1.0
    )

    iou = (
        intersection / union
        if union > 0 else 1.0
    )

    precision = (
        intersection / pred_area
        if pred_area > 0 else 0.0
    )

    recall = (
        intersection / gt_area
        if gt_area > 0 else 0.0
    )

    pixel_accuracy = (
        (pred == gt).mean()
    )

    gt_fraction = gt.mean()
    pred_fraction = pred.mean()

    area_ratio = (
        pred_area / gt_area
        if gt_area > 0 else np.nan
    )

    return {
        "dice": float(dice),
        "iou": float(iou),
        "precision": float(precision),
        "recall": float(recall),
        "pixel_accuracy": float(pixel_accuracy),
        "gt_foreground_fraction": float(gt_fraction),
        "pred_foreground_fraction": float(pred_fraction),
        "foreground_area_ratio": float(area_ratio),
    }


# ------------------------------------------------------------
# 6. EVALUATE ONE WWTP SPLIT
# ------------------------------------------------------------

def evaluate_split(split):

    image_dir = os.path.join(WWTP_ROOT, "images", split)
    mask_dir = os.path.join(WWTP_ROOT, "masks", split)

    image_files = sorted(
        glob.glob(os.path.join(image_dir, "*"))
    )

    print(f"\n{'='*65}")
    print(f"WWTP {split.upper()} EVALUATION")
    print(f"Images: {len(image_files)}")
    print(f"{'='*65}")

    results = []

    for image_path in tqdm(image_files):

        stem = os.path.splitext(
            os.path.basename(image_path)
        )[0]

        mask_path = os.path.join(
            mask_dir,
            stem + ".png"
        )

        assert os.path.exists(
            mask_path
        ), f"Missing mask: {mask_path}"

        image = np.array(
            Image.open(image_path).convert("RGB")
        )

        gt = np.array(
            Image.open(mask_path).convert("L")
        )

        gt = (gt > 0).astype(np.uint8)

        pred, probability = predict_native(image)

        metrics = compute_metrics(pred, gt)

        row = {
            "image": stem,
            "split": split,
            "height": image.shape[0],
            "width": image.shape[1],
            "gt_positive_pixels": int(gt.sum()),
            "pred_positive_pixels": int(pred.sum()),
            "probability_mean": float(probability.mean()),
            **metrics
        }

        results.append(row)

    df = pd.DataFrame(results)

    # --------------------------------------------------------
    # Primary aggregate
    # Exclude annotation-empty images from primary metric
    # --------------------------------------------------------

    primary = df[df["gt_positive_pixels"] > 0].copy()
    empty = df[df["gt_positive_pixels"] == 0].copy()

    summary = {
        "experiment": "Experiment 6 — Hybrid Source-Domain Robustness",
        "checkpoint": CKPT,
        "checkpoint_epoch": int(
            checkpoint.get("epoch", -1)
        ),
        "wwtp_split": split,

        "total_images": int(len(df)),
        "annotated_positive_images": int(len(primary)),
        "annotation_empty_images": int(len(empty)),

        "tile_size": TILE,
        "stride": STRIDE,
        "overlap_pixels": TILE - STRIDE,
        "threshold": THRESHOLD,

        "dice_mean": float(primary["dice"].mean()),
        "dice_std": float(primary["dice"].std()),
        "dice_median": float(primary["dice"].median()),

        "iou_mean": float(primary["iou"].mean()),
        "iou_std": float(primary["iou"].std()),
        "iou_median": float(primary["iou"].median()),

        "precision_mean": float(primary["precision"].mean()),
        "recall_mean": float(primary["recall"].mean()),
        "pixel_accuracy_mean": float(
            primary["pixel_accuracy"].mean()
        ),

        "gt_foreground_fraction_mean": float(
            primary["gt_foreground_fraction"].mean()
        ),

        "pred_foreground_fraction_mean": float(
            primary["pred_foreground_fraction"].mean()
        ),

        "foreground_area_ratio_median": float(
            primary["foreground_area_ratio"].median()
        ),
    }

    csv_path = os.path.join(
        OUT_DIR,
        f"experiment6_wwtp_{split}_per_image.csv"
    )

    json_path = os.path.join(
        OUT_DIR,
        f"experiment6_wwtp_{split}_summary.json"
    )

    df.to_csv(csv_path, index=False)

    with open(json_path, "w") as f:
        json.dump(summary, f, indent=2)

    print("\nRESULTS")
    print("-" * 50)
    print(f"Primary images: {len(primary)}")
    print(f"Empty images:   {len(empty)}")
    print(f"Dice:           {summary['dice_mean']:.6f}")
    print(f"IoU:            {summary['iou_mean']:.6f}")
    print(f"Precision:      {summary['precision_mean']:.6f}")
    print(f"Recall:         {summary['recall_mean']:.6f}")
    print(f"Pixel Accuracy: {summary['pixel_accuracy_mean']:.6f}")
    print(
        f"GT FG fraction: "
        f"{summary['gt_foreground_fraction_mean']:.6f}"
    )
    print(
        f"Pred FG fraction: "
        f"{summary['pred_foreground_fraction_mean']:.6f}"
    )
    print(
        f"Median area ratio: "
        f"{summary['foreground_area_ratio_median']:.3f}x"
    )

    return df, summary


# ------------------------------------------------------------
# 7. RUN VALIDATION + TEST
# ------------------------------------------------------------

val_df, val_summary = evaluate_split("validation")
test_df, test_summary = evaluate_split("test")

# ------------------------------------------------------------
# 8. COMPARISON SUMMARY
# ------------------------------------------------------------

comparison = pd.DataFrame([
    {
        "experiment": "Exp6 Hybrid Source Robustness",
        "split": "WWTP validation",
        "dice": val_summary["dice_mean"],
        "iou": val_summary["iou_mean"],
        "precision": val_summary["precision_mean"],
        "recall": val_summary["recall_mean"],
        "pred_fg_fraction": val_summary[
            "pred_foreground_fraction_mean"
        ]
    },
    {
        "experiment": "Exp6 Hybrid Source Robustness",
        "split": "WWTP test",
        "dice": test_summary["dice_mean"],
        "iou": test_summary["iou_mean"],
        "precision": test_summary["precision_mean"],
        "recall": test_summary["recall_mean"],
        "pred_fg_fraction": test_summary[
            "pred_foreground_fraction_mean"
        ]
    }
])

comparison_path = os.path.join(
    OUT_DIR,
    "experiment6_wwtp_summary_comparison.csv"
)

comparison.to_csv(comparison_path, index=False)

print("\n" + "="*65)
print("EXPERIMENT 6 COMPLETE")
print("="*65)
print(comparison.to_string(index=False))
print("\nSaved to:", OUT_DIR)

In [ ]:
import matplotlib.pyplot as plt

# WWTP external test Dice scores
experiments = [
    "Exp. 1\nBaseline U-Net",
    "Exp. 2A\nStrong Aug.",
    "Exp. 2B\nAppearance Aug.",
    "Exp. 2C\nDeepLabV3+",
    "Exp. 3\nDANN",
    "Exp. 4\nPseudo-label",
    "Exp. 5\nEMA Teacher",
    "Exp. 6\nHybrid Source-only"
]

wwtp_dice = [
    0.0604,
    0.0475,
    0.0631,
    0.0566,
    0.0188,
    0.0192,
    0.0197,
    0.0157
]

plt.figure(figsize=(13, 6))

bars = plt.bar(experiments, wwtp_dice)

plt.title("WWTP Test Dice Across All Completed Experiments")
plt.xlabel("Experiment")
plt.ylabel("WWTP Test Dice")
plt.ylim(0, 0.075)
plt.xticks(rotation=0)
plt.grid(axis="y", linestyle="--", alpha=0.4)

# Display score above each bar
for bar, score in zip(bars, wwtp_dice):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.001,
        f"{score:.4f}",
        ha="center",
        va="bottom",
        fontsize=9
    )

plt.tight_layout()
plt.savefig("wwtp_dice_all_experiments.png", dpi=300, bbox_inches="tight")
plt.show()